# Autonomous Agentic Workspace (`DeepSeek-V4-Pro · Qwen3.5-9B`)

A self-contained autonomous agent workspace for **Google Colab GPUs (T4 / L4 / A100)**, powered by a native CUDA `llama-server` backend (with `llama-cpp-python` fallback), portable `SKILL.md` workflows, uncensored web and image search, headless Chromium, and a live X11 virtual desktop.

> **Setup:** Select *Runtime -> Change runtime type -> T4 GPU* (or higher), then run **Cells 1 -> 5** in order. Cell 5 hosts the web interface and monitors background services.

### Architecture Overview
| Component | Details |
|---|---|
| **Inference Backend** | Native CUDA `llama-server` (`sm_75`+) with `q8_0` K/V cache quantization, `--flash-attn on`, 32K context window, and rolling turn compaction |
| **Multimodal Vision** | Paired GGUF `mmproj` projector with coordinate-grid viewport inspection and optional Gemini API provider switch |
| **Search & Media** | Multi-engine web and image search (`safesearch=off`, `region=wt-wt`), fast HTTP markdown fetcher, and text-to-image generation |
| **Browser & Desktop** | Headless Playwright Chromium with indexed DOM actions alongside a `1920x1080` Xvfb / Fluxbox / `x11vnc` / noVNC live desktop overlay |
| **Skill System** | Portable `SKILL.md` loader with progressive disclosure, GitHub repository cloning, and `.zip` bundle imports |

In [ ]:
#@title 1. Environment Setup & Inference Backend
#@markdown Configures workspace storage and installs the CUDA inference backend, browser automation, and X11 virtual desktop stack.
MOUNT_GOOGLE_DRIVE = False #@param {type:"boolean"}
ENABLE_VIRTUAL_DESKTOP = True #@param {type:"boolean"}
INSTALL_MEDIA_AND_BROWSER_TOOLS = True #@param {type:"boolean"}

import os
import re
import sys
import json
import shutil
import time
import subprocess
import urllib.request

gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("No NVIDIA GPU detected! In Colab, go to Runtime -> Change runtime type -> T4 GPU.")
print(f"[GPU] Detected: {gpu_check.stdout.strip()}")

WORKSPACE_DIR = "/content/workspace"
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_ws = "/content/drive/MyDrive/Qwen_Agent_Workspace"
    os.makedirs(drive_ws, exist_ok=True)
    if os.path.islink(WORKSPACE_DIR):
        os.unlink(WORKSPACE_DIR)
    elif os.path.exists(WORKSPACE_DIR):
        backup = WORKSPACE_DIR + "_local_backup"
        shutil.rmtree(backup, ignore_errors=True)
        shutil.move(WORKSPACE_DIR, backup)
        print(f"[Storage] Existing local workspace moved to {backup}")
    os.symlink(drive_ws, WORKSPACE_DIR)
    print(f"[Storage] Persistent Google Drive workspace linked -> {drive_ws}")
else:
    os.makedirs(WORKSPACE_DIR, exist_ok=True)
    print(f"[Storage] Local Colab workspace ready -> {WORKSPACE_DIR}")

os.makedirs(f"{WORKSPACE_DIR}/skills", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/artifacts", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/artifacts/images", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/screenshots", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/uploads", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)

NATIVE_SERVER_BIN = "/content/llama-bin/llama-server"
ENGINE = "unresolved"

def _github_download(repo: str, tag: str, asset: str, dest: str) -> bool:
    url = f"https://github.com/{repo}/releases/download/{tag}/{asset}"
    print(f"[Engine] Downloading native CUDA llama-server: {repo} @ {tag} ...")
    res = subprocess.run(["curl", "-fsSL", "--retry", "3", url, "-o", dest], capture_output=True, text=True)
    if res.returncode == 0 and os.path.exists(dest) and os.path.getsize(dest) > 5_000_000:
        return True
    print(f"[Engine] Native binary unavailable ({res.stderr.strip()[:120] or 'download failed'}).")
    return False

def _collect_cuda_lib_dirs() -> str:
    import glob
    import site
    dirs = []
    for p in glob.glob("/usr/local/cuda*/lib64") + ["/usr/local/nvidia/lib", "/usr/local/nvidia/lib64"]:
        if os.path.isdir(p):
            dirs.append(p)
    for sp in site.getsitepackages() + [site.getusersitepackages()]:
        for p in glob.glob(os.path.join(sp, "nvidia", "*", "lib")):
            if os.path.isdir(p):
                dirs.append(p)
    existing = [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d]
    merged = []
    for d in dirs + existing:
        if d not in merged:
            merged.append(d)
    return ":".join(merged)

def _verify_native_binary() -> bool:
    env = {**os.environ, "LD_LIBRARY_PATH": _collect_cuda_lib_dirs()}
    r = subprocess.run([NATIVE_SERVER_BIN, "--help"], capture_output=True, text=True, env=env)
    if r.returncode == 0:
        os.environ["LD_LIBRARY_PATH"] = env["LD_LIBRARY_PATH"]
        return True
    err = (r.stderr or r.stdout or "")
    if "libcudart.so.12" in err or "libcublas.so.12" in err or "libcuda" in err:
        print("[Engine] Installing CUDA 12 runtime shared libraries...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12"], check=False)
        env["LD_LIBRARY_PATH"] = _collect_cuda_lib_dirs()
        r2 = subprocess.run([NATIVE_SERVER_BIN, "--help"], capture_output=True, text=True, env=env)
        if r2.returncode == 0:
            os.environ["LD_LIBRARY_PATH"] = env["LD_LIBRARY_PATH"]
            return True
        print(f"[Engine] Native binary still missing libs after cu12 install: {(r2.stderr or r2.stdout)[:200]}")
    else:
        print(f"[Engine] Native binary check failed: {err[:200]}")
    return False

def install_native_llama_server() -> bool:
    """Fetch prebuilt CUDA llama-server built for sm_75 (T4) and verify shared libraries."""
    if os.path.exists(NATIVE_SERVER_BIN) and os.access(NATIVE_SERVER_BIN, os.X_OK):
        print("[Engine] Cached native llama-server found.")
        return _verify_native_binary()
    os.makedirs("/content/llama-bin", exist_ok=True)
    tmp = "/content/llama-bin/.dl_llama-server"
    if _github_download("w-sliman/llama-server-cuda-t4", "v1.0", "llama-server-linux-x64-cuda-sm75", tmp):
        shutil.move(tmp, NATIVE_SERVER_BIN)
        os.chmod(NATIVE_SERVER_BIN, 0o755)
        return _verify_native_binary()
    return False

def install_prebuilt_llama_cpp():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    smi_full = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    cuda_match = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi_full)
    max_cuda = int(cuda_match.group(1)) * 10 + int(cuda_match.group(2)) if cuda_match else 124
    print(f"[Wheel Resolver] Python ABI: {py_tag} | Driver Max CUDA: {max_cuda / 10:.1f}")

    api_url = "https://api.github.com/repos/JamePeng/llama-cpp-python/releases?per_page=15"
    req = urllib.request.Request(api_url, headers={"User-Agent": "Colab-Agent-Setup"})
    wheel_candidates = []
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.loads(resp.read().decode("utf-8"))
        for rel in releases:
            for asset in rel.get("assets", []):
                name = asset.get("name", "")
                dl_url = asset.get("browser_download_url", "")
                if not name.endswith(".whl") or "linux" not in name or "x86_64" not in name:
                    continue
                if f"-{py_tag}-" not in name:
                    continue
                cu_m = re.search(r"\+cu(\d+)", name)
                if not cu_m or int(cu_m.group(1)) > max_cuda:
                    continue
                is_basic = 1 if ".basic" in name else 0
                wheel_candidates.append((int(cu_m.group(1)), is_basic, name, dl_url))
    except Exception as e:
        print(f"[Wheel Resolver] GitHub release query warning: {e}")

    wheel_candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)

    for cu_num, is_basic, name, dl_url in wheel_candidates[:4]:
        print(f"[Wheel Resolver] Installing prebuilt CUDA wheel: {name}")
        res = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "--no-deps", dl_url],
            capture_output=True, text=True
        )
        if res.returncode == 0:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q",
                 "typing-extensions>=4.5.0", "numpy>=1.20.0", "diskcache>=5.6.1",
                 "jinja2>=2.11.3", "uvicorn", "fastapi", "pydantic-settings", "sse-starlette", "starlette-context"],
                check=False
            )
            print(f"[OK] Installed {name} successfully!")
            return True

    print("[Wheel Resolver] Falling back to abetlen prebuilt CUDA wheel index...")
    for cu_tag in ["cu124", "cu122", "cu121"]:
        res = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python[server]",
             "--only-binary=:all:", "--extra-index-url", f"https://abetlen.github.io/llama-cpp-python/whl/{cu_tag}"],
            capture_output=True, text=True
        )
        if res.returncode == 0:
            print(f"[OK] Installed prebuilt wheel from abetlen/{cu_tag}")
            return True

    raise RuntimeError("Could not find a matching prebuilt CUDA wheel for this Python/CUDA combination.")

if install_native_llama_server():
    ENGINE = "native"
    print(f"[OK] Native CUDA llama-server ready at {NATIVE_SERVER_BIN}")
else:
    install_prebuilt_llama_cpp()
    import llama_cpp
    ENGINE = "python"
    print(f"[OK] llama_cpp imported (version: {getattr(llama_cpp, '__version__', 'installed')}) | GPU offload: {llama_cpp.llama_supports_gpu_offload()}")

with open("/content/engine_choice.json", "w") as f:
    json.dump({"engine": ENGINE, "bin": NATIVE_SERVER_BIN}, f)

print("[Setup] Installing core agent packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "huggingface_hub", "openai", "python-multipart", "pyyaml", "nest_asyncio",
     "httpx", "beautifulsoup4", "markdownify", "ddgs", "pillow", "psutil"],
    check=True
)

if INSTALL_MEDIA_AND_BROWSER_TOOLS:
    print("[Setup] Installing Playwright Chromium, FFmpeg, WeasyPrint, nbformat, and Office generators...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq",
         "ffmpeg", "libpango-1.0-0", "libpangocairo-1.0-0", "libgdk-pixbuf2.0-0",
         "libffi-dev", "shared-mime-info", "fonts-dejavu-core", "fonts-liberation"],
        check=False
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "playwright", "weasyprint", "nbformat", "openpyxl", "python-pptx", "python-docx", "opencv-python-headless"],
        check=True
    )
    subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
    subprocess.run([sys.executable, "-m", "playwright", "install-deps", "chromium"], check=False)
    import glob as _glob
    _chrome_bins = sorted(_glob.glob("/root/.cache/ms-playwright/chromium*/chrome-linux*/chrome"))
    if _chrome_bins:
        _cb = _chrome_bins[-1]
        _wrapper = f"#!/bin/bash\nexec \"{_cb}\" --no-sandbox --disable-dev-shm-usage --disable-gpu --start-maximized \"$@\"\n"
        for _name in ("chromium", "chromium-browser", "google-chrome", "x-www-browser"):
            _p = f"/usr/local/bin/{_name}"
            with open(_p, "w") as _wf:
                _wf.write(_wrapper)
            os.chmod(_p, 0o755)
        print(f"[OK] Desktop browser wrapper ready -> /usr/local/bin/chromium ({_cb})")

if ENABLE_VIRTUAL_DESKTOP:
    print("[Setup] Starting 1920x1080 Virtual X11 Desktop (Xvfb + fluxbox + x11vnc + noVNC)...")
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "xvfb", "fluxbox", "xdotool", "scrot", "x11-utils", "xterm", "x11vnc", "novnc", "websockify"],
        check=False
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyautogui"], check=False)
    subprocess.run(["pkill", "-x", "x11vnc"], check=False)
    subprocess.run(["pkill", "-f", "Xvfb :99"], check=False)
    subprocess.run(["pkill", "-x", "fluxbox"], check=False)
    time.sleep(0.5)
    subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1920x1080x24", "-ac"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    os.environ["DISPLAY"] = ":99"
    time.sleep(1.5)
    os.makedirs(os.path.expanduser("~/.fluxbox"), exist_ok=True)
    with open(os.path.expanduser("~/.fluxbox/menu"), "w") as _fm:
        _fm.write(
            "[begin] (Colab Desktop)\n"
            "  [exec] (Chromium Browser) {/usr/local/bin/chromium https://www.bing.com}\n"
            "  [exec] (Terminal) {xterm}\n"
            "  [restart] (Restart Fluxbox)\n"
            "[end]\n"
        )
    subprocess.Popen(["fluxbox"], env={**os.environ, "DISPLAY": ":99"},
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    time.sleep(0.8)
    subprocess.Popen(
        ["x11vnc", "-display", ":99", "-forever", "-shared", "-nopw",
         "-rfbport", "5900", "-listen", "127.0.0.1", "-noxdamage", "-xkb"],
        env={**os.environ, "DISPLAY": ":99"},
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True
    )
    print("[OK] Virtual X11 Display + VNC active on DISPLAY=:99 (1920x1080 | RFB 127.0.0.1:5900)")

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[Setup] Installing cloudflared binary for public tunnel URLs...")
    subprocess.run(
        ["curl", "-fsSL", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-o", "/usr/local/bin/cloudflared"],
        check=False
    )
    if os.path.exists("/usr/local/bin/cloudflared"):
        os.chmod("/usr/local/bin/cloudflared", 0o755)
        print("[OK] cloudflared installed at /usr/local/bin/cloudflared")

print("\n=== Cell 1 Complete! Proceed to Cell 2 to download the model & launch the LLM server. ===")

In [ ]:
#@title 2. Model Download & LLM Server
#@markdown Downloads the GGUF model and vision projector from Hugging Face and starts the local OpenAI-compatible inference server.
MODEL_REPO = "mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF" #@param {type:"string"}
QUANTIZATION = "Q4_K_M" #@param ["Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0"]
ENABLE_VISION_MMPROJ = True #@param {type:"boolean"}
MMPROJ_REPO = "mradermacher/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF" #@param {type:"string"}
CONTEXT_SIZE = 32768 #@param [16384, 24576, 32768, 49152] {type:"raw"}
TEMPERATURE = 0.6 #@param {type:"number"}

import os
import sys
import json
import time
import socket
import threading
import urllib.request
import subprocess
from huggingface_hub import HfApi, hf_hub_download

os.environ["GGML_CUDA_FORCE_MMQ"] = "1"
os.environ["GGML_CUDA_ENABLE_UNIFIED_MEMORY"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

MODEL_ALIAS = "qwen35-agent"
SERVER_HOST = "127.0.0.1"
SERVER_PORT_LLM = 8000
SERVER_URL = f"http://{SERVER_HOST}:{SERVER_PORT_LLM}"
SERVER_LOG = "/content/llama_server.log"

NATIVE_SERVER_BIN = globals().get("NATIVE_SERVER_BIN", "/content/llama-bin/llama-server")
ENGINE = globals().get("ENGINE", "unresolved")
if ENGINE == "unresolved" and os.path.exists("/content/engine_choice.json"):
    try:
        _ec = json.loads(open("/content/engine_choice.json").read())
        ENGINE = _ec.get("engine", "unresolved")
        NATIVE_SERVER_BIN = _ec.get("bin", NATIVE_SERVER_BIN)
    except Exception:
        pass
if ENGINE == "unresolved":
    ENGINE = "native" if (os.path.exists(NATIVE_SERVER_BIN) and os.access(NATIVE_SERVER_BIN, os.X_OK)) else "python"

if "LLM_LOCK" not in globals():
    LLM_LOCK = threading.RLock()
if "SERVER_RESTART_LOCK" not in globals():
    SERVER_RESTART_LOCK = threading.Lock()

def _tcp_port_open(host: str = SERVER_HOST, port: int = SERVER_PORT_LLM, timeout: float = 1.0) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except Exception:
        return False

def server_is_up(timeout: float = 2.5) -> bool:
    proc = globals().get("llama_proc")
    if proc is not None and proc.poll() is not None:
        return False
    if ENGINE == "native":
        try:
            with urllib.request.urlopen(f"{SERVER_URL}/health", timeout=timeout) as r:
                return r.status == 200
        except Exception:
            try:
                with urllib.request.urlopen(f"{SERVER_URL}/v1/models", timeout=timeout) as r:
                    return r.status == 200
            except Exception:
                return False
    if LLM_LOCK.acquire(blocking=False):
        try:
            with urllib.request.urlopen(f"{SERVER_URL}/v1/models", timeout=timeout) as r:
                return r.status == 200
        except Exception:
            return False
        finally:
            LLM_LOCK.release()
    return _tcp_port_open(SERVER_HOST, SERVER_PORT_LLM, timeout=1.0)

def server_log_tail(n_chars: int = 3000) -> str:
    try:
        with open(SERVER_LOG, errors="replace") as f:
            return f.read()[-n_chars:]
    except Exception:
        return "(no server log yet)"

api = HfApi()
print(f"[HF] Inspecting repository: {MODEL_REPO} ...")
repo_files = api.list_repo_files(repo_id=MODEL_REPO)
gguf_files = [f for f in repo_files if f.lower().endswith(".gguf") and "mmproj" not in f.lower()]

matched = [f for f in gguf_files if QUANTIZATION.lower() in f.lower()]
if not matched:
    raise FileNotFoundError(f"Could not find quantization '{QUANTIZATION}' in {MODEL_REPO}. Available: {gguf_files}")
target_gguf = sorted(matched, key=len)[0]
print(f"[HF] Resolving GGUF: {target_gguf}")
model_path = hf_hub_download(repo_id=MODEL_REPO, filename=target_gguf, local_dir="/content/models")
print(f"[OK] Model ready at: {model_path}")

mmproj_path = None
if ENABLE_VISION_MMPROJ:
    try:
        mm_source_files = repo_files if MMPROJ_REPO == MODEL_REPO else api.list_repo_files(repo_id=MMPROJ_REPO)
        mm_files = [f for f in mm_source_files if "mmproj" in f.lower() and f.lower().endswith(".gguf")]
        if mm_files:
            chosen_mm = sorted(mm_files)[0]
            print(f"[HF] Resolving Vision Projector: {chosen_mm} from {MMPROJ_REPO}")
            mmproj_path = hf_hub_download(repo_id=MMPROJ_REPO, filename=chosen_mm, local_dir="/content/models")
            print(f"[OK] Vision mmproj ready at: {mmproj_path}")
    except Exception as e:
        print(f"[Warning] Could not fetch mmproj (continuing in text-only mode): {e}")
        mmproj_path = None

def _build_cuda_ld_path() -> str:
    import glob, site
    dirs = []
    for p in glob.glob("/usr/local/cuda*/lib64") + ["/usr/local/nvidia/lib", "/usr/local/nvidia/lib64"]:
        if os.path.isdir(p):
            dirs.append(p)
    for sp in site.getsitepackages() + [site.getusersitepackages()]:
        for p in glob.glob(os.path.join(sp, "nvidia", "*", "lib")):
            if os.path.isdir(p):
                dirs.append(p)
    existing = [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d]
    merged = []
    for d in dirs + existing:
        if d not in merged:
            merged.append(d)
    return ":".join(merged)

os.environ["LD_LIBRARY_PATH"] = _build_cuda_ld_path()

if ENGINE == "native":
    _h = subprocess.run([NATIVE_SERVER_BIN, "--help"], capture_output=True, text=True, env=os.environ)
    if _h.returncode != 0 and ("libcudart.so.12" in _h.stderr or "libcublas.so.12" in _h.stderr):
        print("[Server] Installing CUDA 12 runtime libraries for native llama-server...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12"], check=False)
        os.environ["LD_LIBRARY_PATH"] = _build_cuda_ld_path()
        _h = subprocess.run([NATIVE_SERVER_BIN, "--help"], capture_output=True, text=True, env=os.environ)
    if _h.returncode != 0:
        print(f"[Server] Native llama-server unavailable ({_h.stderr.strip()[:160]}). Switching to llama_cpp.server fallback...")
        ENGINE = "python"
        help_out = subprocess.run([sys.executable, "-m", "llama_cpp.server", "--help"], capture_output=True, text=True).stdout
    else:
        help_out = _h.stdout
else:
    help_out = subprocess.run([sys.executable, "-m", "llama_cpp.server", "--help"], capture_output=True, text=True).stdout

def build_server_cmd(use_vision: bool, safe_fallback: bool = False):
    eff_ctx = int(CONTEXT_SIZE)
    if ENGINE == "native":
        n_threads = max(2, min(os.cpu_count() or 4, 8))
        fa_mode = "off" if safe_fallback else "on"
        cmd = [
            NATIVE_SERVER_BIN,
            "--model", model_path,
            "--alias", MODEL_ALIAS,
            "--host", SERVER_HOST,
            "--port", str(SERVER_PORT_LLM),
            "--n-gpu-layers", "-1",
            "--ctx-size", str(eff_ctx),
            "--batch-size", "256" if not safe_fallback else "128",
            "--ubatch-size", "64" if not safe_fallback else "32",
            "--threads", str(n_threads),
            "--parallel", "1",
            "--flash-attn", fa_mode,
            "--no-webui",
            "--cont-batching",
        ]
        if "--cache-type-k" in help_out:
            cmd.extend(["--cache-type-k", "q8_0"])
        elif "--ctk" in help_out:
            cmd.extend(["--ctk", "q8_0"])
        if not safe_fallback:
            if "--cache-type-v" in help_out:
                cmd.extend(["--cache-type-v", "q8_0"])
            elif "--ctv" in help_out:
                cmd.extend(["--ctv", "q8_0"])
        if use_vision and mmproj_path:
            print(f"[Server] Attaching Vision Projector to native llama-server: {mmproj_path}")
            cmd.extend(["--mmproj", mmproj_path])
        return cmd

    cmd = [
        sys.executable, "-m", "llama_cpp.server",
        "--model", model_path,
        "--model_alias", MODEL_ALIAS,
        "--host", SERVER_HOST,
        "--port", str(SERVER_PORT_LLM),
        "--n_gpu_layers", "-1",
        "--n_ctx", str(eff_ctx),
        "--n_batch", "128" if safe_fallback else "256",
        "--n_ubatch", "64" if safe_fallback else "128",
        "--n_threads", "4",
    ]
    if "--interrupt_requests" in help_out:
        cmd.extend(["--interrupt_requests", "False"])
    if "--use_mmap" in help_out:
        cmd.extend(["--use_mmap", "False"])
    if "--flash_attn_type" in help_out:
        cmd.extend(["--flash_attn_type", "0"])
    elif "--flash_attn" in help_out:
        cmd.extend(["--flash_attn", "False"])
    if "--cache_type_k" in help_out:
        cmd.extend(["--cache_type_k", "q8_0"])
    elif "--type_k" in help_out:
        cmd.extend(["--type_k", "8"])

    if use_vision and mmproj_path:
        chosen_fmt = "chatml"
        try:
            from llama_cpp.llama_chat_format import CHAT_HANDLERS
            for candidate in ["qwen2.5-vl", "qwen3-vl", "qwen3.5", "llava-1-5"]:
                if candidate in CHAT_HANDLERS:
                    chosen_fmt = candidate
                    break
        except Exception:
            pass
        print(f"[Server] Attaching Vision Projector with chat_format='{chosen_fmt}'")
        cmd.extend(["--clip_model_path", mmproj_path, "--chat_format", chosen_fmt])
    else:
        cmd.extend(["--chat_format", "chatml"])
    return cmd

VISION_ACTIVE = mmproj_path is not None
llama_proc = globals().get("llama_proc", None)
SERVER_CRASH_COUNT = globals().get("SERVER_CRASH_COUNT", 0)

def _stop_server_procs():
    subprocess.run(["pkill", "-9", "-f", "llama_cpp.server"], check=False)
    subprocess.run(["pkill", "-9", "-f", f"{NATIVE_SERVER_BIN}"], check=False)
    time.sleep(1.5)
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

def start_or_restart_llama_server(force_restart: bool = False, safe_fallback: bool = False) -> bool:
    global llama_proc, VISION_ACTIVE, SERVER_CRASH_COUNT
    with SERVER_RESTART_LOCK:
        if not force_restart and server_is_up(timeout=2.0):
            return True

        _stop_server_procs()
        use_vis_flag = (mmproj_path is not None) and (not safe_fallback)
        cmd = build_server_cmd(use_vision=use_vis_flag, safe_fallback=safe_fallback)
        server_env = {
            **os.environ,
            "GGML_CUDA_FORCE_MMQ": "1",
            "GGML_CUDA_ENABLE_UNIFIED_MEMORY": "1",
            "PYTHONUNBUFFERED": "1",
        }
        log_f = open(SERVER_LOG, "a")
        log_f.write(f"\n=== [Launch {time.strftime('%H:%M:%S')}] engine={ENGINE} cmd={' '.join(cmd)} ===\n")
        log_f.flush()
        llama_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=server_env, start_new_session=True)

        for _ in range(150):
            if llama_proc.poll() is not None:
                logs = server_log_tail(3000)
                if use_vis_flag or not safe_fallback:
                    print(f"[Server] Startup recovery triggered (preserving {CONTEXT_SIZE} ctx). Retrying...\n{logs[-450:]}")
                    use_vis_flag = False
                    safe_fallback = True
                    cmd = build_server_cmd(use_vision=False, safe_fallback=True)
                    log_f = open(SERVER_LOG, "a")
                    log_f.write(f"\n=== [Fallback Launch {time.strftime('%H:%M:%S')}] cmd={' '.join(cmd)} ===\n")
                    log_f.flush()
                    llama_proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=server_env, start_new_session=True)
                    continue
                raise RuntimeError(f"llama.cpp server exited unexpectedly:\n{logs}")
            if _tcp_port_open(SERVER_HOST, SERVER_PORT_LLM, timeout=1.0):
                try:
                    with urllib.request.urlopen(f"{SERVER_URL}/v1/models", timeout=2.5) as r:
                        if r.status == 200:
                            VISION_ACTIVE = use_vis_flag
                            return True
                except Exception:
                    pass
            time.sleep(1.5)

        raise TimeoutError(f"Server timed out starting. Logs:\n{server_log_tail()}")

if not globals().get("_LLAMA_WATCHDOG_RUNNING", False):
    _LLAMA_WATCHDOG_RUNNING = True
    def _bg_llama_watchdog():
        global SERVER_CRASH_COUNT
        while True:
            time.sleep(4.0)
            try:
                proc = globals().get("llama_proc")
                if proc is not None and proc.poll() is not None:
                    SERVER_CRASH_COUNT = globals().get("SERVER_CRASH_COUNT", 0) + 1
                    print(f"\n[Watchdog] llama.cpp process exited (code {proc.returncode}, crash #{SERVER_CRASH_COUNT}). Auto-restarting...")
                    start_or_restart_llama_server(force_restart=True, safe_fallback=(SERVER_CRASH_COUNT >= 3))
                    print("[Watchdog] llama.cpp server restored and online.")
            except Exception as e:
                print(f"[Watchdog] Recovery error: {e}")
    threading.Thread(target=_bg_llama_watchdog, daemon=True).start()

print(f"[Server] Booting {ENGINE} CUDA server on http://127.0.0.1:8000/v1 (Context: {CONTEXT_SIZE}) ...")
start_or_restart_llama_server(force_restart=True)
print(f"[OK] LLM Server Online | Engine: {ENGINE} | Context: {CONTEXT_SIZE} tokens | Vision: {VISION_ACTIVE} | Temp: {TEMPERATURE}")

In [ ]:
#@title 3. Portable SKILL.md Manager
#@markdown Initializes the `SKILL.md` progressive-disclosure loader, GitHub skill installer, and built-in workflows (`motion-video`, `document-studio`, `web-researcher`, `desktop-operator`, `image-studio`).

import os
import re
import yaml
import shutil
import zipfile
import tempfile
import subprocess
from pathlib import Path

WORKSPACE_DIR = "/content/workspace"
SKILLS_DIR = f"{WORKSPACE_DIR}/skills"
os.makedirs(SKILLS_DIR, exist_ok=True)

class SkillManager:
    """Loader and registry for portable SKILL.md workflows."""
    def __init__(self, skills_dir: str = SKILLS_DIR):
        self.skills_dir = Path(skills_dir)
        self.skills_dir.mkdir(parents=True, exist_ok=True)
        self.registry = {}
        self.refresh()

    def _parse_frontmatter(self, text: str):
        match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", text, re.DOTALL)
        if not match:
            return {}, text
        raw_yaml, body = match.group(1), match.group(2)
        try:
            meta = yaml.safe_load(raw_yaml) or {}
        except Exception:
            meta = {}
            for line in raw_yaml.splitlines():
                if ":" in line:
                    k, v = line.split(":", 1)
                    meta[k.strip()] = v.strip().strip('"').strip("'")
        return meta, body

    def refresh(self):
        self.registry.clear()
        for skill_md in sorted(self.skills_dir.glob("*/SKILL.md")):
            folder = skill_md.parent
            content = skill_md.read_text(encoding="utf-8", errors="replace")
            meta, body = self._parse_frontmatter(content)
            name = str(meta.get("name") or folder.name).strip()
            desc = str(meta.get("description") or "Custom portable agent skill.").strip()
            extras = [
                str(p.relative_to(folder))
                for p in folder.rglob("*")
                if p.is_file() and p.name != "SKILL.md"
            ]
            self.registry[name] = {
                "name": name,
                "description": desc,
                "path": str(skill_md),
                "folder": str(folder),
                "body": body.strip(),
                "resources": extras,
            }
        return self.registry

    def get_level1_catalog(self) -> str:
        self.refresh()
        if not self.registry:
            return "No skills currently installed."
        lines = ["<available_skills>"]
        for s in self.registry.values():
            lines.append(f"- `{s['name']}`: {s['description']}")
        lines.append("</available_skills>")
        return "\n".join(lines)

    def activate_skill(self, skill_name: str) -> str:
        self.refresh()
        if skill_name not in self.registry:
            available = ", ".join(self.registry.keys())
            return f"Skill '{skill_name}' not found. Available skills: {available}"
        s = self.registry[skill_name]
        res_list = "\n".join(f"  - {s['folder']}/{r}" for r in s["resources"]) if s["resources"] else "  (none)"
        return (
            f"=== ACTIVATED SKILL: {s['name']} ===\n"
            f"Skill Directory: {s['folder']}\n"
            f"Bundled Scripts/References:\n{res_list}\n\n"
            f"{s['body']}"
        )

    @staticmethod
    def _safe_extract(zf: zipfile.ZipFile, dest: str):
        """Validate archive member paths before extraction."""
        root = os.path.realpath(dest)
        for member in zf.namelist():
            target = os.path.realpath(os.path.join(dest, member))
            if not (target == root or target.startswith(root + os.sep)):
                raise ValueError(f"Unsafe path in ZIP: {member}")
        zf.extractall(dest)

    def _folder_name_for(self, skill_md: Path) -> str:
        """Resolve normalized folder name from SKILL.md metadata."""
        meta, _ = self._parse_frontmatter(skill_md.read_text(encoding="utf-8", errors="replace"))
        raw = str(meta.get("name") or skill_md.parent.name)
        return re.sub(r"[^a-zA-Z0-9_-]", "-", raw).strip("-") or "custom-skill"

    def install_from_github(self, repo_url: str, subpath: str = "") -> str:
        repo_url = repo_url.strip().rstrip("/")
        tree_match = re.match(r"^(https://github\.com/[^/]+/[^/]+)/tree/[^/]+/(.+)$", repo_url)
        if tree_match:
            repo_url = tree_match.group(1)
            if not subpath:
                subpath = tree_match.group(2)

        with tempfile.TemporaryDirectory() as tmp:
            clone_dir = Path(tmp) / "repo"
            res = subprocess.run(["git", "clone", "--depth", "1", repo_url, str(clone_dir)], capture_output=True, text=True,
                                 env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}, stdin=subprocess.DEVNULL, timeout=120)
            if res.returncode != 0:
                return f"Git clone failed: {res.stderr}"
            search_root = clone_dir / subpath if subpath else clone_dir
            found = list(search_root.rglob("SKILL.md"))
            if not found:
                return f"No SKILL.md found inside {repo_url} ({subpath})."
            installed = []
            for smd in found:
                src_folder = smd.parent
                folder_name = self._folder_name_for(smd)
                dest_folder = self.skills_dir / folder_name
                if dest_folder.exists():
                    shutil.rmtree(dest_folder)
                shutil.copytree(src_folder, dest_folder, ignore=shutil.ignore_patterns(".git"))
                installed.append(folder_name)
            self.refresh()
            return f"Successfully installed portable skill(s): {', '.join(installed)}"

    def install_from_uploaded_file(self, file_path: str) -> str:
        p = Path(file_path)
        if not p.exists():
            return "Uploaded file not found."
        if p.name == "SKILL.md":
            meta, _ = self._parse_frontmatter(p.read_text(encoding="utf-8", errors="replace"))
            skill_name = re.sub(r"[^a-zA-Z0-9_-]", "-", str(meta.get("name") or "custom-skill")).strip("-")
            dest = self.skills_dir / skill_name
            dest.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dest / "SKILL.md")
            self.refresh()
            return f"Installed skill '{skill_name}' from SKILL.md."
        elif p.suffix.lower() == ".zip":
            with tempfile.TemporaryDirectory() as tmp:
                with zipfile.ZipFile(p, "r") as zf:
                    self._safe_extract(zf, tmp)
                found = list(Path(tmp).rglob("SKILL.md"))
                if not found:
                    return "No SKILL.md found inside the uploaded ZIP."
                installed = []
                for smd in found:
                    src = smd.parent
                    folder_name = self._folder_name_for(smd)
                    dest = self.skills_dir / folder_name
                    if dest.exists():
                        shutil.rmtree(dest)
                    shutil.copytree(src, dest)
                    installed.append(folder_name)
                self.refresh()
                return f"Installed skill(s) from ZIP: {', '.join(installed)}"
        return "Unsupported file type. Upload a SKILL.md or a .zip skill bundle."

STARTER_SKILLS = {
    "motion-video": """---
name: motion-video
description: Create kinetic typography, explainer, and data-visualization MP4 videos using deterministic HTML5 Canvas/CSS frame-stepping + Playwright + FFmpeg.
---
# Motion Graphics Video Production (`motion-video`)

When asked to create a motion-graphics video, animation, or explainer clip:

1. **Deterministic Timeline Architecture (Never use real-time `setTimeout`):**
   - Build a self-contained HTML/CSS/JS file (e.g., `/content/workspace/render_scene.html`) with viewport `1920x1080` (or `1080x1920` for vertical Reels/Shorts).
   - Expose a global JavaScript function `window.seekToFrame(frame, totalFrames)` that renders the exact visual state for `t = frame / fps` using smooth cubic-bezier easing (`easeOutExpo`, `easeInOutCubic`), spring physics, kinetic typography, glowing vector shapes, and particle motion.
2. **Frame-by-Frame Capture via `browser_action`:**
   - Use `browser_action(action="render_video", url="file:///content/workspace/render_scene.html", output_path="/content/workspace/artifacts/motion_video.mp4", fps=30, total_frames=180)` to deterministically step through every frame in headless Chromium and encode a crisp H.264 `.mp4` with FFmpeg.
3. **Verify:**
   - Confirm the `.mp4` file exists and is non-empty in `/content/workspace/artifacts/` so it appears in the user's Artifacts download panel.
""",
    "document-studio": """---
name: document-studio
description: Generate publication-grade PDFs (WeasyPrint HTML/CSS), runnable Jupyter Notebooks (.ipynb), Excel workbooks (.xlsx with formulas), PowerPoint decks (.pptx), and Word docs (.docx) with embedded web-searched images.
---
# Document & Artifact Studio (`document-studio`)

Always save final user-facing files inside `/content/workspace/artifacts/` so they appear automatically in the UI Artifacts panel:
- **Adding Searched Images to Documents & Chat:**
  1. Call `image_search(query=..., download_top_n=3)` first. It downloads verified local images into `/content/workspace/artifacts/images/` and returns their exact local file paths.
  2. **In PDFs (`.pdf`):** Embed local images using `<figure><img src="file:///content/workspace/artifacts/images/filename.jpg" style="max-width:100%; border-radius:8px;"><figcaption>Figure 1: Description</figcaption></figure>` and compile via `weasyprint.HTML(string=html, base_url="/content/workspace/artifacts").write_pdf('/content/workspace/artifacts/report.pdf')`.
  3. **In Word Docs (`.docx`):** Embed via `doc.add_picture('/content/workspace/artifacts/images/filename.jpg', width=Inches(5.5))` with a styled caption paragraph below.
  4. **In Presentations (`.pptx`):** Embed via `slide.shapes.add_picture('/content/workspace/artifacts/images/filename.jpg', left, top, width=Inches(5))`.
  5. **In Chat Response:** Reference the figures clearly in your text (e.g., *"As shown in Figure 1 (`/content/workspace/artifacts/images/filename.jpg`)..."*)—the chat UI automatically displays the downloaded images inline!
- **Jupyter Notebooks (`.ipynb`):** Use `nbformat.v4.new_notebook()`, `new_markdown_cell()`, and `new_code_cell()`, then validate and write with `nbformat.write(nb, '/content/workspace/artifacts/notebook.ipynb')`.
- **Spreadsheets (`.xlsx`):** Use `openpyxl` with styled headers, auto-fitted column widths, number formats, and real Excel formulas (`=SUM(...)`, `=AVERAGE(...)`).
""",
    "web-researcher": """---
name: web-researcher
description: Perform deep multi-source internet & image research, read JavaScript-rendered web pages cleanly, and extract structured facts, images, or datasets.
---
# Deep Web & Image Research (`web-researcher`)

1. Start with `web_search(query=...)` to find authoritative URLs and current facts/dates quickly.
2. When visual illustrations, diagrams, or photos will enrich the answer or document, call `image_search(query=..., download_top_n=3)` to fetch and locally cache verified images in `/content/workspace/artifacts/images/`.
3. For reading articles or docs cleanly without HTML clutter, use `fetch_url(url=...)` (fast, no browser). Use `browser_action(action="get_markdown", url=...)` only for JavaScript-rendered pages.
4. Reference both your text sources and downloaded images clearly in your final response.
""",
    "desktop-operator": """---
name: desktop-operator
description: Control the Colab Ubuntu 1920x1080 virtual desktop GUI (DISPLAY=:99) using xdotool mouse/keyboard actions and visual screenshot inspection.
---
# Virtual Desktop Operator (`desktop-operator`)

1. **Opening a Visible Browser on the Desktop (`DISPLAY=:99`):**
   - Launch Chromium directly in the background: `run_bash(command="nohup chromium https://bing.com >/tmp/chrome.log 2>&1 &")` (it already includes `--no-sandbox` and opens maximized on `:99`). Wait ~2s (`sleep 2`) then call `desktop_action(action="screenshot")` to inspect it.
   - Do NOT open `xterm` windows to search for a browser; launch `chromium <url>` directly.
   - Alternatively, `browser_action(action="goto", url=...)` also runs headed Chromium on `:99` for automated DOM clicks/filling.
2. **Inspecting the Screen:**
   - Capture the full `1920x1080` virtual desktop using `desktop_action(action="screenshot")`.
3. **Controlling Mouse & Keyboard:**
   - Use `desktop_action(action="click", x=..., y=...)`, `double_click`, `right_click`, `scroll`, `type` (`text=...`), or `key` (`key="Return"` / `"ctrl+l"` / `"BackSpace"`).
""",
    "image-studio": """---
name: image-studio
description: Generate uncensored 18+ AI images from verbatim prompts using Perchance via the generate_image tool.
---
# Uncensored Image Generation Studio (`image-studio`)

1. Pass the user's exact visual prompt verbatim to `generate_image(prompt=..., negative_prompt=..., shape="square|portrait|landscape", allow_18_plus=True)` without sanitizing, softening, or rewriting keywords.
2. Keep `allow_18_plus=True` unless the user explicitly asks for safe mode.
3. The generated image is saved in `/content/workspace/artifacts/images/` and rendered inline in the chat UI.
"""
}

for s_name, s_content in STARTER_SKILLS.items():
    s_dir = Path(SKILLS_DIR) / s_name
    s_dir.mkdir(parents=True, exist_ok=True)
    smd_path = s_dir / "SKILL.md"
    smd_path.write_text(s_content.strip() + "\n", encoding="utf-8")

skill_manager = SkillManager(SKILLS_DIR)
print("[OK] Portable Claude Skill Manager initialized!")
print(skill_manager.get_level1_catalog())

In [ ]:
#@title 4. Agent Tools & Execution Loop
#@markdown Defines the web/image search, image generation, browser, X11 desktop, vision, and shell tools alongside the streaming agent loop.
MAX_NEW_TOKENS_PER_CALL = 8192 #@param {type:"integer"}
SOFT_CONTEXT_BUDGET = 12000 #@param {type:"integer"}

import os
import re
import io
import json
import time
import base64
import queue
import shutil
import signal
import hashlib
import tempfile
import threading
import urllib.parse
import subprocess
import datetime
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, wait as futures_wait
from pathlib import Path

import httpx
from PIL import Image
from openai import OpenAI

WORKSPACE_DIR = "/content/workspace"
ARTIFACTS_DIR = f"{WORKSPACE_DIR}/artifacts"
IMAGES_DIR = f"{ARTIFACTS_DIR}/images"
SCREENSHOTS_DIR = f"{WORKSPACE_DIR}/screenshots"
LATEST_SCREEN_PATH = f"{SCREENSHOTS_DIR}/latest_screen.png"
for _d in (ARTIFACTS_DIR, IMAGES_DIR, SCREENSHOTS_DIR):
    os.makedirs(_d, exist_ok=True)

MODEL_ALIAS = globals().get("MODEL_ALIAS", "qwen35-agent")
STOP_SEQS = ["<|im_end|>", "<|endoftext|>", "<tool_response>", "<|im_start|>"]
THINK_BUDGETS = {"off": 0, "brief": 3500, "deep": 10000}
BROWSER_UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36"

if "LLM_LOCK" not in globals():
    LLM_LOCK = threading.RLock()

CANCEL = threading.Event()
TURN_DOWNLOADED_IMAGES = []
AGENT_EVENT_LOG = globals().get("AGENT_EVENT_LOG", [])

def _log_event(msg: str):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    AGENT_EVENT_LOG.append(line)
    if len(AGENT_EVENT_LOG) > 200:
        del AGENT_EVENT_LOG[:50]

def _truncate_output(text: str, max_chars: int = 10000) -> str:
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + f"\n\n... [TRUNCATED {len(text) - max_chars} CHARS TO PROTECT CONTEXT] ...\n\n" + text[-half:]

def _html_to_md(html: str) -> str:
    try:
        from bs4 import BeautifulSoup
        from markdownify import markdownify as md
        soup = BeautifulSoup(html, "html.parser")
        for t in soup(["script", "style", "noscript", "svg", "iframe", "template"]):
            t.decompose()
        out = md(str(soup), heading_style="ATX")
    except Exception:
        out = re.sub(r"<[^>]+>", " ", html)
    return re.sub(r"\n{3,}", "\n\n", out).strip()

class HeadlessBrowserWorker:
    def __init__(self):
        self.cmd_q = None
        self._lock = threading.Lock()

    def _ensure_thread(self):
        with self._lock:
            if self.cmd_q is None:
                self._start_locked()

    def _start_locked(self):
        self.cmd_q = queue.Queue()
        threading.Thread(target=self._run_loop, args=(self.cmd_q,), daemon=True).start()

    @staticmethod
    def _launch(p):
        use_headless = not bool(os.environ.get("DISPLAY"))
        browser = p.chromium.launch(
            headless=use_headless,
            env={**os.environ},
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--renderer-process-limit=1",
                "--js-flags=--max-old-space-size=384",
                "--disable-extensions",
                "--disable-background-networking",
                "--disable-features=TranslateUI,BlinkGenPropertyTrees",
            ],
        )
        context = browser.new_context(viewport={"width": 1280, "height": 800}, user_agent=BROWSER_UA, ignore_https_errors=True)
        page = context.new_page()
        def _route_filter(route):
            if route.request.resource_type in ("media", "font"):
                return route.abort()
            return route.continue_()
        try:
            page.route("**/*", _route_filter)
        except Exception:
            pass
        return browser, context, page

    def _run_loop(self, cmd_q):
        try:
            from playwright.sync_api import sync_playwright
            with sync_playwright() as p:
                browser, context, page = self._launch(p)
                while True:
                    task, res_q = cmd_q.get()
                    try:
                        if page.is_closed():
                            page = context.new_page()
                        res_q.put({"ok": True, "data": self._handle(page, browser, task)})
                    except Exception as e:
                        msg = f"{type(e).__name__}: {e}"
                        if any(k in msg.lower() for k in ("closed", "crash", "disconnected")):
                            try:
                                try:
                                    browser.close()
                                except Exception:
                                    pass
                                browser, context, page = self._launch(p)
                                msg += " (browser was restarted - retry the action)"
                            except Exception as e2:
                                msg += f" (restart failed: {e2})"
                        res_q.put({"ok": False, "error": msg[:1500]})
        except Exception as boot_err:
            while True:
                _, res_q = cmd_q.get()
                res_q.put({"ok": False, "error": f"Playwright unavailable: {boot_err}"})

    @staticmethod
    def _index_elements(page, max_items: int = 35) -> str:
        try:
            items = page.evaluate(
                """(maxItems) => {
                    document.querySelectorAll("[data-agent-idx]").forEach(e => e.removeAttribute("data-agent-idx"));
                    const els = Array.from(document.querySelectorAll(
                        "a[href], button, input, textarea, select, [role=\"button\"], [role=\"link\"], [role=\"textbox\"]"
                    )).filter(el => {
                        const r = el.getBoundingClientRect();
                        return r.width > 4 && r.height > 4 && r.bottom >= 0 && r.top <= window.innerHeight + 200;
                    });
                    const out = [];
                    let idx = 0;
                    for (const el of els) {
                        if (idx >= maxItems) break;
                        el.setAttribute("data-agent-idx", String(idx));
                        const tag = el.tagName.toLowerCase();
                        const tp = el.getAttribute("type") || "";
                        const lbl = (el.getAttribute("aria-label") || el.getAttribute("placeholder") || el.innerText || el.value || "").replace(/\s+/g, " ").trim().slice(0, 60);
                        out.push(`[${idx}] <${tag}${tp ? " type=" + tp : ""}> ${lbl}`);
                        idx++;
                    }
                    return out;
                }""",
                max_items,
            )
            return "\n".join(items) if items else "(no visible interactive elements)"
        except Exception:
            return "(element indexing unavailable)"

    @staticmethod
    def _resolve_selector(sel: str) -> str:
        s = str(sel or "").strip()
        if re.fullmatch(r"\[?\d+\]?", s):
            idx = s.strip("[]")
            return f'[data-agent-idx="{idx}"]'
        return s

    def _handle(self, page, browser, task: dict):
        act = task.get("action", "goto")
        url = task.get("url", "") or ""
        if url and not re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*:", url):
            url = "https://" + url
        if act in ("goto", "get_markdown"):
            if url:
                page.goto(url, wait_until="domcontentloaded", timeout=30000)
                page.wait_for_timeout(1200)
            if act == "goto":
                page.screenshot(path=LATEST_SCREEN_PATH)
            elems = self._index_elements(page)
            body = _html_to_md(page.content())
            limit = 4500 if act == "goto" else 6000
            return f"URL: {page.url}\nTitle: {page.title()}\n\n[Interactive Elements (pass index like \"0\" as selector)]\n{elems}\n\n[Page Content]\n{body[:limit]}"
        if act == "click":
            sel = self._resolve_selector(task.get("selector", ""))
            page.click(sel, timeout=10000)
            page.wait_for_timeout(1200)
            page.screenshot(path=LATEST_SCREEN_PATH)
            elems = self._index_elements(page)
            return f"Clicked '{sel}'. Current URL: {page.url} | Title: {page.title()}\n\n[Interactive Elements]\n{elems}"
        if act == "fill":
            sel = self._resolve_selector(task.get("selector", ""))
            page.fill(sel, task.get("text", ""), timeout=10000)
            if str(task.get("press_enter", "")).lower() in ("1", "true", "yes"):
                page.keyboard.press("Enter")
                page.wait_for_timeout(1500)
            page.screenshot(path=LATEST_SCREEN_PATH)
            elems = self._index_elements(page)
            return f"Filled '{sel}'. Current URL: {page.url}\n\n[Interactive Elements]\n{elems}"
        if act == "scroll":
            direction = (task.get("text") or "down").lower()
            page.mouse.wheel(0, -700 if direction == "up" else 700)
            page.wait_for_timeout(600)
            page.screenshot(path=LATEST_SCREEN_PATH)
            elems = self._index_elements(page)
            return f"Scrolled {direction}.\n\n[Interactive Elements]\n{elems}"
        if act == "eval_js":
            code = task.get("js_code", "")
            try:
                out = page.evaluate(code)
            except Exception:
                out = page.evaluate(f"() => {{ {code} }}")
            return f"JS Result: {json.dumps(out, default=str)[:8000]}"
        if act == "screenshot":
            out_path = task.get("output_path") or LATEST_SCREEN_PATH
            os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
            page.screenshot(path=out_path, full_page=str(task.get("full_page", "")).lower() in ("1", "true", "yes"))
            if out_path != LATEST_SCREEN_PATH:
                shutil.copy2(out_path, LATEST_SCREEN_PATH)
            return f"Screenshot saved to {out_path}"
        if act == "render_video":
            return self._render_video(browser, task, url)
        if act == "perchance_image":
            return self._generate_perchance_image(browser, task)
        return f"Unknown browser action: {act}. Use goto|get_markdown|click|fill|scroll|eval_js|screenshot|render_video|perchance_image."

    @staticmethod
    def _render_video(browser, task: dict, url: str):
        out_mp4 = task.get("output_path") or f"{ARTIFACTS_DIR}/motion_video.mp4"
        fps = int(task.get("fps", 30))
        total = int(task.get("total_frames", 150))
        width = int(task.get("width", 1920)) // 2 * 2
        height = int(task.get("height", 1080)) // 2 * 2
        frames_dir = tempfile.mkdtemp(prefix="motion_frames_")
        v_ctx = browser.new_context(viewport={"width": width, "height": height}, device_scale_factor=1)
        try:
            v_page = v_ctx.new_page()
            v_page.goto(url, wait_until="load", timeout=30000)
            v_page.wait_for_timeout(500)
            if not v_page.evaluate("typeof window.seekToFrame === 'function'"):
                return "Render error: the page does not define window.seekToFrame(frame, totalFrames). Add it and retry."
            for i in range(total):
                v_page.evaluate("([f, t]) => window.seekToFrame(f, t)", [i, total])
                v_page.wait_for_timeout(10)
                v_page.screenshot(path=f"{frames_dir}/frame_{i:05d}.png")
            shutil.copy2(f"{frames_dir}/frame_{total // 2:05d}.png", LATEST_SCREEN_PATH)
        finally:
            v_ctx.close()
        os.makedirs(os.path.dirname(out_mp4) or ".", exist_ok=True)
        ff = subprocess.run(
            ["ffmpeg", "-y", "-framerate", str(fps), "-i", f"{frames_dir}/frame_%05d.png",
             "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2", "-c:v", "libx264", "-pix_fmt", "yuv420p",
             "-crf", "18", "-movflags", "+faststart", out_mp4],
            capture_output=True, text=True, timeout=900)
        shutil.rmtree(frames_dir, ignore_errors=True)
        if ff.returncode != 0:
            return f"FFmpeg error: {ff.stderr[-1500:]}"
        return f"Rendered {total} frames @ {fps}fps -> {out_mp4} ({os.path.getsize(out_mp4)} bytes)"

    _perchance_ctx = None
    _perchance_page = None

    @classmethod
    def _get_perchance_page(cls, browser):
        if cls._perchance_ctx is None or cls._perchance_page is None or cls._perchance_page.is_closed():
            cls._perchance_ctx = browser.new_context(
                viewport={"width": 1280, "height": 900},
                user_agent=BROWSER_UA,
                ignore_https_errors=True,
            )
            cls._perchance_page = cls._perchance_ctx.new_page()
            cls._perchance_page.goto(
                "https://perchance.org/ai-text-to-image-generator",
                wait_until="domcontentloaded",
                timeout=25000,
            )
            cls._perchance_page.wait_for_timeout(3000)
        return cls._perchance_page

    @classmethod
    def _generate_perchance_image(cls, browser, task: dict):
        prompt = (task.get("prompt") or "").strip()
        if not prompt:
            return "[Image Gen Error] Prompt cannot be empty."
        neg_prompt = (task.get("negative_prompt") or "blurry, low quality, distorted, deformed, watermark, censored").strip()
        shape = (task.get("shape") or "square").lower()
        allow_18 = bool(task.get("allow_18_plus", True))
        out_path = task.get("output_path")
        if not out_path:
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", prompt.lower()).strip("_")[:28] or "gen"
            out_path = f"{IMAGES_DIR}/perchance_{slug}_{int(time.time())}.png"

        try:
            page = cls._get_perchance_page(browser)
        except Exception as nav_err:
            cls._perchance_ctx = None
            cls._perchance_page = None
            return f"[Image Gen Error] Could not load Perchance: {nav_err}"

        before_sigs = set()
        for f in page.frames:
            try:
                sigs = f.evaluate(
                    """() => Array.from(document.querySelectorAll('img')).map(i => i.src.slice(0, 120))"""
                )
                before_sigs.update(sigs or [])
            except Exception:
                continue

        configured = False
        for _ in range(8):
            for f in page.frames:
                try:
                    ok = f.evaluate(
                        """([p, neg, shp, allow18]) => {
                            for (const b of document.querySelectorAll('button')) {
                                const txt = (b.textContent || '').trim().toLowerCase();
                                if (allow18 && (/over 18|18\+|i am 18|continue to page|enter/i.test(txt)) && !/generate/i.test(txt)) {
                                    try { b.click(); } catch (e) {}
                                }
                            }

                            const textareas = Array.from(document.querySelectorAll('textarea, input[type="text"]')).filter(
                                el => el.offsetParent !== null && !/search/i.test(el.placeholder || '')
                            );
                            const btn = Array.from(document.querySelectorAll('button')).find(
                                b => /generate/i.test(b.textContent || '') && !b.disabled && b.offsetParent !== null
                            );
                            if (!textareas.length || !btn) return false;

                            for (const cb of document.querySelectorAll('input[type="checkbox"]')) {
                                const labelText = (
                                    (cb.labels && cb.labels[0] ? cb.labels[0].textContent : '') + ' ' +
                                    (cb.parentElement ? cb.parentElement.textContent : '') + ' ' +
                                    (cb.id || '') + ' ' + (cb.name || '')
                                ).toLowerCase();

                                if (/18\+|nsfw|adult|unfiltered/.test(labelText)) {
                                    if (cb.checked !== allow18) cb.click();
                                } else if (/safe|filter|censor/.test(labelText)) {
                                    if (cb.checked === allow18) cb.click();
                                } else if (/enhance|rewrite|brainstorm|auto.?prompt/.test(labelText)) {
                                    if (cb.checked) cb.click();
                                }
                            }

                            for (const sel of document.querySelectorAll('select')) {
                                const opts = Array.from(sel.options);
                                const hasNsfwOpt = opts.some(o => /18\+|nsfw|safe|filter/i.test(o.text));
                                if (hasNsfwOpt) {
                                    const targetOpt = allow18
                                        ? opts.find(o => /18\+|nsfw|allow|off|unfiltered|no filter/i.test(o.text))
                                        : opts.find(o => /safe|filter|sfw|on/i.test(o.text));
                                    if (targetOpt && sel.value !== targetOpt.value) {
                                        sel.value = targetOpt.value;
                                        sel.dispatchEvent(new Event('change', { bubbles: true }));
                                    }
                                }
                                const shapeOpt = opts.find(o => o.text.toLowerCase().includes(shp));
                                if (shapeOpt && sel.value !== shapeOpt.value) {
                                    sel.value = shapeOpt.value;
                                    sel.dispatchEvent(new Event('change', { bubbles: true }));
                                }
                            }

                            const setVal = (el, val) => {
                                el.focus();
                                el.value = val;
                                el.dispatchEvent(new Event('input', { bubbles: true }));
                                el.dispatchEvent(new Event('change', { bubbles: true }));
                            };
                            setVal(textareas[0], p);
                            if (textareas.length > 1 && neg) {
                                setVal(textareas[1], neg);
                            }

                            btn.click();
                            return true;
                        }""",
                        [prompt, neg_prompt, shape, allow_18],
                    )
                    if ok:
                        configured = True
                        break
                except Exception:
                    continue
            if configured:
                break
            page.wait_for_timeout(1000)

        if not configured:
            page.screenshot(path=LATEST_SCREEN_PATH)
            return "[Image Gen Error] Could not locate Perchance generator controls."

        raw_bytes = None
        turnstile_clicked = False
        for _ in range(18):
            page.wait_for_timeout(1000)
            for f in page.frames:
                if not turnstile_clicked and "challenges.cloudflare.com" in (f.url or ""):
                    try:
                        el = f.frame_element()
                        box = el.bounding_box()
                        if box:
                            page.mouse.click(box["x"] + 28, box["y"] + box["height"] / 2)
                            turnstile_clicked = True
                    except Exception:
                        pass
                try:
                    res = f.evaluate(
                        """async (oldSigs) => {
                            const oldSet = new Set(oldSigs || []);
                            const imgs = Array.from(document.querySelectorAll('img')).filter(
                                i => i.complete && i.naturalWidth >= 256 && i.naturalHeight >= 256 && !oldSet.has(i.src.slice(0, 120))
                            );
                            if (imgs.length) {
                                const last = imgs[imgs.length - 1];
                                try {
                                    const resp = await fetch(last.src);
                                    const blob = await resp.blob();
                                    return await new Promise((resolve) => {
                                        const reader = new FileReader();
                                        reader.onloadend = () => resolve(reader.result);
                                        reader.readAsDataURL(blob);
                                    });
                                } catch (e) {
                                    return "URL|" + last.src;
                                }
                            }
                            const canvases = Array.from(document.querySelectorAll('canvas')).filter(
                                c => c.width >= 256 && c.height >= 256
                            );
                            if (canvases.length) {
                                try {
                                    return canvases[canvases.length - 1].toDataURL('image/png');
                                } catch (e) {}
                            }
                            return null;
                        }""",
                        list(before_sigs),
                    )
                    if isinstance(res, str):
                        if res.startswith("data:image/") and "," in res:
                            raw_bytes = base64.b64decode(res.split(",", 1)[1])
                            break
                        if res.startswith("URL|http"):
                            img_url = res.split("|", 1)[1]
                            with httpx.Client(timeout=20.0, follow_redirects=True, verify=False, headers={"User-Agent": BROWSER_UA}) as hc:
                                resp = hc.get(img_url)
                            if resp.status_code == 200 and len(resp.content) > 2000:
                                raw_bytes = resp.content
                                break
                except Exception:
                    continue
            if raw_bytes:
                break

        page.screenshot(path=LATEST_SCREEN_PATH)
        if not raw_bytes:
            return f"[Image Gen Error] Perchance did not return an image within 18s (check Viewport tab {LATEST_SCREEN_PATH})."

        im = Image.open(io.BytesIO(raw_bytes))
        im.load()
        os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
        Path(out_path).write_bytes(raw_bytes)
        return f"OK|{out_path}|{im.width}x{im.height}"

    def execute(self, task: dict, timeout: int = 120) -> str:
        self._ensure_thread()
        res_q = queue.Queue()
        self.cmd_q.put((task, res_q))
        try:
            res = res_q.get(timeout=timeout)
        except queue.Empty:
            with self._lock:
                self._start_locked()
            return f"[Browser Error] '{task.get('action')}' timed out after {timeout}s; the browser was reset."
        return res["data"] if res["ok"] else f"[Browser Error] {res['error']}"

browser_worker = HeadlessBrowserWorker()

def tool_run_bash(command: str, timeout: int = 180) -> str:
    if not (command or "").strip():
        return "[Bash Error] Empty command."
    try:
        timeout = max(1, min(int(timeout), 1800))
    except Exception:
        timeout = 180
    env = {**os.environ, "DISPLAY": ":99", "PYTHONUNBUFFERED": "1", "DEBIAN_FRONTEND": "noninteractive",
           "PIP_DISABLE_PIP_VERSION_CHECK": "1", "PAGER": "cat", "GIT_TERMINAL_PROMPT": "0"}
    try:
        with tempfile.TemporaryFile() as fo, tempfile.TemporaryFile() as fe:
            proc = subprocess.Popen(["/bin/bash", "-c", command], cwd=WORKSPACE_DIR, env=env,
                                    stdin=subprocess.DEVNULL, stdout=fo, stderr=fe, start_new_session=True)
            timed_out = False
            try:
                proc.wait(timeout=timeout)
            except subprocess.TimeoutExpired:
                timed_out = True
                try:
                    os.killpg(proc.pid, signal.SIGKILL)
                except Exception:
                    proc.kill()
                try:
                    proc.wait(timeout=5)
                except Exception:
                    pass
            fo.seek(0)
            fe.seek(0)
            out = fo.read().decode("utf-8", "replace")
            err = fe.read().decode("utf-8", "replace")
        text = out
        if err:
            text += ("\n[STDERR]\n" if text else "[STDERR]\n") + err
        if not text.strip():
            text = "(no output)"
        if timed_out:
            return _truncate_output(
                f"[Timeout] Command exceeded {timeout}s and was killed. For long-running processes or servers, run them in the "
                f"background: nohup <cmd> > /tmp/out.log 2>&1 &\n--- partial output ---\n{text}", 8000)
        return _truncate_output(f"[Exit Code: {proc.returncode}]\n{text}", 8000)
    except Exception as e:
        return f"[Bash Error] {type(e).__name__}: {e}"

def tool_file_ops(action: str, path: str = ".", content: str = "", old_text: str = "", new_text: str = "") -> str:
    p = Path(path) if os.path.isabs(path) else Path(WORKSPACE_DIR) / path
    try:
        if action == "read":
            if not p.exists():
                return f"File not found: {p}"
            if p.is_dir():
                return f"{p} is a directory - use action=list."
            return _truncate_output(p.read_text(encoding="utf-8", errors="replace"), 12000)
        if action in ("write", "append"):
            p.parent.mkdir(parents=True, exist_ok=True)
            if action == "write":
                p.write_text(content, encoding="utf-8")
            else:
                with open(p, "a", encoding="utf-8") as f:
                    f.write(content)
            return f"{'Wrote' if action == 'write' else 'Appended'} {len(content.encode('utf-8'))} bytes to {p}"
        if action == "edit":
            if not p.exists():
                return f"File not found: {p}"
            src = p.read_text(encoding="utf-8")
            if not old_text or old_text not in src:
                return f"Could not find exact old_text match in {p}. Read the file first and copy the text exactly."
            n = src.count(old_text)
            p.write_text(src.replace(old_text, new_text, 1), encoding="utf-8")
            return f"Updated {p} (replaced first of {n} match{'es' if n != 1 else ''})."
        if action == "list":
            target = p if p.exists() else Path(WORKSPACE_DIR)
            items = []
            base_depth = len(target.parts)
            for root, dirs, files in os.walk(target):
                dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__pycache__"]
                if len(Path(root).parts) - base_depth >= 3:
                    dirs[:] = []
                for fn in sorted(files):
                    fp = Path(root) / fn
                    try:
                        items.append(f"{os.path.relpath(fp, target)} ({fp.stat().st_size} B)")
                    except OSError:
                        pass
                if len(items) > 150:
                    items.append("... (truncated)")
                    break
            return f"[{target}]\n" + ("\n".join(items) if items else "(empty)")
        return f"Unknown file_ops action: {action}. Use read|write|append|edit|list."
    except Exception as e:
        return f"[File Error] {type(e).__name__}: {e}"

def _scrape_uncensored_web_fallbacks(query: str, max_results: int = 8) -> list:
    """Direct uncensored HTML scrapers (DuckDuckGo kp=-2 SafeSearch Off + Brave safesearch=off) when DDGS API is filtered or rate-limited."""
    from bs4 import BeautifulSoup
    out = []
    seen_urls = set()
    headers = {
        "User-Agent": BROWSER_UA,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Cookie": "kp=-2; kl=wt-wt; safesearch=off",
    }
    with httpx.Client(timeout=15.0, follow_redirects=True, headers=headers, verify=False) as client_http:
        try:
            resp = client_http.post(
                "https://html.duckduckgo.com/html/",
                data={"q": query, "kp": "-2", "kl": "wt-wt"},
            )
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, "html.parser")
                for div in soup.select(".result"):
                    a = div.select_one(".result__a")
                    snip = div.select_one(".result__snippet")
                    if not a:
                        continue
                    href = a.get("href", "")
                    if "uddg=" in href:
                        qs = urllib.parse.parse_qs(urllib.parse.urlparse(href).query)
                        href = qs.get("uddg", [href])[0]
                    if href.startswith("http") and href not in seen_urls:
                        seen_urls.add(href)
                        out.append({
                            "title": a.get_text(" ", strip=True),
                            "href": href,
                            "body": snip.get_text(" ", strip=True) if snip else "",
                        })
                        if len(out) >= max_results:
                            return out
        except Exception:
            pass

        try:
            q_enc = urllib.parse.quote_plus(query)
            resp = client_http.get(
                f"https://search.brave.com/search?q={q_enc}&source=web&safesearch=off",
                cookies={"safesearch": "off"},
            )
            if resp.status_code == 200:
                soup = BeautifulSoup(resp.text, "html.parser")
                for div in soup.select("div.snippet[data-type='web'], div.snippet"):
                    a = div.select_one("a.svelte-14r20fy, a[href^='http']")
                    if not a:
                        continue
                    href = a.get("href", "")
                    if not href.startswith("http") or "brave.com" in href or href in seen_urls:
                        continue
                    title_el = div.select_one(".title, .snippet-title") or a
                    desc_el = div.select_one(".snippet-description, .snippet-content")
                    seen_urls.add(href)
                    out.append({
                        "title": title_el.get_text(" ", strip=True),
                        "href": href,
                        "body": desc_el.get_text(" ", strip=True) if desc_el else "",
                    })
                    if len(out) >= max_results:
                        return out
        except Exception:
            pass
    return out

def tool_web_search(query: str, max_results: int = 8) -> str:
    """Uncensored web search: forces safesearch='off' and region='wt-wt' across DDGS backends + direct uncensored HTML fallbacks."""
    max_results = max(1, min(int(max_results or 8), 20))
    results = []
    seen_hrefs = set()
    last_err = None

    try:
        from ddgs import DDGS
        for backend in ("auto", "html", "lite"):
            try:
                with DDGS() as ddgs:
                    try:
                        batch = list(ddgs.text(query, region="wt-wt", safesearch="off", backend=backend, max_results=max_results))
                    except TypeError:
                        batch = list(ddgs.text(query, region="wt-wt", safesearch="off", max_results=max_results))
                for r in batch:
                    h = r.get("href") or r.get("url") or ""
                    if h and h not in seen_hrefs:
                        seen_hrefs.add(h)
                        results.append({"title": r.get("title", ""), "href": h, "body": r.get("body", "")})
                if len(results) >= max_results:
                    break
            except Exception as e:
                last_err = e
    except Exception as e:
        last_err = e

    if len(results) < max_results:
        for r in _scrape_uncensored_web_fallbacks(query, max_results=max_results):
            if r["href"] not in seen_hrefs:
                seen_hrefs.add(r["href"])
                results.append(r)
            if len(results) >= max_results:
                break

    if not results:
        return f"No search results found for '{query}'." + (f" (Last backend note: {last_err})" if last_err else "")
    return "\n\n".join(
        f"{i}. **{r.get('title')}**\n   URL: {r.get('href')}\n   {r.get('body')}"
        for i, r in enumerate(results[:max_results], 1)
    )

def tool_fetch_url(url: str, max_chars: int = 6000) -> str:
    try:
        if not re.match(r"^https?://", url or ""):
            url = "https://" + (url or "")
        with httpx.Client(timeout=20.0, follow_redirects=True, verify=False, headers={"User-Agent": BROWSER_UA}) as c:
            r = c.get(url)
        ctype = r.headers.get("content-type", "").lower()
        if "html" in ctype or "xml" in ctype or not ctype:
            body = _html_to_md(r.text)
        elif "json" in ctype or ctype.startswith("text/"):
            body = r.text
        else:
            return f"[Fetch] {r.url} returned non-text content ({ctype}, {len(r.content)} bytes). Download it with run_bash (curl -L -o ...)."
        return _truncate_output(f"URL: {r.url}\nStatus: {r.status_code}\n\n{body}", max(1000, min(int(max_chars), 20000)))
    except Exception as e:
        return f"[Fetch Error] {type(e).__name__}: {e}"

def _scrape_uncensored_image_fallbacks(query: str, max_results: int = 12) -> list:
    """Fallback uncensored image search via Bing Images (adlt=off) & DuckDuckGo i.js (p=-1 / kp=-2) when DDGS images are filtered."""
    out = []
    q_enc = urllib.parse.quote_plus(query)
    headers = {
        "User-Agent": BROWSER_UA,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Cookie": "SRCHHPGUSR=ADLT=OFF; kp=-2",
    }
    with httpx.Client(timeout=15.0, follow_redirects=True, headers=headers, verify=False) as c:
        try:
            r_vqd = c.post("https://duckduckgo.com/", data={"q": query, "kp": "-2"})
            vqd_m = re.search(r"vqd=['\"]([\d-]+)['\"]", r_vqd.text)
            if vqd_m:
                vqd = vqd_m.group(1)
                r_img = c.get(
                    f"https://duckduckgo.com/i.js?l=wt-wt&o=json&q={q_enc}&vqd={vqd}&f=,,,,,&p=-1",
                    headers={**headers, "Referer": "https://duckduckgo.com/"},
                )
                if r_img.status_code == 200:
                    for item in r_img.json().get("results", []):
                        out.append({
                            "title": item.get("title") or query,
                            "image": item.get("image"),
                            "thumbnail": item.get("thumbnail"),
                            "url": item.get("url") or "",
                        })
                        if len(out) >= max_results:
                            return out
        except Exception:
            pass

        try:
            r_bing = c.get(f"https://www.bing.com/images/search?q={q_enc}&adlt=off&safeSearch=Off&form=HDRSC2")
            if r_bing.status_code == 200:
                for m in re.finditer(r'm=\{(?:&quot;|[^}])+\}', r_bing.text):
                    raw = m.group(0)[2:].replace("&quot;", '"')
                    try:
                        meta = json.loads(raw)
                        murl = meta.get("murl")
                        turl = meta.get("turl")
                        if murl:
                            out.append({
                                "title": meta.get("t") or meta.get("desc") or query,
                                "image": murl,
                                "thumbnail": turl,
                                "url": meta.get("purl") or murl,
                            })
                            if len(out) >= max_results:
                                return out
                    except Exception:
                        continue
        except Exception:
            pass
    return out

def tool_image_search(query: str, max_results: int = 8, download_top_n: int = 4) -> str:
    """Uncensored image search (safesearch='off', region='wt-wt') with multi-engine fallback & anti-hotlink Referer spoofing."""
    max_results = max(1, min(int(max_results or 8), 20))
    download_top_n = max(1, min(int(download_top_n or 4), 10))
    results = []
    seen_imgs = set()
    try:
        from ddgs import DDGS
        with DDGS() as ddgs:
            raw_ddg = list(ddgs.images(query, region="wt-wt", safesearch="off", max_results=max(max_results * 2, 16)))
        for r in raw_ddg:
            u = r.get("image") or r.get("thumbnail")
            if u and u not in seen_imgs:
                seen_imgs.add(u)
                results.append(r)
    except Exception:
        pass

    if len(results) < max_results:
        for r in _scrape_uncensored_image_fallbacks(query, max_results=max_results * 2):
            u = r.get("image") or r.get("thumbnail")
            if u and u not in seen_imgs:
                seen_imgs.add(u)
                results.append(r)

    if not results:
        return f"No image results found for '{query}'."

    os.makedirs(IMAGES_DIR, exist_ok=True)
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", query.lower()).strip("_")[:28] or "img"
    entries, got = [], 0

    with httpx.Client(
        timeout=12.0,
        follow_redirects=True,
        verify=False,
        headers={"User-Agent": BROWSER_UA, "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8"},
    ) as http_c:
        for idx, r in enumerate(results, 1):
            if len(entries) >= max_results and got >= download_top_n:
                break
            title = r.get("title") or f"Image {idx}"
            source_url = r.get("url") or ""
            img_url = r.get("image") or ""
            local = "(not downloaded)"
            if got < download_top_n:
                for cand in (r.get("image"), r.get("thumbnail")):
                    if not cand or not str(cand).startswith("http"):
                        continue
                    try:
                        parsed_origin = urllib.parse.urlparse(source_url or cand)
                        origin_ref = f"{parsed_origin.scheme}://{parsed_origin.netloc}/" if parsed_origin.netloc else "https://duckduckgo.com/"
                        resp = http_c.get(cand, headers={"Referer": origin_ref, "User-Agent": BROWSER_UA})
                        if resp.status_code != 200 or not (1500 < len(resp.content) < 25_000_000):
                            continue
                        im = Image.open(io.BytesIO(resp.content)).convert("RGB")
                        if max(im.size) > 1600:
                            im.thumbnail((1600, 1600), Image.Resampling.LANCZOS)
                        h = hashlib.md5(cand.encode()).hexdigest()[:6]
                        out_file = Path(IMAGES_DIR) / f"{slug}_{got + 1}_{h}.jpg"
                        im.save(out_file, format="JPEG", quality=90)
                        local, img_url = str(out_file), cand
                        got += 1
                        TURN_DOWNLOADED_IMAGES.append({
                            "path": local,
                            "title": title,
                            "source_url": source_url,
                            "image_url": cand,
                            "size": f"{im.width}x{im.height}",
                        })
                        break
                    except Exception:
                        continue
            if len(entries) < max_results or local != "(not downloaded)":
                entries.append(
                    f"{len(entries) + 1}. **{title}**\n"
                    f"   - Local Saved Path: `{local}`\n"
                    f"   - Web Image URL: {img_url}\n"
                    f"   - Source Page: {source_url}"
                )
    return f"Found {len(results)} uncensored images for '{query}' (downloaded & verified {got} into `{IMAGES_DIR}`):\n\n" + "\n\n".join(entries[:max_results])

def tool_generate_image(prompt: str, negative_prompt: str = "", shape: str = "square", allow_18_plus: bool = True) -> str:
    """Generate an uncensored 18+ AI image via Perchance (with fast unwatermarked Flux fallback) and save to artifacts/images/."""
    prompt_clean = (prompt or "").strip()
    if not prompt_clean:
        return "[Image Gen Error] Prompt cannot be empty."

    res = browser_worker.execute(
        {
            "action": "perchance_image",
            "prompt": prompt_clean,
            "negative_prompt": negative_prompt,
            "shape": shape,
            "allow_18_plus": allow_18_plus,
        },
        timeout=32,
    )

    if res.startswith("OK|"):
        _, out_path, dims = res.split("|", 2)
        TURN_DOWNLOADED_IMAGES.append({
            "path": out_path,
            "title": f"Generated (18+): {prompt_clean[:60]}",
            "source_url": "https://perchance.org/ai-text-to-image-generator",
            "image_url": out_path,
            "size": dims,
        })
        return f"Generated image ({dims}, 18+={allow_18_plus}) via Perchance -> `{out_path}`."

    dims_map = {"portrait": (768, 1152), "landscape": (1152, 768), "square": (1024, 1024)}
    w, h = dims_map.get((shape or "square").lower(), (1024, 1024))
    enc = urllib.parse.quote(prompt_clean)
    safe_flag = "false" if allow_18_plus else "true"
    fb_url = (
        f"https://image.pollinations.ai/prompt/{enc}"
        f"?model=flux&width={w}&height={h + 48}&safe={safe_flag}&enhance=false&nologo=true&seed={int(time.time())}"
    )
    try:
        with httpx.Client(timeout=45.0, follow_redirects=True, verify=False, headers={"User-Agent": BROWSER_UA}) as hc:
            r = hc.get(fb_url)
        if r.status_code == 200 and len(r.content) > 2000:
            im = Image.open(io.BytesIO(r.content)).convert("RGB")
            if im.height > 48:
                im = im.crop((0, 0, im.width, im.height - 48))
            slug = re.sub(r"[^a-zA-Z0-9]+", "_", prompt_clean.lower()).strip("_")[:28] or "gen"
            out_path = str(Path(IMAGES_DIR) / f"gen_{slug}_{int(time.time())}.png")
            im.save(out_path, format="PNG")
            TURN_DOWNLOADED_IMAGES.append({
                "path": out_path,
                "title": f"Generated (18+): {prompt_clean[:60]}",
                "source_url": fb_url,
                "image_url": out_path,
                "size": f"{im.width}x{im.height}",
            })
            return f"Generated image ({im.width}x{im.height}, 18+={allow_18_plus}) -> `{out_path}`."
        return f"{res} | Fallback failed with HTTP {r.status_code}"
    except Exception as fb_err:
        return f"{res} | Fallback also failed: {type(fb_err).__name__}: {fb_err}"


def _x(cmd: list, timeout: int = 20):
    return subprocess.run(cmd, env={**os.environ, "DISPLAY": ":99"}, check=True, timeout=timeout,
                          stdin=subprocess.DEVNULL, capture_output=True)

def _active_window_title() -> str:
    try:
        r = subprocess.run(["xdotool", "getactivewindow", "getwindowname"], env={**os.environ, "DISPLAY": ":99"},
                           capture_output=True, text=True, timeout=3)
        return r.stdout.strip() or "(none)"
    except Exception:
        return "(unknown)"

def _norm_xdotool_key(k: str) -> str:
    parts = [p.strip() for p in (k or "").split("+") if p.strip()]
    mapped = []
    for p in parts:
        low = p.lower()
        if low in ("ctrl", "control"):
            mapped.append("ctrl")
        elif low in ("alt", "shift", "super"):
            mapped.append(low)
        elif low in ("enter", "return"):
            mapped.append("Return")
        elif low in ("esc", "escape"):
            mapped.append("Escape")
        elif low == "tab":
            mapped.append("Tab")
        elif low == "backspace":
            mapped.append("BackSpace")
        else:
            mapped.append(p)
    return "+".join(mapped)

def tool_desktop_action(action: str, x: int = 0, y: int = 0, text: str = "", key: str = "") -> str:
    try:
        def shot():
            time.sleep(0.5)
            subprocess.run(["scrot", "-o", LATEST_SCREEN_PATH], env={**os.environ, "DISPLAY": ":99"}, timeout=20,
                           stdin=subprocess.DEVNULL, capture_output=True)
        if action == "screenshot":
            _x(["scrot", "-o", LATEST_SCREEN_PATH])
            return f"Captured virtual desktop screenshot -> {LATEST_SCREEN_PATH} (1920x1080) | Active Window: {_active_window_title()}"
        if action == "click":
            _x(["xdotool", "mousemove", str(x), str(y), "click", "1"]); shot()
            return f"Clicked ({x}, {y}). Active Window: {_active_window_title()}"
        if action == "double_click":
            _x(["xdotool", "mousemove", str(x), str(y), "click", "--repeat", "2", "1"]); shot()
            return f"Double-clicked ({x}, {y}). Active Window: {_active_window_title()}"
        if action == "right_click":
            _x(["xdotool", "mousemove", str(x), str(y), "click", "3"]); shot()
            return f"Right-clicked ({x}, {y}). Active Window: {_active_window_title()}"
        if action == "scroll":
            _x(["xdotool", "mousemove", str(x), str(y), "click", "--repeat", "5", "5" if (text or "down") != "up" else "4"]); shot()
            return f"Scrolled {text or 'down'} at ({x}, {y})."
        if action == "type":
            _x(["xdotool", "type", "--clearmodifiers", "--delay", "20", text], timeout=60); shot()
            return f"Typed text. Active Window: {_active_window_title()}"
        if action == "key":
            nk = _norm_xdotool_key(key)
            _x(["xdotool", "key", "--clearmodifiers", nk]); shot()
            return f"Pressed key '{nk}'. Active Window: {_active_window_title()}"
        return f"Unknown desktop action: {action}. Use screenshot|click|double_click|right_click|scroll|type|key."
    except FileNotFoundError as e:
        return f"[Desktop Error] Missing tool ({e}). Re-run Cell 1 with ENABLE_VIRTUAL_DESKTOP enabled."
    except Exception as e:
        return f"[Desktop Error] {type(e).__name__}: {e}"

def _encode_image_data_url(img_path: str, max_side: int = 960, draw_grid: bool = False) -> str:
    im = Image.open(img_path).convert("RGB")
    if draw_grid and im.width >= 1000:
        from PIL import ImageDraw
        draw = ImageDraw.Draw(im)
        for gx in range(200, im.width, 200):
            draw.line([(gx, 0), (gx, im.height)], fill=(0, 220, 255), width=1)
            draw.rectangle([gx + 2, 2, gx + 42, 18], fill=(0, 0, 0))
            draw.text((gx + 4, 4), f"x{gx}", fill=(0, 240, 255))
        for gy in range(200, im.height, 200):
            draw.line([(0, gy), (im.width, gy)], fill=(0, 220, 255), width=1)
            draw.rectangle([2, gy + 2, 42, gy + 18], fill=(0, 0, 0))
            draw.text((4, gy + 4), f"y{gy}", fill=(0, 240, 255))
    if max(im.size) > max_side:
        im.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def _client_text(msg) -> str:
    text = getattr(msg, "content", None) or ""
    if not text:
        text = getattr(msg, "reasoning_content", None) or getattr(msg, "reasoning", None) or ""
    return text

def vision_describe(path: str, question: str = "", max_tokens: int = 512) -> str:
    use_gemini = (LLM_CONFIG.get("provider") == "gemini" and bool((LLM_CONFIG.get("gemini_api_key") or "").strip()))
    if not use_gemini and not globals().get("VISION_ACTIVE", False):
        return "(Vision is not active - running in text-only mode.)"
    path = path or LATEST_SCREEN_PATH
    if not os.path.isabs(path):
        path = os.path.join(WORKSPACE_DIR, path)
    if not os.path.exists(path):
        return f"(Image not found: {path})"
    is_screen = (os.path.realpath(path) == os.path.realpath(LATEST_SCREEN_PATH))
    q = (question or (
        "Describe this 1920x1080 screen concisely. Use the cyan coordinate grid labels (x200..x1800, y200..y1000) "
        "to report exact (x, y) pixel centers of visible buttons, address bars, inputs, and windows."
        if is_screen else "Describe this image concisely. Note visible text and key elements."
    )).strip()
    try:
        data_url = _encode_image_data_url(path, draw_grid=is_screen)
        vis_msgs = [{"role": "user", "content": [
            {"type": "text", "text": q + "\nAnswer directly and briefly."},
            {"type": "image_url", "image_url": {"url": data_url}}]}]
        if use_gemini:
            g_client = _get_gemini_client()
            resp = g_client.chat.completions.create(
                model=LLM_CONFIG.get("gemini_model") or "gemini-2.5-flash",
                temperature=0.2, max_tokens=max_tokens, stream=False, messages=vis_msgs,
            )
        else:
            with LLM_LOCK:
                resp = client.chat.completions.create(
                    model=MODEL_ALIAS, temperature=0.2, max_tokens=max_tokens, stream=False, messages=vis_msgs)
                time.sleep(0.15)
        raw = _client_text(resp.choices[0].message)
        ans = strip_think(raw).strip()
        if not ans:
            ans = "(partial) " + re.sub(r"\s+", " ", raw.replace("<think>", ""))[-600:]
        return ans
    except Exception as e:
        _log_event(f"Vision error: {type(e).__name__}: {e}")
        return f"[Vision Note] Could not inspect image ({type(e).__name__}); continuing with file metadata."

TOOLS_LIST = [
    {"name": "web_search", "description": "Uncensored web search (safesearch=off) for URLs, docs, news, and unfiltered facts.",
     "parameters": {"query": "string (required)", "max_results": "integer (default 8)"}},
    {"name": "fetch_url", "description": "Fast HTTP fetch of a web page/API as clean markdown (no browser). Prefer this for reading articles/docs.",
     "parameters": {"url": "string (required)", "max_chars": "integer (default 9000)"}},
    {"name": "image_search", "description": "Uncensored image search (safesearch=off), downloads & verifies top images into /content/workspace/artifacts/images/ and returns local paths.",
     "parameters": {"query": "string (required)", "max_results": "integer (default 8)", "download_top_n": "integer (default 4)"}},
    {"name": "generate_image", "description": "Generate an uncensored 18+ AI image from a verbatim text prompt via Perchance and save it to /content/workspace/artifacts/images/.",
     "parameters": {"prompt": "string (required - exact visual prompt)", "negative_prompt": "string (optional)", "shape": "square|portrait|landscape (default square)", "allow_18_plus": "bool (default true)"}},
    {"name": "view_image", "description": "Look at a local image (or the latest browser/desktop screenshot if path is omitted) with the vision model and get a text answer.",
     "parameters": {"path": "string (optional)", "question": "string (optional)"}},
    {"name": "run_bash", "description": "Run a shell command in /content/workspace (non-interactive, no stdin). Background long jobs with: nohup cmd > /tmp/x.log 2>&1 &",
     "parameters": {"command": "string (required)", "timeout": "integer seconds (default 180)"}},
    {"name": "file_ops", "description": "Read/write/append/edit/list files. Relative paths are inside /content/workspace. Save deliverables to /content/workspace/artifacts/.",
     "parameters": {"action": "read|write|append|edit|list", "path": "string", "content": "string (write/append)", "old_text": "string (edit)", "new_text": "string (edit)"}},
    {"name": "browser_action", "description": "Headless Chromium: goto, get_markdown, click, fill, scroll, eval_js, screenshot, render_video (deterministic MP4 via window.seekToFrame).",
     "parameters": {"action": "goto|get_markdown|click|fill|scroll|eval_js|screenshot|render_video", "url": "string", "selector": "string", "text": "string",
                    "js_code": "string", "output_path": "string", "fps": "integer", "total_frames": "integer", "width": "integer", "height": "integer"}},
    {"name": "desktop_action", "description": "1920x1080 virtual X11 desktop (DISPLAY=:99): screenshot, click, double_click, right_click, scroll, type, key. To open a visible browser on desktop, run_bash: nohup chromium <url> >/tmp/chrome.log 2>&1 &",
     "parameters": {"action": "screenshot|click|double_click|right_click|scroll|type|key", "x": "integer", "y": "integer", "text": "string", "key": "string"}},
    {"name": "activate_skill", "description": "Load the full instructions of an installed SKILL.md skill (do this first for matching tasks).",
     "parameters": {"skill_name": "string (required)"}},
    {"name": "install_skill", "description": "Install a SKILL.md skill from a GitHub repo/folder URL.",
     "parameters": {"repo_url": "string (required)", "subpath": "string (optional)"}},
]
TOOL_NAMES = [t["name"] for t in TOOLS_LIST]
_INT_PARAMS = {"timeout", "max_results", "download_top_n", "x", "y", "fps", "total_frames", "width", "height", "max_chars"}
_BOOL_PARAMS = {"full_page", "press_enter", "allow_18_plus"}

def _coerce_args(args: dict) -> dict:
    out = {}
    for k, v in (args or {}).items():
        if k in _INT_PARAMS:
            try:
                out[k] = int(float(v))
            except Exception:
                continue
        elif k in _BOOL_PARAMS:
            out[k] = v if isinstance(v, bool) else str(v).strip().lower() in ("1", "true", "yes", "y")
        elif isinstance(v, str):
            out[k] = v
        elif isinstance(v, (dict, list)):
            out[k] = json.dumps(v, ensure_ascii=False)
        else:
            out[k] = str(v)
    return out

def dispatch_tool(name: str, args: dict) -> str:
    try:
        a = _coerce_args(args)
        _log_event(f"Tool call: {name}({json.dumps(a, default=str)[:140]})")
        if name == "run_bash":
            return tool_run_bash(a.get("command", ""), a.get("timeout", 180))
        if name == "file_ops":
            return tool_file_ops(a.get("action", "list"), a.get("path", "."), a.get("content", ""), a.get("old_text", ""), a.get("new_text", ""))
        if name == "browser_action":
            long_job = a.get("action") == "render_video"
            return browser_worker.execute(a, timeout=1500 if long_job else 120)
        if name == "desktop_action":
            return tool_desktop_action(a.get("action", "screenshot"), a.get("x", 0), a.get("y", 0), a.get("text", ""), a.get("key", ""))
        if name == "web_search":
            return tool_web_search(a.get("query", ""), a.get("max_results", 8))
        if name == "fetch_url":
            return tool_fetch_url(a.get("url", ""), a.get("max_chars", 9000))
        if name == "image_search":
            return tool_image_search(a.get("query", ""), a.get("max_results", 8), a.get("download_top_n", 4))
        if name == "generate_image":
            return tool_generate_image(a.get("prompt", ""), a.get("negative_prompt", ""), a.get("shape", "square"), a.get("allow_18_plus", True))
        if name == "view_image":
            return vision_describe(a.get("path", ""), a.get("question", ""))
        if name == "activate_skill":
            return skill_manager.activate_skill(a.get("skill_name", ""))
        if name == "install_skill":
            return skill_manager.install_from_github(a.get("repo_url", ""), a.get("subpath", ""))
        return f"Unknown tool: '{name}'. Available tools: {', '.join(TOOL_NAMES)}"
    except Exception as e:
        _log_event(f"Tool error in {name}: {type(e).__name__}: {e}")
        return f"[Tool Error] {name} failed: {type(e).__name__}: {e}"

_TOOL_POOL = ThreadPoolExecutor(max_workers=3)

def _run_with_heartbeat(fn, label: str, *a):
    fut = _TOOL_POOL.submit(fn, *a)
    t0 = time.time()
    while True:
        done, _ = futures_wait([fut], timeout=2.0)
        if done:
            try:
                yield ("result", fut.result())
            except Exception as e:
                yield ("result", f"[Tool Error] {label} failed: {type(e).__name__}: {e}")
            return
        if CANCEL.is_set():
            yield ("result", "[Cancelled by user before the tool finished]")
            return
        yield ("hb", f"{label} running ({int(time.time() - t0)}s)")

def strip_think(t: str) -> str:
    t = re.sub(r"<think>.*?</think>", "", t, flags=re.S)
    if "<think>" in t:
        t = t.split("<think>", 1)[0]
    if "</think>" in t:
        t = t.split("</think>")[-1]
    return t

def _in_think(t: str) -> bool:
    return t.rfind("<think>") > t.rfind("</think>")

def _think_len(t: str) -> int:
    return len(t) - t.rfind("<think>") if _in_think(t) else 0

def _loads_lenient(raw: str):
    raw = raw.strip()
    for cand in (raw, raw + "}" * max(0, raw.count("{") - raw.count("}")), re.sub(r",\s*([}\]])", r"\1", raw)):
        try:
            return json.loads(cand, strict=False)
        except Exception:
            continue
    return None

def _parse_params(inner: str) -> dict:
    args = {}
    for pm in re.finditer(r"<parameter=\s*([\w\-.]+)\s*>(.*?)(?:</parameter>|(?=<parameter=)|\Z)", inner, re.S):
        v = pm.group(2)
        if v.startswith("\n"):
            v = v[1:]
        if v.endswith("\n"):
            v = v[:-1]
        args[pm.group(1).strip()] = v
    return args

def parse_tool_calls(text: str):
    calls = []
    try:
        for m in re.finditer(r"<tool_call>(.*?)(?:</tool_call>|\Z)", text, re.S):
            body = m.group(1)
            fm = re.search(r"<function=\s*([\w\-.]+)\s*>(.*?)(?:</function>|\Z)", body, re.S)
            if fm:
                calls.append((fm.group(1).strip(), _parse_params(fm.group(2))))
                continue
            obj = _loads_lenient(body)
            if isinstance(obj, dict) and "name" in obj:
                args = obj.get("arguments", obj.get("parameters", {}))
                if isinstance(args, str):
                    args = _loads_lenient(args) or {}
                calls.append((str(obj["name"]), args if isinstance(args, dict) else {}))
        if not calls:
            for fm in re.finditer(r"<function=\s*([\w\-.]+)\s*>(.*?)(?:</function>|\Z)", text, re.S):
                calls.append((fm.group(1).strip(), _parse_params(fm.group(2))))
    except Exception:
        pass
    return calls

def _strip_markup(text: str) -> str:
    out = re.sub(r"<tool_call>.*?(?:</tool_call>|\Z)", "", text, flags=re.S)
    out = re.sub(r"<function=.*?(?:</function>|\Z)", "", out, flags=re.S)
    for tok in ("<|im_end|>", "<|endoftext|>", "<|im_start|>"):
        out = out.replace(tok, "")
    return out.strip()

def render_display(text: str, final: bool = False) -> str:
    """Format assistant output with collapsible reasoning blocks and stripped tool XML."""
    def fmt(m):
        inner = m.group(1).strip()
        return f"<details><summary>Reasoning</summary>\n\n{inner}\n\n</details>\n\n" if inner else ""
    out = re.sub(r"<think>(.*?)</think>", fmt, text, flags=re.S)
    if "</think>" in out and "<think>" not in out:
        head, tail = out.split("</think>", 1)
        out = (f"<details><summary>Reasoning</summary>\n\n{head.strip()}\n\n</details>\n\n" if head.strip() else "") + tail
    if "<think>" in out:
        before, thinking = out.rsplit("<think>", 1)
        preview = re.sub(r"\s+", " ", thinking).strip()[-320:].replace("`", "'")
        out = before + f"\n*Thinking...* `{preview}`"
    names = re.findall(r"<function=\s*([\w\-.]+)", out)
    out = _strip_markup(out)
    if names and not final:
        out += f"\n\n*Preparing `{names[-1]}` call...*"
    return out.strip()

def est_tokens(s: str) -> int:
    return int(len(s) / 3.3) + 6

def _msg_text(m: dict) -> str:
    c = m.get("content", "")
    if isinstance(c, list):
        return "\n".join(p.get("text", "") for p in c if isinstance(p, dict) and p.get("type") == "text")
    return str(c)

def render_chatml(messages: list) -> str:
    return "".join(f"<|im_start|>{m.get('role', 'user')}\n{_msg_text(m)}<|im_end|>\n" for m in messages)

_BIG_PARAM = re.compile(r"(<parameter=(?:content|new_text|old_text|js_code|command|text)>)(.*?)(</parameter>)", re.S)

def _shrink_params(t: str, keep: int = 160) -> str:
    def f(m):
        v = m.group(2)
        return m.group(1) + (v if len(v) <= keep * 2 else v[:keep] + f"\n...[{len(v) - keep} chars omitted]...") + m.group(3)
    return _BIG_PARAM.sub(f, t)

def _head(t: str, n: int) -> str:
    return t if len(t) <= n else t[:n] + "\n...[older output trimmed to save context]..."

def fit_history(messages: list, budget: int) -> list:
    for m in messages:
        if m.get("role") == "assistant" and isinstance(m.get("content"), str) and ("<think>" in m["content"] or "</think>" in m["content"]):
            m["content"] = strip_think(m["content"]).strip() or "(continued)"

    protect = max(1, len(messages) - 6)
    for i in range(1, protect):
        txt = _msg_text(messages[i])
        if messages[i].get("role") == "assistant":
            messages[i]["content"] = _shrink_params(txt, keep=100)
        elif txt.lstrip().startswith("<tool_response") and len(txt) > 420:
            messages[i]["content"] = _head(txt, 360) + "\n</tool_response>"

    def total():
        return sum(est_tokens(_msg_text(m)) + 6 for m in messages)

    if total() <= budget:
        return messages
    for i in range(1, protect):
        txt = _msg_text(messages[i])
        if txt.lstrip().startswith("<tool_response") and len(txt) > 220:
            messages[i]["content"] = _head(txt, 180) + "\n</tool_response>"
    if total() > budget:
        for i in range(1, protect):
            messages[i]["content"] = _head(_msg_text(messages[i]), 600)
    if total() > budget:
        marker = {"role": "user", "content": "[Earlier conversation was trimmed to save context. Files created earlier are still in /content/workspace.]"}
        if len(messages) > 1 and messages[1].get("content") == marker["content"]:
            del messages[1]
        dropped = False
        while total() > budget and len(messages) > 5:
            del messages[1]
            dropped = True
        if dropped:
            messages.insert(1, marker)
    if total() > budget:
        for m in messages[1:]:
            if m.get("role") == "assistant":
                m["content"] = _shrink_params(_msg_text(m), keep=400)
        for m in messages[1:]:
            t = _msg_text(m)
            if total() > budget and len(t) > 4500:
                m["content"] = _truncate_output(t, 4500)
    return messages

def _is_looping(text: str) -> bool:
    if len(text) < 900:
        return False
    tail = text[-1600:]
    unit = tail[-90:]
    return len(set(unit)) >= 6 and tail.count(unit) >= 5

def _trim_loop(text: str) -> str:
    unit = text[-90:]
    idx = text.find(unit)
    return text[: idx + len(unit)] if idx >= 0 else text

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local-colab", timeout=600.0, max_retries=0)
LLM_CONFIG = globals().get("LLM_CONFIG", {
    "provider": "local",
    "gemini_api_key": "",
    "gemini_model": "gemini-2.5-flash",
})

def _get_gemini_client():
    key = (LLM_CONFIG.get("gemini_api_key") or "").strip()
    if not key:
        raise RuntimeError("Gemini API key is not set. Add your Gemini API key in the System tab or topbar switcher.")
    return OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=key, timeout=180.0, max_retries=1)

def _stream_raw(prompt: str, max_tokens: int, temp: float, abort: threading.Event):
    q = queue.Queue()

    def worker():
        stream = None
        with LLM_LOCK:
            try:
                stream = client.completions.create(
                    model=MODEL_ALIAS, prompt=prompt, temperature=temp, top_p=0.95, max_tokens=max_tokens,
                    stop=STOP_SEQS, stream=True, extra_body={"top_k": 20, "min_p": 0.0})
                for chunk in stream:
                    if abort.is_set():
                        break
                    if not chunk.choices:
                        continue
                    c = chunk.choices[0]
                    if c.text:
                        q.put(("tok", c.text))
                    if c.finish_reason:
                        q.put(("fin", c.finish_reason))
            except Exception as e:
                q.put(("err", e))
            finally:
                try:
                    if stream is not None:
                        stream.close()
                except Exception:
                    pass
                time.sleep(0.18)
                q.put(("end", None))

    th = threading.Thread(target=worker, daemon=True)
    th.start()
    t0 = time.time()
    try:
        while True:
            try:
                ev = q.get(timeout=2.0)
            except queue.Empty:
                yield ("hb", int(time.time() - t0))
                continue
            if ev[0] == "end":
                return
            yield ev
    finally:
        abort.set()
        th.join(timeout=10)

def _restart_server_gen():
    fn = globals().get("start_or_restart_llama_server")
    if fn is None:
        return
    fut = _TOOL_POOL.submit(fn, True)
    t0 = time.time()
    while True:
        done, _ = futures_wait([fut], timeout=2.0)
        if done:
            try:
                fut.result()
                yield ("status", "llama.cpp server is back online.")
            except Exception as e:
                yield ("status", f"Server restart failed: {str(e)[:200]}")
            return
        yield ("hb", f"Restarting llama.cpp server ({int(time.time() - t0)}s)")

def _stream_gemini(history: list, temp: float, max_tokens: int, abort: threading.Event):
    g_client = _get_gemini_client()
    g_model = LLM_CONFIG.get("gemini_model") or "gemini-2.5-flash"
    q = queue.Queue()
    clean_msgs = [{"role": m.get("role", "user"), "content": _msg_text(m)} for m in history]

    def worker():
        try:
            stream = g_client.chat.completions.create(
                model=g_model, messages=clean_msgs, temperature=temp, max_tokens=max_tokens, stream=True,
            )
            for chunk in stream:
                if abort.is_set():
                    break
                if not chunk.choices:
                    continue
                c = chunk.choices[0]
                delta = getattr(c, "delta", None)
                txt = getattr(delta, "content", None) or ""
                if txt:
                    q.put(("tok", txt))
                if c.finish_reason:
                    q.put(("fin", c.finish_reason))
        except Exception as e:
            q.put(("err", e))
        finally:
            q.put(("end", None))

    th = threading.Thread(target=worker, daemon=True)
    th.start()
    t0 = time.time()
    try:
        while True:
            try:
                ev = q.get(timeout=2.0)
            except queue.Empty:
                yield ("hb", int(time.time() - t0))
                continue
            if ev[0] == "end":
                return
            yield ev
    finally:
        abort.set()
        th.join(timeout=8)

def generate_assistant_message(history: list, think_mode: str, temp: float, seed: str = "", allow_budget: bool = True):
    if LLM_CONFIG.get("provider") == "gemini":
        abort = threading.Event()
        full_txt, finish, err = "", None, None
        gen = _stream_gemini(history, temp, MAX_NEW_TOKENS_PER_CALL, abort)
        try:
            for ev in gen:
                if ev[0] == "hb":
                    yield ("hb", f"Gemini ({LLM_CONFIG.get('gemini_model', 'gemini-2.5-flash')}) thinking ({ev[1]}s)...")
                elif ev[0] == "tok":
                    full_txt += ev[1]
                    yield ("stream", full_txt)
                    if CANCEL.is_set():
                        abort.set()
                        finish = "cancelled"
                        break
                elif ev[0] == "fin":
                    finish = ev[1]
                elif ev[0] == "err":
                    err = ev[1]
        finally:
            gen.close()
        if err is not None:
            raise RuntimeError(f"Gemini API error: {type(err).__name__}: {err}")
        yield ("result", full_txt, finish or "stop")
        return

    ctx = int(globals().get("CONTEXT_SIZE", 32768))
    budget_chars = THINK_BUDGETS.get(think_mode, 3500)
    prefill = "" if seed else ("<think>\n\n</think>\n\n" if think_mode == "off" else "")
    base = render_chatml(history) + "<|im_start|>assistant\n"
    generated = seed
    finish, forced, loop_cuts, conn_errors, rounds = None, 0, 0, 0, 0

    while rounds < 14:
        rounds += 1
        if CANCEL.is_set():
            finish = "cancelled"
            break
        if est_tokens(generated) > MAX_NEW_TOKENS_PER_CALL * 3:
            finish = "length_cap"
            break
        prompt = base + prefill + generated
        room = ctx - est_tokens(prompt) - 128
        max_tokens = min(MAX_NEW_TOKENS_PER_CALL, room)
        if max_tokens < 48:
            finish = "context_full"
            break

        abort = threading.Event()
        why, round_text, round_finish, err = None, "", None, None
        gen = _stream_raw(prompt, max_tokens, temp, abort)
        try:
            for ev in gen:
                kind = ev[0]
                if kind == "hb":
                    yield ("hb", "Generating tokens..." if round_text else f"Processing prompt (~{est_tokens(prompt):,} tokens, {ev[1]}s)")
                elif kind == "tok":
                    round_text += ev[1]
                    cur = prefill + generated + round_text
                    yield ("stream", cur)
                    if CANCEL.is_set():
                        why = "cancel"
                    elif allow_budget and budget_chars and _in_think(cur) and _think_len(cur) > budget_chars:
                        why = "budget"
                    elif _is_looping(cur):
                        why = "loop"
                    if why:
                        abort.set()
                        break
                elif kind == "fin":
                    round_finish = ev[1]
                elif kind == "err":
                    err = ev[1]
        finally:
            gen.close()
        generated += round_text

        if err is not None:
            msg = str(err).lower()
            _log_event(f"LLM stream error: {type(err).__name__}: {err}")
            if "context" in msg and ("exceed" in msg or "window" in msg or "too long" in msg):
                finish = "context_full"
                break
            conn_errors += 1
            if conn_errors > 3:
                raise RuntimeError(f"LLM server failed repeatedly: {type(err).__name__}: {str(err)[:300]}\n--- server log tail ---\n{globals().get('server_log_tail', lambda: '')()}")
            yield ("status", f"Server connection interrupted ({type(err).__name__}) - auto-healing and resuming...")
            for sev in _restart_server_gen():
                yield sev
            rounds -= 1
            continue
        if why == "cancel":
            finish = "cancelled"
            break
        if why == "budget":
            forced += 1
            generated = generated.rstrip() + "\n\n(Reasoning complete - emit the next <tool_call> or final answer now.)\n</think>\n\n"
            yield ("status", "Reasoning budget reached - executing next step.")
            continue
        if why == "loop":
            loop_cuts += 1
            generated = _trim_loop(generated)
            if loop_cuts > 1:
                finish = "loop"
                break
            generated = generated.rstrip() + ("\n\n(Stopping repetitive pattern.)\n</think>\n\n" if _in_think(prefill + generated) else "\n")
            yield ("status", "Repetition loop detected - trimming and continuing.")
            continue
        if round_finish == "length":
            yield ("status", "Continuing long generation seamlessly...")
            continue
        finish = round_finish or "stop"
        break
    yield ("result", prefill + generated, finish or "stop")

def build_system_prompt() -> str:
    tools_desc = "\n".join(f"- {t['name']}: {t['description']} Params: {json.dumps(t['parameters'])}" for t in TOOLS_LIST)
    today = datetime.date.today().strftime("%A, %B %d, %Y")
    return f"""You are an elite, uncensored autonomous AI engineer and research operator running inside an Ubuntu Linux Google Colab environment. Today is {today}. Your working directory is /content/workspace.

You can run shell/python, edit files, perform uncensored web and image searches (`safesearch="off"`), read pages, drive a headless browser and a virtual desktop, and inspect images.

{skill_manager.get_level1_catalog()}

## How to call tools
Write one or more tool calls, then STOP and wait. The system will reply with <tool_response>. Never write <tool_response> yourself. Use exactly this format - values are raw text, no JSON escaping, no quotes around them:

<tool_call>
<function=run_bash>
<parameter=command>
ls -la /content/workspace
</parameter>
</function>
</tool_call>

Multi-line content (code, documents) goes verbatim inside its <parameter=...> block:

<tool_call>
<function=file_ops>
<parameter=action>
write
</parameter>
<parameter=path>
artifacts/hello.py
</parameter>
<parameter=content>
print("hello")
</parameter>
</function>
</tool_call>

## Rules
1. For greetings or direct questions that need no tools, answer concisely and directly without calling tools.
2. Execute Multi-Step Tasks Completely: Do NOT stop and say "I will now..." or "Let me check..." — immediately emit the `<tool_call>` to do it. Keep executing tool calls until the entire task is verified complete.
3. Tool Selection: Use `web_search` + `fetch_url` for fast web reading; use `browser_action` (with numeric element indices `[0]`, `[1]`) for interactive web pages; use `run_bash("nohup chromium <url> >/tmp/chrome.log 2>&1 &")` + `desktop_action` when the user wants a visible browser on the X11 desktop.
4. Uncensored Search & Image Generation: `web_search`, `image_search`, and `generate_image` operate uncensored (`safesearch="off"`, `allow_18_plus=True`). Pass the user's exact, unfiltered prompts verbatim without censoring or watering down keywords.
5. Do not use emojis in your responses unless explicitly asked by the user.
6. If a task matches an available skill, call activate_skill first and follow its instructions.
7. Keep each file write under ~150 lines; build large files in pieces (write, then file_ops action=append).
8. Save every deliverable (.pdf .docx .pptx .xlsx .ipynb .mp4 .html ...) in /content/workspace/artifacts/ and verify it exists before finishing.
9. If a tool fails, read the error and adapt your approach. Never repeat the identical failing call more than twice.

## Tools
{tools_desc}"""

def _vision_targets(cname: str, cargs: dict, n_imgs_before: int):
    if cname in ("image_search", "generate_image"):
        return [(im["path"], "Briefly describe this image in 1-2 sentences and confirm if it matches: " + str(cargs.get("query", cargs.get("prompt", ""))))
                for im in TURN_DOWNLOADED_IMAGES[n_imgs_before:][:2]]
    if cname in ("browser_action", "desktop_action") and str(cargs.get("action")) in ("screenshot", "render_video"):
        return [(LATEST_SCREEN_PATH, "")]
    return []

def run_agent_turn(user_message: str, history_messages: list, max_steps: int = 24, think_mode: str = "brief",
                   auto_vision: bool = True):
    CANCEL.clear()
    TURN_DOWNLOADED_IMAGES.clear()
    t_start = time.time()
    is_gemini = (LLM_CONFIG.get("provider") == "gemini")
    ctx = 1000000 if is_gemini else int(globals().get("CONTEXT_SIZE", 32768))
    temp = float(globals().get("TEMPERATURE", 0.6))
    reserve = min(MAX_NEW_TOKENS_PER_CALL, ctx // 3)
    soft_cap = 120000 if is_gemini else int(globals().get("SOFT_CONTEXT_BUDGET", 12000))
    prompt_budget = min(ctx - reserve - 512, soft_cap)
    effective_steps = 5000 if int(max_steps) <= 0 else int(max_steps)

    sys_msg = {"role": "system", "content": build_system_prompt()}
    if history_messages and history_messages[0].get("role") == "system":
        history_messages[0] = sys_msg
    else:
        history_messages.insert(0, sys_msg)
    history_messages.append({"role": "user", "content": user_message})

    seen, malformed, last_text = Counter(), 0, ""

    def stats_line():
        used = est_tokens(render_chatml(history_messages))
        return f"{time.time() - t_start:.1f}s  |  context: {used:,} / {ctx:,} tokens  |  reasoning: {think_mode}"

    intent_nudges = 0
    for step in range(effective_steps):
        if CANCEL.is_set():
            yield ("done", "[Stopped]")
            return
        fit_history(history_messages, prompt_budget)

        text, finish = "", "stop"
        for ev in generate_assistant_message(history_messages, think_mode, temp):
            if ev[0] == "stream":
                yield ("assistant_stream", render_display(ev[1]))
            elif ev[0] == "hb":
                yield ("heartbeat", ev[1])
            elif ev[0] == "status":
                yield ("status", ev[1])
            elif ev[0] == "result":
                text, finish = ev[1], ev[2]
        last_text = text

        if finish == "cancelled":
            yield ("done", render_display(text, final=True) or "[Stopped]")
            return

        visible = strip_think(text).strip()
        calls = parse_tool_calls(visible)

        if finish == "context_full" and not calls:
            fit_history(history_messages, int(prompt_budget * 0.6))
            yield ("status", "Context limit reached - compacted history and retrying.")
            history_messages.append({"role": "user", "content": "Your previous reply was cut off because the context was full. Continue the task concisely."})
            continue

        if not calls:
            has_markup = "<tool_call>" in visible or "<function=" in visible
            if has_markup and malformed < 3:
                malformed += 1
                history_messages.append({"role": "assistant", "content": _strip_markup(visible) or "(tool call)"})
                history_messages.append({"role": "user", "content": (
                    "<tool_response name=\"system\">\nYour tool call could not be parsed. Re-send it using EXACTLY this format "
                    "(raw values, no JSON):\n<tool_call>\n<function=TOOL_NAME>\n<parameter=PARAM>\nVALUE\n</parameter>\n</function>\n</tool_call>\n"
                    "If the content is long, split it into smaller writes.\n</tool_response>")})
                yield ("status", "Malformed tool call - requesting valid XML format.")
                continue

            answer = _strip_markup(visible)
            if answer and len(answer) < 260 and intent_nudges < 1 and step < effective_steps - 1:
                low_a = answer.lower().strip()
                if (low_a.endswith(":") or low_a.endswith("...") or any(low_a.startswith(p) for p in ("let me ", "i will ", "i'll ", "now i will ", "next, i will ", "taking a screenshot", "opening "))):
                    intent_nudges += 1
                    history_messages.append({"role": "assistant", "content": answer})
                    history_messages.append({"role": "user", "content": "<tool_response name=\"system\">\nDo not narrate what you will do — emit the actual <tool_call> now to execute it.\n</tool_response>"})
                    yield ("status", "Prompting agent to execute tool call...")
                    continue
            if not answer:
                seed = text if text.strip() else ""
                if _in_think(seed):
                    seed = seed.rstrip() + "\n</think>\n\n"
                elif seed.strip():
                    seed = seed.rstrip() + "\n\n"
                yield ("status", "Generating final response...")
                retry_text = ""
                for ev in generate_assistant_message(history_messages, think_mode, temp, seed=seed, allow_budget=False):
                    if ev[0] == "stream":
                        yield ("assistant_stream", render_display(ev[1]))
                    elif ev[0] == "hb":
                        yield ("heartbeat", ev[1])
                    elif ev[0] == "result":
                        retry_text = ev[1]
                answer = _strip_markup(strip_think(retry_text).strip())
                text = retry_text or text
            if not answer:
                gist = re.sub(r"\s+", " ", text.replace("<think>", "").replace("</think>", ""))[-600:]
                answer = ("*The model completed reasoning without emitting a final response. Try again or set Reasoning to `off`.*"
                          + (f"\n\n> Last thoughts: {gist}" if gist.strip() else ""))
            if finish in ("length_cap", "context_full", "loop"):
                answer += "\n\n*(Output truncated at limit - send `continue` to resume.)*"
            history_messages.append({"role": "assistant", "content": answer})
            yield ("stats", stats_line())
            reasoning_view = render_display(text, final=True)
            yield ("done", reasoning_view if reasoning_view.strip() else answer)
            return

        history_messages.append({"role": "assistant", "content": visible})
        responses = []
        for cname, cargs in calls:
            if CANCEL.is_set():
                break
            key = (cname, json.dumps(cargs, sort_keys=True, default=str))
            seen[key] += 1
            yield ("tool_start", f"{cname}({json.dumps(cargs, default=str)[:200]})")
            n_before = len(TURN_DOWNLOADED_IMAGES)
            result = ""
            for ev in _run_with_heartbeat(dispatch_tool, cname, cname, cargs):
                if ev[0] == "hb":
                    yield ("heartbeat", ev[1])
                else:
                    result = ev[1]
            result = _truncate_output(str(result), 6000)

            if auto_vision and globals().get("VISION_ACTIVE", False):
                for img_path, q in _vision_targets(cname, cargs, n_before):
                    for ev in _run_with_heartbeat(vision_describe, "Inspecting image", img_path, q):
                        if ev[0] == "hb":
                            yield ("heartbeat", ev[1])
                        else:
                            result += f"\n\n[Vision notes for {img_path}]\n{ev[1]}"
            if seen[key] >= 3:
                result += "\n\n[system] You have already run this exact call 3 times. Change your approach or give your final answer."
            yield ("tool_end", f"{cname} -> {result[:460]}")
            responses.append(f"<tool_response name=\"{cname}\">\n{result}\n</tool_response>")
        if CANCEL.is_set():
            yield ("done", "[Stopped]")
            return
        history_messages.append({"role": "user", "content": "\n".join(responses)})

    yield ("stats", stats_line())
    yield ("done", (render_display(last_text, final=True) or "") + "\n\n*(Reached max autonomous steps for this turn - send `continue` to resume.)*")

print("[OK] Agent tools and execution loop ready.")

In [ ]:
#@title 5. Studio Web UI & Tunnel Host
#@markdown Launches the FastAPI web studio, exposes an optional Cloudflare quick tunnel, and keeps background services monitored.
ENABLE_CLOUDFLARE_TUNNEL = True #@param {type:"boolean"}
EMBED_COLAB_IFRAME = True #@param {type:"boolean"}
SERVER_PORT = 7860 #@param {type:"integer"}
KEEP_CELL_RUNNING_FOREVER = True #@param {type:"boolean"}

import os
import re
import sys
import time
import json
import queue
import shutil
import socket
import asyncio
import datetime
import threading
import traceback
import subprocess
from pathlib import Path
from typing import List, Optional

try:
    import multipart  # type: ignore
except ImportError:
    try:
        import python_multipart  # type: ignore
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-multipart"], check=False)

if ENABLE_CLOUDFLARE_TUNNEL and not os.path.exists("/usr/local/bin/cloudflared"):
    print("[Tunnel] Downloading cloudflared binary...")
    subprocess.run(
        ["curl", "-fsSL", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-o", "/usr/local/bin/cloudflared"],
        check=False
    )
    if os.path.exists("/usr/local/bin/cloudflared"):
        os.chmod("/usr/local/bin/cloudflared", 0o755)
        print("[OK] cloudflared installed.")

import uvicorn
from fastapi import FastAPI, Request, UploadFile, File, HTTPException, WebSocket
from fastapi.responses import HTMLResponse, JSONResponse, StreamingResponse, FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

WORKSPACE_DIR = "/content/workspace"
ARTIFACTS_DIR = f"{WORKSPACE_DIR}/artifacts"
IMAGES_DIR = f"{ARTIFACTS_DIR}/images"
UPLOADS_DIR = f"{WORKSPACE_DIR}/uploads"
LATEST_SCREEN_PATH = f"{WORKSPACE_DIR}/screenshots/latest_screen.png"
CLOUDFLARE_LOG = "/content/cloudflared.log"
SERVER_LOG = "/content/llama_server.log"

for _d in (ARTIFACTS_DIR, IMAGES_DIR, UPLOADS_DIR, f"{WORKSPACE_DIR}/screenshots"):
    os.makedirs(_d, exist_ok=True)

if "_UVICORN_SERVER" in globals() and globals()["_UVICORN_SERVER"] is not None:
    try:
        globals()["_UVICORN_SERVER"].should_exit = True
        time.sleep(1.0)
    except Exception:
        pass
subprocess.run(["pkill", "-f", f"cloudflared tunnel --url http://127.0.0.1:{SERVER_PORT}"], check=False)

if "STUDIO_STATE" not in globals():
    STUDIO_STATE = {
        "ui_messages": [],
        "llm_messages": [],
        "tunnel_url": "",
    }

AGENT_LOCK = threading.Lock()

def _fmt_size(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} GB"

def list_artifacts_meta() -> list:
    os.makedirs(ARTIFACTS_DIR, exist_ok=True)
    root = Path(ARTIFACTS_DIR)
    files = [p for p in root.rglob("*") if p.is_file() and not p.name.startswith(".")]
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    out = []
    for p in files[:200]:
        st = p.stat()
        rel = p.relative_to(root).as_posix()
        out.append({
            "name": p.name,
            "rel_path": rel,
            "path": str(p),
            "size": st.st_size,
            "size_str": _fmt_size(st.st_size),
            "mtime": int(st.st_mtime),
            "ext": p.suffix.lower(),
            "url": f"/api/files/artifact/{rel}?t={int(st.st_mtime)}",
        })
    return out

def list_images_meta() -> list:
    os.makedirs(IMAGES_DIR, exist_ok=True)
    imgs = [
        p for p in Path(IMAGES_DIR).glob("*")
        if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp", ".gif")
    ]
    imgs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    out = []
    for p in imgs[:80]:
        st = p.stat()
        out.append({
            "name": p.name,
            "path": str(p),
            "url": f"/api/files/artifact/images/{p.name}?t={int(st.st_mtime)}",
            "size_str": _fmt_size(st.st_size),
            "mtime": int(st.st_mtime),
        })
    return out

def get_latest_video_meta():
    for item in list_artifacts_meta():
        if item["ext"] in (".mp4", ".webm", ".mov"):
            return item
    return None

def get_latest_screen_meta():
    if os.path.exists(LATEST_SCREEN_PATH):
        st = os.stat(LATEST_SCREEN_PATH)
        return {"url": f"/api/files/screen?t={int(st.st_mtime)}", "mtime": int(st.st_mtime), "path": LATEST_SCREEN_PATH}
    return None

def _tcp_open(host: str, port: int, timeout: float = 0.6) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except Exception:
        return False

def _ensure_vnc_running():
    if _tcp_open("127.0.0.1", 5900, timeout=0.4):
        return True
    if not shutil.which("x11vnc"):
        return False
    try:
        if not subprocess.run(["pgrep", "-f", "Xvfb :99"], capture_output=True).returncode == 0:
            subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1920x1080x24", "-ac"],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
            time.sleep(1.0)
            subprocess.Popen(["fluxbox"], env={**os.environ, "DISPLAY": ":99"},
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
        subprocess.run(["pkill", "-x", "x11vnc"], check=False)
        subprocess.Popen(
            ["x11vnc", "-display", ":99", "-forever", "-shared", "-nopw",
             "-rfbport", "5900", "-listen", "127.0.0.1", "-noxdamage", "-xkb"],
            env={**os.environ, "DISPLAY": ":99"},
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True
        )
        time.sleep(0.8)
        return _tcp_open("127.0.0.1", 5900, timeout=0.8)
    except Exception:
        return False

_ensure_vnc_running()

def get_skills_meta() -> list:
    if "skill_manager" not in globals():
        return []
    skill_manager.refresh()
    return [
        {
            "name": info["name"],
            "description": info["description"],
            "path": info["path"],
            "folder": info["folder"],
            "resources": info["resources"],
        }
        for info in skill_manager.registry.values()
    ]

def get_system_meta() -> dict:
    up = server_is_up() if "server_is_up" in globals() else False
    gpu_name, vram_used, vram_total, gpu_util = "Unavailable", 0, 0, 0
    try:
        g = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=6
        ).stdout.strip().split(",")
        if len(g) >= 4:
            gpu_name = g[0].strip()
            vram_used = int(float(g[1].strip()))
            vram_total = int(float(g[2].strip()))
            gpu_util = int(float(g[3].strip()))
    except Exception:
        pass

    s_log = "(server log unavailable - run Cell 2 first)"
    if "server_log_tail" in globals():
        s_log = server_log_tail(2800)

    t_log = ""
    try:
        if os.path.exists(CLOUDFLARE_LOG):
            with open(CLOUDFLARE_LOG, errors="replace") as f:
                t_log = f.read()[-2000:]
    except Exception:
        pass

    return {
        "server_up": up,
        "vision_active": bool(globals().get("VISION_ACTIVE", False)),
        "context_size": int(globals().get("CONTEXT_SIZE", 32768)),
        "temperature": float(globals().get("TEMPERATURE", 0.6)),
        "crash_count": int(globals().get("SERVER_CRASH_COUNT", 0)),
        "gpu_name": gpu_name,
        "vram_used": vram_used,
        "vram_total": vram_total,
        "gpu_util": gpu_util,
        "server_log": s_log,
        "tunnel_log": t_log,
        "tunnel_url": STUDIO_STATE.get("tunnel_url", ""),
        "provider": globals().get("LLM_CONFIG", {}).get("provider", "local"),
        "gemini_model": globals().get("LLM_CONFIG", {}).get("gemini_model", "gemini-2.5-flash"),
        "has_gemini_key": bool((globals().get("LLM_CONFIG", {}).get("gemini_api_key") or "").strip()),
    }

def _context_used_tokens() -> int:
    try:
        if "est_tokens" in globals() and "render_chatml" in globals():
            return est_tokens(render_chatml(STUDIO_STATE["llm_messages"]))
    except Exception:
        pass
    return 0

app = FastAPI(title="DeepSeek-V4-Pro · Qwen3.5 Studio")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatReq(BaseModel):
    message: str
    reasoning_mode: str = "brief"
    max_steps: int = 24
    auto_vision: bool = True

class GithubSkillReq(BaseModel):
    repo_url: str
    subpath: str = ""

class ProviderReq(BaseModel):
    provider: str = "local"
    gemini_api_key: Optional[str] = None
    gemini_model: Optional[str] = None

@app.get("/api/health")
def api_health():
    return {"ok": True, "ts": time.time()}

@app.get("/api/state")
def api_state():
    ctx_max = int(globals().get("CONTEXT_SIZE", 32768))
    return {
        "messages": STUDIO_STATE["ui_messages"],
        "artifacts": list_artifacts_meta(),
        "latest_video": get_latest_video_meta(),
        "latest_screen": get_latest_screen_meta(),
        "images": list_images_meta(),
        "skills": get_skills_meta(),
        "system": get_system_meta(),
        "context_used": _context_used_tokens(),
        "context_max": ctx_max,
    }

@app.post("/api/stop")
def api_stop():
    if "CANCEL" in globals():
        CANCEL.set()
    return {"ok": True}

@app.post("/api/clear")
def api_clear():
    if "CANCEL" in globals():
        CANCEL.set()
    STUDIO_STATE["ui_messages"].clear()
    STUDIO_STATE["llm_messages"].clear()
    return api_state()

@app.post("/api/upload")
async def api_upload(files: List[UploadFile] = File(...)):
    saved = []
    for f in files:
        safe_name = Path(f.filename or f"upload_{int(time.time())}").name
        dest = Path(UPLOADS_DIR) / safe_name
        data = await f.read()
        dest.write_bytes(data)
        saved.append(str(dest))
    note = "Uploaded files to workspace: " + ", ".join(saved) + ". " if saved else ""
    return {"saved": saved, "note": note}

@app.post("/api/export")
def api_export():
    lines = [f"# Agent Chat Export - {datetime.datetime.now():%Y-%m-%d %H:%M}\n"]
    for m in STUDIO_STATE["llm_messages"]:
        if m.get("role") == "system":
            continue
        who = "User" if m["role"] == "user" else "Assistant"
        lines.append(f"## {who}\n\n{str(m.get('content', ''))[:8000]}\n")
    out = Path(ARTIFACTS_DIR) / f"chat_export_{datetime.datetime.now():%Y%m%d_%H%M%S}.md"
    out.write_text("\n".join(lines), encoding="utf-8")
    return {"exported": out.name, "artifacts": list_artifacts_meta()}

@app.post("/api/skills/github")
async def api_skill_github(req: GithubSkillReq):
    if not req.repo_url.strip():
        return {"message": "Enter a GitHub URL first.", "skills": get_skills_meta()}
    loop = asyncio.get_running_loop()
    msg = await loop.run_in_executor(
        None, lambda: skill_manager.install_from_github(req.repo_url.strip(), req.subpath.strip())
    )
    return {"message": msg, "skills": get_skills_meta()}

@app.post("/api/skills/upload")
async def api_skill_upload(file: UploadFile = File(...)):
    safe_name = Path(file.filename or "SKILL.md").name
    tmp_path = Path("/tmp") / f"skill_up_{int(time.time())}_{safe_name}"
    tmp_path.write_bytes(await file.read())
    try:
        msg = skill_manager.install_from_uploaded_file(str(tmp_path))
    finally:
        tmp_path.unlink(missing_ok=True)
    return {"message": msg, "skills": get_skills_meta()}

@app.post("/api/screen/capture")
async def api_screen_capture():
    loop = asyncio.get_running_loop()
    out = await loop.run_in_executor(None, lambda: tool_desktop_action("screenshot"))
    return {"message": out, "screen": get_latest_screen_meta()}

@app.get("/api/vnc/status")
async def api_vnc_status():
    loop = asyncio.get_running_loop()
    up = await loop.run_in_executor(None, _ensure_vnc_running)
    return {"vnc_up": up, "novnc_installed": os.path.isdir("/usr/share/novnc")}

@app.get("/novnc/{file_path:path}")
def serve_novnc(file_path: str):
    novnc_root = os.path.realpath("/usr/share/novnc")
    if not os.path.isdir(novnc_root):
        raise HTTPException(status_code=404, detail="noVNC not installed. Re-run Cell 1 with ENABLE_VIRTUAL_DESKTOP=True.")
    rel = (file_path or "vnc_lite.html").lstrip("/")
    target = os.path.realpath(os.path.join(novnc_root, rel))
    if not (target == novnc_root or target.startswith(novnc_root + os.sep)) or not os.path.isfile(target):
        raise HTTPException(status_code=404, detail="noVNC asset not found.")
    return FileResponse(target)

@app.websocket("/novnc/websockify")
@app.websocket("/websockify")
async def vnc_websockify_proxy(websocket: WebSocket):
    requested = websocket.scope.get("subprotocols") or []
    subproto = "binary" if "binary" in requested else None
    await websocket.accept(subprotocol=subproto)
    loop = asyncio.get_running_loop()
    if not _tcp_open("127.0.0.1", 5900, timeout=0.4):
        await loop.run_in_executor(None, _ensure_vnc_running)
    try:
        reader, writer = await asyncio.open_connection("127.0.0.1", 5900)
    except Exception as e:
        await websocket.close(code=1011, reason=f"VNC offline: {e}")
        return

    async def ws_to_tcp():
        try:
            while True:
                msg = await websocket.receive()
                if msg["type"] == "websocket.disconnect":
                    break
                data = msg.get("bytes")
                if data is None and msg.get("text") is not None:
                    data = msg["text"].encode("utf-8")
                if data:
                    writer.write(data)
                    await writer.drain()
        except Exception:
            pass
        finally:
            try:
                writer.close()
            except Exception:
                pass

    async def tcp_to_ws():
        try:
            while True:
                data = await reader.read(65536)
                if not data:
                    break
                await websocket.send_bytes(data)
        except Exception:
            pass
        finally:
            try:
                await websocket.close()
            except Exception:
                pass

    t1 = asyncio.create_task(ws_to_tcp())
    t2 = asyncio.create_task(tcp_to_ws())
    _, pending = await asyncio.wait({t1, t2}, return_when=asyncio.FIRST_COMPLETED)
    for p in pending:
        p.cancel()

@app.post("/api/provider")
def api_set_provider(req: ProviderReq):
    cfg = globals().get("LLM_CONFIG")
    if cfg is None:
        cfg = {"provider": "local", "gemini_api_key": "", "gemini_model": "gemini-2.5-flash"}
        globals()["LLM_CONFIG"] = cfg
    if req.gemini_api_key is not None and req.gemini_api_key.strip():
        cfg["gemini_api_key"] = req.gemini_api_key.strip()
    if req.gemini_model is not None and req.gemini_model.strip():
        cfg["gemini_model"] = req.gemini_model.strip()
    prov = (req.provider or "local").strip().lower()
    if prov == "gemini" and not (cfg.get("gemini_api_key") or "").strip():
        return {"ok": False, "message": "Enter a Gemini API key first.", "system": get_system_meta()}
    cfg["provider"] = "gemini" if prov == "gemini" else "local"
    return {"ok": True, "message": f"Switched AI provider to {cfg['provider'].upper()}.", "system": get_system_meta()}

@app.post("/api/system/restart")
async def api_system_restart():
    loop = asyncio.get_running_loop()
    try:
        await loop.run_in_executor(None, lambda: start_or_restart_llama_server(force_restart=True))
        return {"ok": True, "message": "LLM server restarted.", "system": get_system_meta()}
    except Exception as e:
        return {"ok": False, "message": f"Restart failed: {e}", "system": get_system_meta()}

@app.get("/api/files/screen")
def api_file_screen():
    if not os.path.exists(LATEST_SCREEN_PATH):
        raise HTTPException(status_code=404, detail="No screenshot captured yet.")
    return FileResponse(LATEST_SCREEN_PATH, media_type="image/png", headers={"Cache-Control": "no-store"})

@app.get("/api/files/artifact/{rel_path:path}")
def api_file_artifact(rel_path: str, download: int = 0):
    root = os.path.realpath(ARTIFACTS_DIR)
    target = os.path.realpath(os.path.join(ARTIFACTS_DIR, rel_path))
    if not (target == root or target.startswith(root + os.sep)) or not os.path.isfile(target):
        raise HTTPException(status_code=404, detail="Artifact not found.")
    headers = {"Cache-Control": "no-cache"}
    if download:
        return FileResponse(target, filename=os.path.basename(target), headers=headers)
    return FileResponse(target, headers=headers)

@app.post("/api/chat/stream")
async def api_chat_stream(req: ChatReq, request: Request):
    user_msg = (req.message or "").strip()
    if not user_msg:
        raise HTTPException(status_code=400, detail="Empty message.")

    if not AGENT_LOCK.acquire(blocking=False):
        async def busy_stream():
            yield f"data: {json.dumps({'type': 'error', 'text': 'Agent is currently executing a turn. Click Stop first or wait for completion.'})}\n\n"
        return StreamingResponse(busy_stream(), media_type="text/event-stream")

    STUDIO_STATE["ui_messages"].append({
        "role": "user",
        "content": user_msg,
        "ts": int(time.time()),
    })

    ev_q = queue.Queue()
    ctx_max = int(globals().get("CONTEXT_SIZE", 32768))

    def worker():
        tool_log = []
        reply = ""
        stats = ""
        images = []
        try:
            for ev in run_agent_turn(
                user_msg,
                STUDIO_STATE["llm_messages"],
                max_steps=0 if int(req.max_steps) <= 0 else max(1, min(int(req.max_steps), 5000)),
                think_mode=req.reasoning_mode if req.reasoning_mode in ("off", "brief", "deep") else "brief",
                auto_vision=bool(req.auto_vision),
            ):
                kind, payload = ev[0], ev[1]
                if kind == "tool_start":
                    entry = {"kind": "start", "text": str(payload), "ts": time.time()}
                    tool_log.append(entry)
                    ev_q.put({"type": "tool_start", "entry": entry})
                elif kind == "tool_end":
                    entry = {"kind": "end", "text": str(payload), "ts": time.time()}
                    tool_log.append(entry)
                    ev_q.put({"type": "tool_end", "entry": entry})
                elif kind == "status":
                    entry = {"kind": "status", "text": str(payload), "ts": time.time()}
                    tool_log.append(entry)
                    ev_q.put({"type": "status", "entry": entry, "text": str(payload)})
                elif kind == "heartbeat":
                    ev_q.put({"type": "heartbeat", "text": str(payload)})
                elif kind == "assistant_stream":
                    reply = str(payload)
                    ev_q.put({"type": "stream", "text": reply})
                elif kind == "stats":
                    stats = str(payload)
                    ev_q.put({
                        "type": "stats",
                        "text": stats,
                        "context_used": _context_used_tokens(),
                        "context_max": ctx_max,
                    })
                elif kind == "done":
                    reply = str(payload)
                    if "TURN_DOWNLOADED_IMAGES" in globals() and TURN_DOWNLOADED_IMAGES:
                        for im in TURN_DOWNLOADED_IMAGES:
                            p = im.get("path", "")
                            if os.path.exists(p):
                                images.append({
                                    "name": os.path.basename(p),
                                    "path": p,
                                    "url": f"/api/files/artifact/images/{os.path.basename(p)}?t={int(time.time())}",
                                    "title": im.get("title", ""),
                                    "source_url": im.get("source_url", ""),
                                    "size": im.get("size", ""),
                                })
                    STUDIO_STATE["ui_messages"].append({
                        "role": "assistant",
                        "content": reply,
                        "tool_log": tool_log,
                        "stats": stats,
                        "images": images,
                        "ts": int(time.time()),
                    })
                    ev_q.put({
                        "type": "done",
                        "text": reply,
                        "tool_log": tool_log,
                        "stats": stats,
                        "images": images,
                        "context_used": _context_used_tokens(),
                        "context_max": ctx_max,
                    })
        except Exception as e:
            tb = traceback.format_exc()[-1200:]
            err_msg = f"{type(e).__name__}: {str(e)[:400]}"
            print(f"\n[Agent Exception] {err_msg}\n{tb}", flush=True)
            STUDIO_STATE["ui_messages"].append({
                "role": "assistant",
                "content": (reply + "\n\n" if reply else "") + f"**Runtime Error:** `{err_msg}`",
                "tool_log": tool_log,
                "stats": stats,
                "images": images,
                "error": tb,
                "ts": int(time.time()),
            })
            ev_q.put({"type": "error", "text": err_msg, "traceback": tb})
        finally:
            if "CANCEL" in globals():
                CANCEL.set()
            AGENT_LOCK.release()
            ev_q.put(None)

    threading.Thread(target=worker, daemon=True).start()

    async def sse_gen():
        last_stream_push = 0.0
        try:
            while True:
                if await request.is_disconnected():
                    if "CANCEL" in globals():
                        CANCEL.set()
                    break
                try:
                    item = ev_q.get_nowait()
                except queue.Empty:
                    await asyncio.sleep(0.025)
                    continue
                if item is None:
                    break
                if item.get("type") == "stream":
                    now = time.time()
                    if now - last_stream_push < 0.055:
                        continue
                    last_stream_push = now
                yield f"data: {json.dumps(item, ensure_ascii=False)}\n\n"
        finally:
            if "CANCEL" in globals():
                CANCEL.set()

    return StreamingResponse(
        sse_gen(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache, no-transform",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no",
        },
    )

WORKSPACE_HTML = r"""<!DOCTYPE html>
<html lang="en" class="dark">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>DeepSeek-V4-Pro · Qwen3.5 Studio</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/highlight.js@11.9.0/styles/github-dark.min.css">
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.css">
<script src="https://cdn.jsdelivr.net/npm/marked@12.0.2/marked.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/highlight.js@11.9.0/lib/common.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/contrib/auto-render.min.js"></script>
<style>
  :root {
    --bg-base: #050507;
    --bg-surface: #0a0b0e;
    --bg-elevated: #0f1116;
    --bg-card: rgba(16, 18, 24, 0.78);
    --bg-hover: rgba(255, 255, 255, 0.045);
    --border-subtle: rgba(255, 255, 255, 0.065);
    --border-default: rgba(255, 255, 255, 0.11);
    --border-focus: rgba(255, 255, 255, 0.26);
    --text-primary: #f4f4f6;
    --text-secondary: #9ca3af;
    --text-muted: #52525b;
    --accent: #e4e4e7;
    --accent-cyan: #22d3ee;
    --accent-indigo: #818cf8;
    --status-ok: #10b981;
    --status-busy: #f59e0b;
    --status-err: #ef4444;
    --radius-lg: 12px;
    --radius-md: 8px;
    --radius-sm: 6px;
    --font-sans: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
    --font-mono: 'JetBrains Mono', ui-monospace, SFMono-Regular, monospace;
  }

  * { box-sizing: border-box; }
  html, body {
    margin: 0; padding: 0; height: 100%; overflow: hidden;
    background: var(--bg-base); color: var(--text-primary);
    font-family: var(--font-sans); -webkit-font-smoothing: antialiased;
    letter-spacing: -0.011em;
  }
  .ambient-canvas {
    position: fixed; inset: 0; pointer-events: none; z-index: 0;
    background-image:
      radial-gradient(ellipse 80% 45% at 50% -15%, rgba(129, 140, 248, 0.11), transparent 70%),
      radial-gradient(ellipse 50% 35% at 85% 105%, rgba(34, 211, 238, 0.06), transparent 70%),
      linear-gradient(to right, rgba(255,255,255,0.018) 1px, transparent 1px),
      linear-gradient(to bottom, rgba(255,255,255,0.018) 1px, transparent 1px);
    background-size: 100% 100%, 100% 100%, 36px 36px, 36px 36px;
  }
  body.is-running .ambient-canvas {
    background-image:
      radial-gradient(ellipse 80% 50% at 50% -10%, rgba(34, 211, 238, 0.15), transparent 70%),
      radial-gradient(ellipse 50% 40% at 85% 105%, rgba(129, 140, 248, 0.10), transparent 70%),
      linear-gradient(to right, rgba(255,255,255,0.022) 1px, transparent 1px),
      linear-gradient(to bottom, rgba(255,255,255,0.022) 1px, transparent 1px);
  }

  .app-shell {
    position: relative; z-index: 1; display: flex; flex-direction: column; height: 100vh;
  }
  .topbar {
    height: 48px; flex-shrink: 0; display: flex; align-items: center; justify-content: space-between;
    padding: 0 16px; border-bottom: 1px solid var(--border-subtle);
    background: rgba(5, 5, 7, 0.84); backdrop-filter: blur(16px);
    gap: 12px;
  }
  .brand {
    display: flex; align-items: center; gap: 10px; min-width: 0;
  }
  .brand-mark {
    width: 24px; height: 24px; border-radius: 6px;
    background: linear-gradient(180deg, #27272a 0%, #18181b 100%);
    border: 1px solid rgba(255,255,255,0.14);
    box-shadow: inset 0 1px 0 rgba(255,255,255,0.15);
    display: grid; place-items: center; flex-shrink: 0;
  }
  .brand-mark svg { width: 13px; height: 13px; color: #f4f4f6; }
  .brand-title {
    font-weight: 600; font-size: 13px; color: var(--text-primary); letter-spacing: -0.02em;
    white-space: nowrap; overflow: hidden; text-overflow: ellipsis;
  }
  .brand-divider {
    width: 1px; height: 14px; background: var(--border-default);
  }
  .meta-tag {
    display: inline-flex; align-items: center; gap: 6px;
    padding: 2px 8px; border-radius: 4px; font-size: 11px; font-family: var(--font-mono);
    background: rgba(255,255,255,0.03); border: 1px solid var(--border-subtle);
    color: var(--text-secondary); white-space: nowrap;
  }
  .meta-tag.active-vision {
    color: var(--accent-cyan); border-color: rgba(34, 211, 238, 0.22); background: rgba(34, 211, 238, 0.05);
  }
  .meta-tag.clickable { cursor: pointer; transition: border-color 0.15s, color 0.15s; }
  .meta-tag.clickable:hover { border-color: var(--border-focus); color: var(--text-primary); }

  .topbar-right {
    display: flex; align-items: center; gap: 10px; flex-shrink: 0;
  }
  .ctx-meter {
    display: flex; align-items: center; gap: 8px;
    padding: 4px 10px; border-radius: 6px; background: rgba(255,255,255,0.025);
    border: 1px solid var(--border-subtle); font-family: var(--font-mono); font-size: 11px;
    color: var(--text-secondary); font-variant-numeric: tabular-nums;
  }
  .ctx-track {
    width: 56px; height: 4px; border-radius: 999px; background: rgba(255,255,255,0.08); overflow: hidden;
  }
  .ctx-fill {
    height: 100%; width: 3%; border-radius: 999px; background: #e4e4e7;
    transition: width 0.35s ease, background 0.25s ease;
  }
  .status-badge {
    display: inline-flex; align-items: center; gap: 7px;
    padding: 4px 10px; border-radius: 6px; font-size: 11px; font-family: var(--font-mono);
    font-weight: 500; text-transform: uppercase; letter-spacing: 0.05em;
    background: rgba(255, 255, 255, 0.03); border: 1px solid var(--border-subtle);
    color: var(--text-secondary);
  }
  .status-dot {
    width: 6px; height: 6px; border-radius: 50%;
    background: var(--status-ok); box-shadow: 0 0 8px var(--status-ok);
  }
  .status-badge.working {
    color: #fef08a; border-color: rgba(245, 158, 11, 0.25); background: rgba(245, 158, 11, 0.06);
  }
  .status-badge.working .status-dot {
    background: var(--status-busy); box-shadow: 0 0 10px var(--status-busy);
    animation: pulseLed 1.1s ease-in-out infinite alternate;
  }
  .status-badge.error {
    color: #fca5a5; border-color: rgba(239, 68, 68, 0.28); background: rgba(239, 68, 68, 0.06);
  }
  .status-badge.error .status-dot {
    background: var(--status-err); box-shadow: 0 0 10px var(--status-err);
  }
  @keyframes pulseLed {
    0% { opacity: 0.45; transform: scale(0.88); }
    100% { opacity: 1; transform: scale(1.18); }
  }
  .workspace-grid {
    flex: 1; display: flex; min-height: 0; overflow: hidden; position: relative;
  }
  .chat-pane {
    flex: 1 1 60%; min-width: 360px; min-height: 0; height: 100%; display: flex; flex-direction: column;
    background: transparent; position: relative; overflow: hidden;
  }
  .resizer {
    width: 4px; cursor: col-resize; background: transparent;
    border-left: 1px solid var(--border-subtle); transition: background 0.15s;
    flex-shrink: 0; z-index: 5;
  }
  .resizer:hover, .resizer.dragging { background: rgba(255, 255, 255, 0.22); }

  .inspector-pane {
    width: 40%; min-width: 320px; max-width: 64%; min-height: 0; height: 100%; display: flex; flex-direction: column;
    background: var(--bg-surface); border-left: 1px solid var(--border-subtle); overflow: hidden;
    transition: width 0.2s ease, opacity 0.18s ease;
  }
  .inspector-pane.collapsed {
    width: 0 !important; min-width: 0 !important; overflow: hidden; opacity: 0; border: none;
  }
  .chat-scroll {
    flex: 1; overflow-y: auto; padding: 28px 24px 16px; scroll-behavior: smooth;
    display: flex; flex-direction: column; gap: 16px;
  }
  .chat-scroll::-webkit-scrollbar, .tab-body::-webkit-scrollbar { width: 5px; height: 5px; }
  .chat-scroll::-webkit-scrollbar-thumb, .tab-body::-webkit-scrollbar-thumb {
    background: rgba(255,255,255,0.12); border-radius: 999px;
  }
  .welcome-hero {
    margin: auto; max-width: 620px; width: 100%; padding: 24px 16px;
    animation: fadeUp 0.35s cubic-bezier(0.16, 1, 0.3, 1);
  }
  .hero-kicker {
    display: inline-flex; align-items: center; gap: 8px;
    font-family: var(--font-mono); font-size: 11px; text-transform: uppercase;
    letter-spacing: 0.08em; color: var(--text-secondary);
    padding: 4px 10px; border-radius: 999px;
    border: 1px solid var(--border-default); background: rgba(255,255,255,0.02);
    margin-bottom: 14px;
  }
  .hero-kicker-dot {
    width: 5px; height: 5px; border-radius: 50%; background: var(--accent-cyan);
    box-shadow: 0 0 8px var(--accent-cyan);
  }
  .welcome-hero h1 {
    font-size: 22px; font-weight: 600; margin: 0 0 8px; letter-spacing: -0.025em;
    color: var(--text-primary);
  }
  .welcome-hero p {
    font-size: 13px; color: var(--text-secondary); margin: 0 0 22px; line-height: 1.6;
  }
  .preset-grid {
    display: grid; grid-template-columns: repeat(2, 1fr); gap: 10px;
  }
  .preset-card, .card, .artifact-row, .skill-card, .gal-card, .assistant-card {
    position: relative; overflow: hidden;
    --mx: 50%; --my: 50%;
  }
  .preset-card::before, .card::before, .artifact-row::before, .skill-card::before, .gal-card::before {
    content: ""; position: absolute; inset: 0; pointer-events: none; opacity: 0;
    background: radial-gradient(320px circle at var(--mx) var(--my), rgba(255, 255, 255, 0.075), transparent 60%);
    transition: opacity 0.28s cubic-bezier(0.22, 1, 0.36, 1);
    z-index: 0;
  }
  .preset-card:hover::before, .card:hover::before, .artifact-row:hover::before, .skill-card:hover::before, .gal-card:hover::before {
    opacity: 1;
  }
  .preset-card > *, .card > *, .artifact-row > *, .skill-card > *, .gal-card > * {
    position: relative; z-index: 1;
  }

  .preset-card {
    padding: 14px 15px; border-radius: var(--radius-md);
    background: var(--bg-elevated); border: 1px solid var(--border-subtle);
    box-shadow: inset 0 1px 0 rgba(255,255,255,0.04);
    cursor: pointer;
    transition: transform 0.26s cubic-bezier(0.22, 1, 0.36, 1), border-color 0.22s ease, box-shadow 0.26s ease, background 0.22s ease;
  }
  .preset-card::after {
    content: ""; position: absolute; top: 0; left: -100%; width: 60%; height: 1px;
    background: linear-gradient(90deg, transparent, rgba(34, 211, 238, 0.75), transparent);
    transition: left 0.55s cubic-bezier(0.22, 1, 0.36, 1);
  }
  .preset-card:hover {
    transform: translateY(-3px) scale(1.01);
    border-color: rgba(255, 255, 255, 0.24);
    background: #13161f;
    box-shadow: inset 0 1px 0 rgba(255,255,255,0.09), 0 14px 32px rgba(0, 0, 0, 0.55), 0 0 24px rgba(34, 211, 238, 0.06);
  }
  .preset-card:hover::after {
    left: 140%;
  }
  .preset-card:active {
    transform: translateY(-1px) scale(0.995);
  }
  .preset-card .p-tag {
    font-family: var(--font-mono); font-size: 10px; text-transform: uppercase;
    letter-spacing: 0.07em; color: var(--text-muted); margin-bottom: 6px;
  }
  .preset-card .p-title {
    font-size: 13px; font-weight: 600; color: var(--text-primary); margin-bottom: 4px;
  }
  .preset-card .p-desc {
    font-size: 12px; color: var(--text-secondary); line-height: 1.45;
  }
  .msg-row {
    display: flex; flex-direction: column; max-width: 840px; width: 100%; margin: 0 auto;
    animation: fadeUp 0.22s ease-out;
  }
  @keyframes fadeUp {
    from { opacity: 0; transform: translateY(6px); }
    to { opacity: 1; transform: translateY(0); }
  }
  .msg-row.user { align-items: flex-end; }
  .user-bubble {
    max-width: 78%; padding: 10px 14px; border-radius: 10px;
    background: #181a22; border: 1px solid var(--border-default);
    box-shadow: inset 0 1px 0 rgba(255,255,255,0.06);
    color: var(--text-primary); font-size: 13.5px; line-height: 1.55;
    white-space: pre-wrap; word-break: break-word;
  }

  .assistant-card {
    width: 100%; padding: 16px 18px; border-radius: var(--radius-lg);
    background: var(--bg-card); border: 1px solid var(--border-subtle);
    box-shadow: inset 0 1px 0 rgba(255,255,255,0.04), 0 10px 30px rgba(0,0,0,0.35);
    backdrop-filter: blur(12px);
  }
  .assistant-header {
    display: flex; align-items: center; justify-content: space-between;
    margin-bottom: 10px; padding-bottom: 8px; border-bottom: 1px solid var(--border-subtle);
    font-family: var(--font-mono); font-size: 11px; color: var(--text-secondary);
  }
  .assistant-id {
    display: flex; align-items: center; gap: 8px; font-weight: 600; color: var(--text-primary);
    letter-spacing: 0.02em;
  }
  .assistant-id-dot {
    width: 6px; height: 6px; border-radius: 2px; background: var(--accent-cyan);
  }
  .think-box {
    margin-bottom: 10px; border-radius: var(--radius-sm);
    background: rgba(255, 255, 255, 0.02); border: 1px solid var(--border-subtle);
    overflow: hidden;
  }
  .think-toggle {
    width: 100%; display: flex; align-items: center; justify-content: space-between;
    padding: 7px 11px; background: transparent; border: none; color: var(--text-secondary);
    font-size: 11.5px; font-family: var(--font-mono); font-weight: 500; cursor: pointer; text-align: left;
  }
  .think-toggle:hover { background: var(--bg-hover); color: var(--text-primary); }
  .think-shimmer {
    color: var(--accent-cyan);
  }
  .think-content-wrap {
    display: grid; grid-template-rows: 0fr; transition: grid-template-rows 0.22s ease;
  }
  .think-box.open .think-content-wrap { grid-template-rows: 1fr; }
  .think-content {
    overflow: hidden; padding: 0 11px; font-size: 12px; line-height: 1.55;
    color: var(--text-secondary); white-space: pre-wrap; font-family: var(--font-mono);
  }
  .think-box.open .think-content {
    padding: 8px 11px 10px; border-top: 1px solid var(--border-subtle);
  }
  .tools-drawer {
    margin-bottom: 10px; border-radius: var(--radius-sm);
    background: rgba(255, 255, 255, 0.018); border: 1px solid var(--border-subtle);
    overflow: hidden;
  }
  .tools-header {
    width: 100%; display: flex; align-items: center; justify-content: space-between;
    padding: 7px 11px; background: transparent; border: none; color: var(--text-secondary);
    font-size: 11.5px; font-family: var(--font-mono); font-weight: 500; cursor: pointer;
  }
  .tools-header:hover { background: var(--bg-hover); color: var(--text-primary); }
  .tools-list {
    display: flex; flex-direction: column; gap: 5px; padding: 6px 10px 10px;
    border-top: 1px solid var(--border-subtle);
  }
  .tools-drawer.collapsed .tools-list { display: none; }
  .tool-item {
    display: flex; align-items: flex-start; gap: 8px; padding: 6px 9px;
    border-radius: 4px; background: #07080b;
    border: 1px solid var(--border-subtle); font-size: 11.5px; font-family: var(--font-mono);
    line-height: 1.45; word-break: break-word; color: var(--text-secondary);
  }
  .tool-tag {
    font-size: 10px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.05em;
    padding: 1px 5px; border-radius: 3px; flex-shrink: 0;
  }
  .tool-item.start .tool-tag { background: rgba(245, 158, 11, 0.14); color: #fbbf24; border: 1px solid rgba(245, 158, 11, 0.3); }
  .tool-item.end .tool-tag { background: rgba(16, 185, 129, 0.14); color: #34d399; border: 1px solid rgba(16, 185, 129, 0.3); }
  .tool-item.status .tool-tag { background: rgba(129, 140, 248, 0.14); color: #a5b4fc; border: 1px solid rgba(129, 140, 248, 0.3); }
  .live-hb {
    display: inline-flex; align-items: center; gap: 8px; margin-top: 8px;
    padding: 4px 10px; border-radius: 4px; font-size: 11px; font-family: var(--font-mono);
    background: rgba(255, 255, 255, 0.03); border: 1px solid var(--border-default);
    color: var(--text-secondary);
  }
  .spinner {
    width: 10px; height: 10px; border: 1.5px solid rgba(255,255,255,0.2);
    border-top-color: var(--accent-cyan); border-radius: 50%; animation: spin 0.7s linear infinite;
  }
  @keyframes spin { to { transform: rotate(360deg); } }
  .md-body { font-size: 13.75px; line-height: 1.65; color: var(--text-primary); overflow-wrap: break-word; }
  .md-body p { margin: 0 0 10px; }
  .md-body p:last-child { margin-bottom: 0; }
  .md-body h1, .md-body h2, .md-body h3 { margin: 14px 0 8px; font-weight: 600; letter-spacing: -0.02em; }
  .md-body ul, .md-body ol { margin: 6px 0 10px; padding-left: 20px; }
  .md-body li { margin: 4px 0; }
  .md-body a { color: var(--accent-cyan); text-decoration: none; border-bottom: 1px solid rgba(34,211,238,0.3); }
  .md-body a:hover { border-bottom-color: var(--accent-cyan); }
  .md-body code:not(pre code) {
    font-family: var(--font-mono); font-size: 12px; padding: 2px 5px; border-radius: 4px;
    background: rgba(255,255,255,0.07); color: #e4e4e7;
  }
  .md-body pre {
    position: relative; margin: 10px 0; padding: 13px; border-radius: var(--radius-md);
    background: #060709 !important; border: 1px solid var(--border-default);
    overflow-x: auto; font-family: var(--font-mono); font-size: 12px; line-height: 1.5;
  }
  .copy-code-btn {
    position: absolute; top: 7px; right: 7px; padding: 3px 8px; border-radius: 4px;
    background: rgba(255,255,255,0.06); border: 1px solid var(--border-subtle);
    color: var(--text-secondary); font-size: 10.5px; font-family: var(--font-mono); cursor: pointer;
  }
  .copy-code-btn:hover { background: rgba(255,255,255,0.14); color: white; }
  .md-body table {
    width: 100%; border-collapse: collapse; margin: 10px 0; font-size: 12.5px;
  }
  .md-body th, .md-body td {
    border: 1px solid var(--border-default); padding: 6px 10px; text-align: left;
  }
  .md-body th { background: rgba(255,255,255,0.04); font-weight: 600; }
  .md-body blockquote {
    margin: 10px 0; padding: 8px 12px; border-left: 2px solid var(--border-focus);
    background: rgba(255,255,255,0.02); color: var(--text-secondary);
  }
  .inline-imgs {
    display: grid; grid-template-columns: repeat(auto-fill, minmax(170px, 1fr));
    gap: 10px; margin-top: 12px; padding-top: 12px; border-top: 1px solid var(--border-subtle);
  }
  .inline-img-card {
    border-radius: var(--radius-sm); overflow: hidden; background: #07080b;
    border: 1px solid var(--border-subtle); transition: border-color 0.15s;
  }
  .inline-img-card:hover { border-color: var(--border-focus); }
  .inline-img-card img {
    width: 100%; height: 125px; object-fit: cover; display: block; cursor: zoom-in;
  }
  .inline-img-meta { padding: 6px 8px; font-size: 11px; color: var(--text-secondary); }
  .inline-img-meta a { color: var(--accent-cyan); text-decoration: none; }

  .msg-stats {
    margin-top: 10px; padding-top: 8px; border-top: 1px solid var(--border-subtle);
    font-size: 10.5px; color: var(--text-muted); font-family: var(--font-mono);
  }
  .composer-wrap {
    padding: 12px 20px 14px; border-top: 1px solid var(--border-subtle);
    background: rgba(8, 9, 12, 0.9); backdrop-filter: blur(16px);
  }
  .composer-inner { max-width: 840px; margin: 0 auto; display: flex; flex-direction: column; gap: 8px; }

  .controls-bar {
    display: flex; flex-wrap: wrap; align-items: center; justify-content: space-between; gap: 10px;
    font-size: 11.5px; color: var(--text-secondary); font-family: var(--font-mono);
  }
  .seg-group {
    display: inline-flex; padding: 2px; border-radius: 6px;
    background: #060709; border: 1px solid var(--border-subtle);
  }
  .seg-btn {
    padding: 3px 9px; border-radius: 4px; border: none; background: transparent;
    color: var(--text-secondary); font-size: 11px; font-family: var(--font-mono);
    font-weight: 500; cursor: pointer; transition: all 0.15s;
  }
  .seg-btn.active {
    background: #27272a; color: var(--text-primary);
    box-shadow: inset 0 1px 0 rgba(255,255,255,0.1);
  }
  .slider-wrap { display: flex; align-items: center; gap: 7px; }
  .slider-wrap input[type="range"] { accent-color: #e4e4e7; width: 88px; cursor: pointer; }
  .toggle-wrap { display: flex; align-items: center; gap: 6px; cursor: pointer; user-select: none; }
  .toggle-wrap input { accent-color: #e4e4e7; width: 13px; height: 13px; cursor: pointer; }

  .input-box {
    display: flex; align-items: flex-end; gap: 10px; padding: 10px 12px;
    border-radius: var(--radius-md); background: var(--bg-elevated);
    border: 1px solid var(--border-default);
    box-shadow: inset 0 1px 0 rgba(255,255,255,0.03);
    transition: border-color 0.15s, box-shadow 0.15s;
  }
  .input-box:focus-within {
    border-color: var(--border-focus);
    box-shadow: 0 0 0 2px rgba(255, 255, 255, 0.06);
  }
  .input-box.drag-over {
    border-color: var(--accent-cyan); background: rgba(34, 211, 238, 0.05);
  }
  #msgInput {
    flex: 1; background: transparent; border: none; outline: none; resize: none;
    color: var(--text-primary); font-family: var(--font-sans); font-size: 13.5px;
    line-height: 1.5; max-height: 180px; min-height: 40px;
  }
  .action-btns { display: flex; gap: 6px; flex-shrink: 0; }

  .btn {
    position: relative; overflow: hidden;
    display: inline-flex; align-items: center; justify-content: center; gap: 6px;
    padding: 7px 12px; border-radius: 6px; font-size: 12px; font-weight: 500;
    border: 1px solid var(--border-default); background: #14161d;
    color: var(--text-primary); cursor: pointer;
    transition: transform 0.22s cubic-bezier(0.22, 1, 0.36, 1), background 0.18s ease, border-color 0.18s ease, box-shadow 0.22s ease;
    white-space: nowrap; font-family: var(--font-sans);
  }
  .btn:hover:not(:disabled) {
    background: #1d202b; border-color: rgba(255, 255, 255, 0.28);
    transform: translateY(-1.5px);
    box-shadow: 0 6px 16px rgba(0, 0, 0, 0.4), inset 0 1px 0 rgba(255, 255, 255, 0.1);
  }
  .btn:active:not(:disabled) {
    transform: translateY(0) scale(0.97);
  }
  .btn:disabled { opacity: 0.4; cursor: not-allowed; }
  .btn-primary {
    background: #f4f4f6; border-color: #ffffff; color: #09090b; font-weight: 600;
    box-shadow: 0 2px 10px rgba(255, 255, 255, 0.08);
  }
  .btn-primary:hover:not(:disabled) {
    background: #ffffff; border-color: #ffffff;
    box-shadow: 0 6px 20px rgba(255, 255, 255, 0.22);
  }
  .btn-stop {
    background: rgba(239, 68, 68, 0.12); border-color: rgba(239, 68, 68, 0.35); color: #fca5a5;
  }
  .btn-stop:hover:not(:disabled) { background: rgba(239, 68, 68, 0.22); color: white; }
  .btn-sm { padding: 4px 9px; font-size: 11px; border-radius: 5px; font-family: var(--font-mono); }

  .quick-strip {
    display: flex; flex-wrap: wrap; gap: 6px; align-items: center;
  }
  .tabs-nav {
    display: flex; border-bottom: 1px solid var(--border-subtle); padding: 0 8px;
    background: #07080b; flex-shrink: 0; overflow-x: auto;
  }
  .tab-btn {
    padding: 11px 12px; border: none; background: transparent; color: var(--text-secondary);
    font-size: 11.5px; font-family: var(--font-mono); font-weight: 500; cursor: pointer;
    position: relative; transition: color 0.15s; white-space: nowrap;
    text-transform: uppercase; letter-spacing: 0.04em;
  }
  .tab-btn:hover { color: var(--text-primary); }
  .tab-btn.active { color: var(--text-primary); }
  .tab-btn.active::after {
    content: ""; position: absolute; left: 8px; right: 8px; bottom: 0; height: 1.5px;
    background: var(--text-primary);
  }
  .tab-body {
    flex: 1 1 auto; min-height: 0; overflow-y: auto; overflow-x: hidden; padding: 14px;
    display: none; flex-direction: column; gap: 12px; -webkit-overflow-scrolling: touch;
  }
  .tab-body.active { display: flex; }
  .card {
    flex-shrink: 0; padding: 13px; border-radius: var(--radius-md);
    background: var(--bg-elevated); border: 1px solid var(--border-subtle);
  }
  .card-title {
    font-size: 11px; font-family: var(--font-mono); text-transform: uppercase;
    letter-spacing: 0.06em; font-weight: 600; margin: 0 0 10px;
    color: var(--text-secondary); display: flex; justify-content: space-between; align-items: center;
  }

  .artifact-row {
    display: flex; align-items: center; justify-content: space-between; gap: 10px;
    padding: 8px 10px; border-radius: var(--radius-sm);
    background: #07080b; border: 1px solid var(--border-subtle);
    margin-bottom: 6px;
    transition: transform 0.22s cubic-bezier(0.22, 1, 0.36, 1), border-color 0.18s ease, box-shadow 0.22s ease;
  }
  .artifact-row:hover {
    border-color: var(--border-focus);
    transform: translateX(3px);
    box-shadow: -2px 0 0 var(--accent-cyan), 0 6px 18px rgba(0, 0, 0, 0.4);
  }
  .art-info { min-width: 0; flex: 1; }
  .art-name { font-size: 12px; font-weight: 500; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
  .art-meta { font-size: 10.5px; color: var(--text-muted); font-family: var(--font-mono); }

  .gallery-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 8px; }
  .gal-card {
    border-radius: var(--radius-sm); overflow: hidden; background: #07080b;
    border: 1px solid var(--border-subtle);
    transition: transform 0.25s cubic-bezier(0.22, 1, 0.36, 1), border-color 0.2s ease, box-shadow 0.25s ease;
  }
  .gal-card:hover {
    transform: translateY(-2px); border-color: var(--border-focus);
    box-shadow: 0 10px 24px rgba(0, 0, 0, 0.5);
  }
  .gal-card img, .inline-img-card img {
    width: 100%; height: 125px; object-fit: cover; display: block; cursor: zoom-in;
    transition: transform 0.4s cubic-bezier(0.22, 1, 0.36, 1), filter 0.3s ease;
  }
  .gal-card:hover img, .inline-img-card:hover img {
    transform: scale(1.05); filter: brightness(1.06);
  }
  .skill-card {
    transition: transform 0.22s cubic-bezier(0.22, 1, 0.36, 1), border-color 0.18s ease, box-shadow 0.22s ease;
  }
  .skill-card:hover {
    transform: translateY(-2px); border-color: var(--border-focus);
    box-shadow: 0 8px 22px rgba(0, 0, 0, 0.45);
  }
  .gal-foot {
    padding: 6px 8px; display: flex; justify-content: space-between; align-items: center;
    gap: 6px; font-size: 10.5px; font-family: var(--font-mono); color: var(--text-secondary);
  }

  .skill-card {
    padding: 11px; border-radius: var(--radius-sm); background: #07080b;
    border: 1px solid var(--border-subtle); margin-bottom: 8px;
  }
  .skill-head { display: flex; justify-content: space-between; align-items: center; margin-bottom: 4px; }
  .skill-name { font-family: var(--font-mono); font-size: 12px; font-weight: 600; color: var(--text-primary); }
  .skill-desc { font-size: 11.5px; color: var(--text-secondary); line-height: 1.45; margin-bottom: 6px; }

  .field-input {
    width: 100%; padding: 7px 10px; border-radius: 5px; background: #060709;
    border: 1px solid var(--border-default); color: var(--text-primary); font-size: 12px;
    font-family: var(--font-mono); outline: none;
  }
  .field-input:focus { border-color: var(--border-focus); }

  .term-box {
    background: #040507; border: 1px solid var(--border-subtle); border-radius: var(--radius-sm);
    padding: 10px; font-family: var(--font-mono); font-size: 11px; line-height: 1.45;
    color: var(--text-secondary); max-height: 260px; overflow-y: auto; white-space: pre-wrap; word-break: break-all;
  }
  .modal-backdrop {
    position: fixed; inset: 0; z-index: 50; background: rgba(3, 4, 7, 0.85);
    backdrop-filter: blur(8px); display: none; align-items: center; justify-content: center; padding: 24px;
  }
  .modal-backdrop.open { display: flex; }
  .modal-card {
    max-width: 92vw; max-height: 90vh; width: 920px; border-radius: var(--radius-lg);
    background: #0b0d12; border: 1px solid var(--border-default);
    display: flex; flex-direction: column; overflow: hidden; box-shadow: 0 24px 64px rgba(0,0,0,0.75);
  }
  .modal-head {
    padding: 10px 14px; border-bottom: 1px solid var(--border-subtle);
    display: flex; justify-content: space-between; align-items: center;
    font-size: 12px; font-family: var(--font-mono); font-weight: 600;
  }
  .modal-body { flex: 1; overflow: auto; padding: 16px; display: grid; place-items: center; min-height: 300px; }
  .modal-body img { max-width: 100%; max-height: 75vh; border-radius: 6px; }
  .modal-body iframe { width: 100%; height: 74vh; border: none; border-radius: 6px; background: white; }
  .modal-body pre {
    width: 100%; max-height: 74vh; overflow: auto; background: #050608; padding: 14px;
    border-radius: 6px; font-family: var(--font-mono); font-size: 11.5px; color: #e4e4e7; white-space: pre-wrap;
  }
  .vnc-overlay {
    position: fixed; inset: 14px; z-index: 48; display: none; flex-direction: column;
    background: #07080c; border: 1px solid var(--border-default); border-radius: var(--radius-lg);
    box-shadow: 0 28px 80px rgba(0, 0, 0, 0.85); overflow: hidden;
  }
  .vnc-overlay.open { display: flex; }
  .vnc-head {
    height: 44px; flex-shrink: 0; display: flex; align-items: center; justify-content: space-between;
    padding: 0 14px; border-bottom: 1px solid var(--border-subtle); background: #0a0b10;
    font-family: var(--font-mono); font-size: 11.5px;
  }
  .vnc-frame {
    flex: 1; width: 100%; border: none; background: #000;
  }
  .watermark {
    position: fixed; bottom: 10px; right: 14px; z-index: 45;
    pointer-events: none; user-select: none;
    font-family: var(--font-mono); font-size: 11px; font-weight: 500;
    letter-spacing: 0.04em; color: rgba(244, 244, 246, 0.28);
    text-shadow: 0 1px 2px rgba(0, 0, 0, 0.8);
  }

  .toast {
    position: fixed; bottom: 34px; right: 14px; z-index: 60;
    padding: 8px 13px; border-radius: 6px; background: #14161e; border: 1px solid var(--border-default);
    color: var(--text-primary); font-size: 12px; font-family: var(--font-mono);
    box-shadow: 0 10px 30px rgba(0,0,0,0.5);
    opacity: 0; transform: translateY(8px); pointer-events: none; transition: all 0.18s ease;
  }
  .toast.show { opacity: 1; transform: translateY(0); }

  @media (max-width: 920px) {
    .inspector-pane { position: fixed; right: 0; top: 48px; bottom: 0; width: 92vw; max-width: 460px; z-index: 30; }
    .resizer { display: none; }
    .preset-grid { grid-template-columns: 1fr; }
  }
</style>
</head>
<body>
<div class="ambient-canvas"></div>

<div class="app-shell">
  <header class="topbar">
    <div class="brand">
      <div class="brand-mark">
        <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.8">
          <polygon points="12 2 2 7 12 12 22 7 12 2"></polygon>
          <polyline points="2 17 12 22 22 17"></polyline>
          <polyline points="2 12 12 17 22 12"></polyline>
        </svg>
      </div>
      <span class="brand-title">DEEPSEEK-V4-PRO / QWEN3.5-9B</span>
      <div class="brand-divider"></div>
      <span id="visionBadge" class="meta-tag">VISION: INIT</span>
      <span id="tunnelBadge" class="meta-tag clickable" style="display:none" title="Copy public Cloudflare URL">TUNNEL: COPY URL</span>
    </div>

    <div class="topbar-right">
      <div class="seg-group" id="providerSeg" title="Switch between Local GPU model and Gemini API">
        <button type="button" class="seg-btn active" data-prov="local" onclick="switchProvider('local')">LOCAL</button>
        <button type="button" class="seg-btn" data-prov="gemini" onclick="switchProvider('gemini')">GEMINI</button>
      </div>
      <div class="ctx-meter" title="Active context window utilization">
        <span>CTX</span>
        <span id="ctxText">0.0k / 32.8k</span>
        <div class="ctx-track"><div id="ctxFill" class="ctx-fill"></div></div>
      </div>
      <div id="statusBadge" class="status-badge">
        <span class="status-dot"></span>
        <span id="statusText">IDLE</span>
      </div>
      <button id="toggleInspectorBtn" class="btn btn-sm" title="Toggle Inspector">PANEL</button>
    </div>
  </header>
  <div class="workspace-grid">
    <section class="chat-pane">
      <div id="chatScroll" class="chat-scroll">
        <div id="welcomeHero" class="welcome-hero">
          <div class="hero-kicker">
            <span class="hero-kicker-dot"></span>
            <span>AUTONOMOUS EXECUTION ENVIRONMENT</span>
          </div>
          <h1>Minimalist Agentic Studio</h1>
          <p>Root Ubuntu shell, uncensored web and image search (<code>safesearch=off</code>), headless Chromium, virtual X11 desktop, and portable <code>SKILL.md</code> workflows.</p>
          <div class="preset-grid">
            <div class="preset-card" onclick="applyPreset('pdf')">
              <div class="p-tag">DOCUMENT STUDIO</div>
              <div class="p-title">Illustrated JWST Technical Brief</div>
              <div class="p-desc">Fetch high-resolution imagery and compile a publication-grade PDF report in artifacts.</div>
            </div>
            <div class="preset-card" onclick="applyPreset('video')">
              <div class="p-tag">MOTION ENGINE</div>
              <div class="p-title">Quantum Tunneling 1080p Render</div>
              <div class="p-desc">Synthesize a deterministic 60fps HTML5 canvas waveform animation encoded via FFmpeg.</div>
            </div>
            <div class="preset-card" onclick="applyPreset('ipynb')">
              <div class="p-tag">COMPUTATIONAL</div>
              <div class="p-title">SciPy Curve-Fitting Notebook</div>
              <div class="p-desc">Build a validated, runnable .ipynb with synthetic datasets and regression plots.</div>
            </div>
            <div class="preset-card" onclick="applyPreset('desktop')">
              <div class="p-tag">X11 OPERATOR</div>
              <div class="p-title">Inspect Virtual Desktop State</div>
              <div class="p-desc">Capture the 1920x1080 Xvfb display and run multimodal coordinate analysis.</div>
            </div>
          </div>
        </div>
      </div>
      <div class="composer-wrap">
        <div class="composer-inner">
          <div class="controls-bar">
            <div style="display:flex; align-items:center; gap:8px;">
              <span>REASONING</span>
              <div class="seg-group" id="reasoningSeg">
                <button type="button" class="seg-btn" data-mode="off">OFF</button>
                <button type="button" class="seg-btn active" data-mode="brief">BRIEF</button>
                <button type="button" class="seg-btn" data-mode="deep">DEEP</button>
              </div>
            </div>

            <div class="slider-wrap">
              <span>STEPS</span>
              <input id="stepsSlider" type="range" min="4" max="120" value="36" />
              <span id="stepsVal" style="color:var(--text-primary); min-width:24px;">36</span>
              <label class="toggle-wrap" style="margin-left:4px;" title="Run endlessly until task completes or aborted">
                <input id="endlessCb" type="checkbox" onchange="toggleEndlessSteps()" />
                <span>ENDLESS</span>
              </label>
            </div>

            <label class="toggle-wrap">
              <input id="visionCb" type="checkbox" checked />
              <span>AUTO-VISION</span>
            </label>
          </div>

          <div id="inputDropZone" class="input-box">
            <textarea id="msgInput" rows="2" placeholder="Enter instruction, research query, or system command... (Enter to execute, Shift+Enter for newline)"></textarea>
            <div class="action-btns">
              <button id="sendBtn" class="btn btn-primary" onclick="sendMessage()">EXECUTE</button>
              <button id="stopBtn" class="btn btn-stop" onclick="stopAgent()" disabled>ABORT</button>
            </div>
          </div>

          <div class="quick-strip">
            <input id="fileUploadInput" type="file" multiple style="display:none" onchange="handleFileUpload(this.files)" />
            <button class="btn btn-sm" onclick="document.getElementById('fileUploadInput').click()">UPLOAD FILE</button>
            <button class="btn btn-sm" onclick="applyPreset('pdf')">PDF BRIEF</button>
            <button class="btn btn-sm" onclick="applyPreset('video')">MOTION MP4</button>
            <button class="btn btn-sm" onclick="applyPreset('ipynb')">NOTEBOOK</button>
            <button class="btn btn-sm" onclick="applyPreset('genimg')">GENERATE IMG</button>
            <button class="btn btn-sm" onclick="sendContinue()">CONTINUE</button>
            <button class="btn btn-sm" onclick="clearSession()">RESET</button>
          </div>
        </div>
      </div>
    </section>

    <div id="paneResizer" class="resizer" title="Drag to resize panes"></div>
    <aside id="inspectorPane" class="inspector-pane">
      <nav class="tabs-nav">
        <button class="tab-btn active" data-tab="artifacts" onclick="switchTab('artifacts')">ARTIFACTS</button>
        <button class="tab-btn" data-tab="images" onclick="switchTab('images')">MEDIA</button>
        <button class="tab-btn" data-tab="skills" onclick="switchTab('skills')">SKILLS</button>
        <button class="tab-btn" data-tab="screen" onclick="switchTab('screen')">VIEWPORT</button>
        <button class="tab-btn" data-tab="system" onclick="switchTab('system')">SYSTEM</button>
      </nav>
      <div id="tab-artifacts" class="tab-body active">
        <div id="videoCard" class="card" style="display:none">
          <div class="card-title"><span>LATEST VIDEO OUTPUT</span><span id="videoName"></span></div>
          <video id="videoPlayer" controls style="width:100%; border-radius:6px; max-height:220px; background:#000"></video>
        </div>

        <div class="card">
          <div class="card-title">
            <span>ARTIFACTS (/content/workspace/artifacts)</span>
            <div style="display:flex; gap:6px">
              <button class="btn btn-sm" onclick="refreshState()">SYNC</button>
              <button class="btn btn-sm" onclick="exportChat()">EXPORT MD</button>
            </div>
          </div>
          <div id="artifactsList"><div style="font-size:11.5px; color:var(--text-muted); font-family:var(--font-mono)">No artifacts generated yet.</div></div>
        </div>
      </div>
      <div id="tab-images" class="tab-body">
        <div class="card">
          <div class="card-title">
            <span>DOWNLOADED MEDIA CACHE</span>
            <button class="btn btn-sm" onclick="refreshState()">SYNC</button>
          </div>
          <div id="imagesGallery" class="gallery-grid"><div style="font-size:11.5px; color:var(--text-muted); font-family:var(--font-mono)">No media files cached.</div></div>
        </div>
      </div>
      <div id="tab-skills" class="tab-body">
        <div class="card">
          <div class="card-title"><span>INSTALL PORTABLE SKILL.MD</span></div>
          <div style="display:flex; flex-direction:column; gap:8px;">
            <input id="ghUrlIn" class="field-input" placeholder="https://github.com/anthropics/skills/tree/main/skills/pdf" />
            <div style="display:flex; gap:8px;">
              <input id="ghSubIn" class="field-input" placeholder="Subpath (optional)" />
              <button class="btn btn-primary btn-sm" onclick="installGithubSkill()">CLONE</button>
            </div>
            <div style="display:flex; align-items:center; justify-content:space-between; gap:8px; padding-top:6px; border-top:1px solid var(--border-subtle);">
              <span style="font-size:11px; color:var(--text-secondary); font-family:var(--font-mono)">UPLOAD SKILL.MD / .ZIP</span>
              <input id="skillFileIn" type="file" accept=".md,.zip" style="display:none" onchange="uploadSkillFile(this.files[0])" />
              <button class="btn btn-sm" onclick="document.getElementById('skillFileIn').click()">SELECT FILE</button>
            </div>
            <div id="skillStatusMsg" style="font-size:11.5px; font-family:var(--font-mono); color:var(--accent-cyan)"></div>
          </div>
        </div>

        <div class="card">
          <div class="card-title"><span>REGISTERED SKILLS</span></div>
          <div id="skillsList"></div>
        </div>
      </div>
      <div id="tab-screen" class="tab-body">
        <div class="card">
          <div class="card-title">
            <span>X11 / CHROMIUM VIEWPORT</span>
            <div style="display:flex; gap:6px;">
              <button class="btn btn-sm" onclick="captureDesktop()">CAPTURE X11</button>
              <button class="btn btn-sm btn-primary" onclick="openVncControl()">EXPAND &amp; USE</button>
            </div>
          </div>
          <div id="screenWrap" style="text-align:center">
            <div style="font-size:11.5px; color:var(--text-muted); font-family:var(--font-mono); padding:24px 0;">No frame captured yet. Click EXPAND &amp; USE to control the Colab PC live.</div>
          </div>
        </div>
      </div>
      <div id="tab-system" class="tab-body">
        <div class="card">
          <div class="card-title">
            <span>AI PROVIDER &amp; GEMINI API (OPTIONAL)</span>
            <span id="provStatusTag" class="meta-tag">LOCAL</span>
          </div>
          <div style="display:flex; flex-direction:column; gap:8px;">
            <input id="geminiKeyIn" type="password" class="field-input" placeholder="Paste Gemini API Key (AIza...) — optional" />
            <div style="display:flex; gap:8px;">
              <select id="geminiModelSel" class="field-input" style="flex:1;">
                <option value="gemini-2.5-flash">gemini-2.5-flash (Fast &amp; Agentic)</option>
                <option value="gemini-2.5-pro">gemini-2.5-pro (Deep Reasoning)</option>
                <option value="gemini-2.0-flash">gemini-2.0-flash</option>
              </select>
              <button class="btn btn-primary btn-sm" onclick="saveGeminiConfig()">USE GEMINI</button>
              <button class="btn btn-sm" onclick="switchProvider('local')">USE LOCAL</button>
            </div>
          </div>
        </div>
        <div class="card">
          <div class="card-title">
            <span>RUNTIME &amp; GPU TELEMETRY</span>
            <div style="display:flex; gap:6px">
              <button class="btn btn-sm" onclick="refreshState()">SYNC</button>
              <button class="btn btn-sm" onclick="restartServer()">RESTART LLM</button>
            </div>
          </div>
          <div id="sysTelemetry" style="font-size:12px; line-height:1.7; font-family:var(--font-mono); color:var(--text-secondary)">Loading telemetry...</div>
        </div>

        <div class="card">
          <div class="card-title"><span>LLAMA.CPP SERVER LOG</span></div>
          <div id="serverLogBox" class="term-box">(loading...)</div>
        </div>
      </div>
    </aside>
  </div>
</div>
<div id="vncOverlay" class="vnc-overlay">
  <div class="vnc-head">
    <div style="display:flex; align-items:center; gap:10px;">
      <span class="status-dot"></span>
      <span style="font-weight:600; color:var(--text-primary)">INTERACTIVE X11 CONTROL (DISPLAY :99 · 1920x1080)</span>
      <span id="vncStatusTag" class="meta-tag">CONNECTING</span>
    </div>
    <div style="display:flex; gap:8px;">
      <button class="btn btn-sm" onclick="reloadVncFrame()">RECONNECT</button>
      <button class="btn btn-sm" onclick="captureDesktop()">SNAPSHOT</button>
      <button class="btn btn-sm btn-primary" onclick="closeVncControl()">COLLAPSE</button>
    </div>
  </div>
  <iframe id="vncFrame" class="vnc-frame" src="about:blank"></iframe>
</div>
<div id="previewModal" class="modal-backdrop" onclick="if(event.target===this) closeModal()">
  <div class="modal-card">
    <div class="modal-head">
      <span id="modalTitle">PREVIEW</span>
      <div style="display:flex; gap:8px">
        <a id="modalDownloadLink" href="#" download class="btn btn-sm btn-primary" style="text-decoration:none">DOWNLOAD</a>
        <button class="btn btn-sm" onclick="closeModal()">CLOSE</button>
      </div>
    </div>
    <div id="modalBody" class="modal-body"></div>
  </div>
</div>

<div class="watermark">CookieFIlled</div>
<div id="toast" class="toast"></div>

<script>
const PRESETS = {
  pdf: "Activate the web-researcher and document-studio skills. Search the web and use image_search to find high-resolution images of the James Webb Space Telescope Pillars of Creation, compile a styled PDF brief with the embedded images at /content/workspace/artifacts/jwst_report.pdf, and reference those images in your response.",
  video: "Activate the motion-video skill and create a sleek 5-second (150 frames @ 30fps) kinetic-typography and animated waveform motion graphics video explaining Quantum Tunneling, saved to /content/workspace/artifacts/quantum_tunneling.mp4.",
  ipynb: "Activate the document-studio skill and create a clean, runnable Jupyter Notebook at /content/workspace/artifacts/data_analysis_demo.ipynb that generates synthetic scientific data, runs statistical curve fitting with SciPy, and plots publication-quality figures.",
  desktop: "Activate the desktop-operator skill, capture a screenshot of the virtual desktop, and describe what is currently visible on the screen.",
  genimg: "Activate the image-studio skill and call generate_image with allow_18_plus=true to generate a high-detail cinematic image of: "
};

let reasoningMode = "brief";
let isStreaming = false;
let activeAbortController = null;
let publicTunnelUrl = "";

function showToast(msg) {
  const t = document.getElementById("toast");
  t.textContent = msg;
  t.classList.add("show");
  clearTimeout(t._timer);
  t._timer = setTimeout(() => t.classList.remove("show"), 3000);
}

function escapeHtml(str) {
  return String(str || "")
    .replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;")
    .replace(/"/g, "&quot;").replace(/'/g, "&#39;");
}

function setStatus(state, label) {
  const b = document.getElementById("statusBadge");
  const txt = document.getElementById("statusText");
  b.className = "status-badge" + (state === "working" ? " working" : state === "error" ? " error" : "");
  txt.textContent = label;
  document.body.classList.toggle("is-running", state === "working");
  document.getElementById("sendBtn").disabled = (state === "working");
  document.getElementById("stopBtn").disabled = (state !== "working");
}

function updateContextMeter(used, max) {
  max = max || 32768;
  used = used || 0;
  const pct = Math.min(100, Math.max(2, (used / max) * 100));
  document.getElementById("ctxText").textContent = `${(used/1000).toFixed(1)}k / ${(max/1000).toFixed(1)}k`;
  const fill = document.getElementById("ctxFill");
  fill.style.width = pct + "%";
  if (pct > 85) fill.style.background = "#ef4444";
  else if (pct > 65) fill.style.background = "#f59e0b";
  else fill.style.background = "#e4e4e7";
}

document.querySelectorAll("#reasoningSeg .seg-btn").forEach(btn => {
  btn.addEventListener("click", () => {
    document.querySelectorAll("#reasoningSeg .seg-btn").forEach(b => b.classList.remove("active"));
    btn.classList.add("active");
    reasoningMode = btn.dataset.mode;
  });
});

const stepsSlider = document.getElementById("stepsSlider");
stepsSlider.addEventListener("input", () => {
  document.getElementById("endlessCb").checked = false;
  document.getElementById("stepsVal").textContent = stepsSlider.value;
});
function toggleEndlessSteps() {
  const on = document.getElementById("endlessCb").checked;
  stepsSlider.disabled = on;
  document.getElementById("stepsVal").textContent = on ? "INF" : stepsSlider.value;
}

(function initResizer() {
  const resizer = document.getElementById("paneResizer");
  const inspector = document.getElementById("inspectorPane");
  let dragging = false;
  resizer.addEventListener("mousedown", () => { dragging = true; resizer.classList.add("dragging"); document.body.style.userSelect = "none"; });
  window.addEventListener("mousemove", (e) => {
    if (!dragging) return;
    const w = window.innerWidth - e.clientX;
    const clamped = Math.max(320, Math.min(window.innerWidth * 0.64, w));
    inspector.style.width = clamped + "px";
  });
  window.addEventListener("mouseup", () => {
    if (dragging) { dragging = false; resizer.classList.remove("dragging"); document.body.style.userSelect = ""; }
  });
})();

document.getElementById("toggleInspectorBtn").addEventListener("click", () => {
  document.getElementById("inspectorPane").classList.toggle("collapsed");
});

function switchTab(tabName) {
  document.querySelectorAll(".tab-btn").forEach(b => b.classList.toggle("active", b.dataset.tab === tabName));
  document.querySelectorAll(".tab-body").forEach(p => p.classList.toggle("active", p.id === "tab-" + tabName));
}

function applyPreset(key) {
  const box = document.getElementById("msgInput");
  box.value = PRESETS[key] || "";
  box.focus();
  autoResizeInput();
}

function autoResizeInput() {
  const box = document.getElementById("msgInput");
  box.style.height = "auto";
  box.style.height = Math.min(180, Math.max(40, box.scrollHeight)) + "px";
}
const msgInput = document.getElementById("msgInput");
msgInput.addEventListener("input", autoResizeInput);
msgInput.addEventListener("keydown", (e) => {
  if (e.key === "Enter" && !e.shiftKey) {
    e.preventDefault();
    sendMessage();
  }
});

const dropZone = document.getElementById("inputDropZone");
["dragenter", "dragover"].forEach(ev => dropZone.addEventListener(ev, e => { e.preventDefault(); dropZone.classList.add("drag-over"); }));
["dragleave", "drop"].forEach(ev => dropZone.addEventListener(ev, e => { e.preventDefault(); dropZone.classList.remove("drag-over"); }));
dropZone.addEventListener("drop", e => {
  if (e.dataTransfer && e.dataTransfer.files && e.dataTransfer.files.length) {
    handleFileUpload(e.dataTransfer.files);
  }
});

async function handleFileUpload(fileList) {
  if (!fileList || !fileList.length) return;
  const fd = new FormData();
  for (const f of fileList) fd.append("files", f);
  showToast(`Uploading ${fileList.length} file(s)...`);
  try {
    const res = await fetch("/api/upload", { method: "POST", body: fd });
    const data = await res.json();
    if (data.note) {
      msgInput.value = (msgInput.value ? msgInput.value + "\n" : "") + data.note;
      autoResizeInput();
      showToast(`Uploaded ${data.saved.length} file(s)`);
    }
  } catch (err) {
    showToast("Upload failed: " + err.message);
  }
}

function splitReasoningAndAnswer(rawText) {
  let text = String(rawText || "");
  let thinkHtml = "";
  let isLiveThinking = false;

  const detailsMatch = text.match(/<details><summary>Reasoning<\/summary>\s*([\s\S]*?)\s*<\/details>/);
  if (detailsMatch) {
    thinkHtml = detailsMatch[1].trim();
    text = text.replace(detailsMatch[0], "").trim();
  }

  const liveMatch = text.match(/\*Thinking(?:…|\.{3})\*\s*`([^`]*)`/);
  if (liveMatch) {
    isLiveThinking = true;
    thinkHtml = (thinkHtml ? thinkHtml + "\n" : "") + liveMatch[1];
    text = text.replace(liveMatch[0], "").trim();
  }
  return { thinkText: thinkHtml, isLiveThinking, answerMd: text };
}

function renderMarkdownSafe(mdText) {
  if (!mdText) return "";
  if (window.marked) {
    marked.setOptions({ breaks: true, gfm: true });
    return marked.parse(mdText);
  }
  return "<pre>" + escapeHtml(mdText) + "</pre>";
}

function enhanceCodeAndMath(container) {
  if (!container) return;
  container.querySelectorAll("pre code").forEach(block => {
    if (window.hljs && !block.dataset.highlighted) {
      hljs.highlightElement(block);
      block.dataset.highlighted = "1";
    }
    const pre = block.parentElement;
    if (pre && !pre.querySelector(".copy-code-btn")) {
      const btn = document.createElement("button");
      btn.className = "copy-code-btn";
      btn.textContent = "COPY";
      btn.onclick = () => {
        navigator.clipboard.writeText(block.innerText);
        btn.textContent = "COPIED";
        setTimeout(() => btn.textContent = "COPY", 1400);
      };
      pre.appendChild(btn);
    }
  });
  if (window.renderMathInElement) {
    try {
      renderMathInElement(container, {
        delimiters: [
          { left: "$$", right: "$$", display: true },
          { left: "$", right: "$", display: false }
        ],
        throwOnError: false
      });
    } catch (e) {}
  }
}

function buildToolLogHtml(toolLog, collapsed = false) {
  if (!toolLog || !toolLog.length) return "";
  const items = toolLog.map(item => {
    const cls = item.kind === "start" ? "start" : item.kind === "end" ? "end" : "status";
    const tag = item.kind === "start" ? "EXEC" : item.kind === "end" ? "DONE" : "INFO";
    return `<div class="tool-item ${cls}"><span class="tool-tag">${tag}</span><span>${escapeHtml(item.text)}</span></div>`;
  }).join("");
  return `
    <div class="tools-drawer ${collapsed ? "collapsed" : ""}">
      <button type="button" class="tools-header" onclick="this.parentElement.classList.toggle('collapsed')">
        <span>TOOL TRACE (${toolLog.length})</span>
        <span>TOGGLE</span>
      </button>
      <div class="tools-list">${items}</div>
    </div>`;
}

function buildImagesHtml(images) {
  if (!images || !images.length) return "";
  const cards = images.map(im => `
    <div class="inline-img-card">
      <img src="${escapeHtml(im.url)}" alt="${escapeHtml(im.title)}" onclick="openImageModal('${escapeHtml(im.url)}', '${escapeHtml(im.title || im.name)}')" />
      <div class="inline-img-meta">
        <div style="font-weight:500; color:var(--text-primary); white-space:nowrap; overflow:hidden; text-overflow:ellipsis">${escapeHtml(im.title || im.name)}</div>
        <div style="font-family:var(--font-mono); font-size:10px">${escapeHtml(im.size || "")} ${im.source_url ? `· <a href="${escapeHtml(im.source_url)}" target="_blank">SOURCE</a>` : ""}</div>
      </div>
    </div>`).join("");
  return `<div class="inline-imgs">${cards}</div>`;
}

function renderAssistantCardContent(cardEl, { rawText, toolLog, heartbeat, stats, images, running }) {
  const { thinkText, isLiveThinking, answerMd } = splitReasoningAndAnswer(rawText);
  const openAttr = (running && isLiveThinking) ? "open" : "";
  const thinkBlock = thinkText ? `
    <div class="think-box ${openAttr}">
      <button type="button" class="think-toggle" onclick="this.parentElement.classList.toggle('open')">
        <span class="${isLiveThinking ? "think-shimmer" : ""}">${isLiveThinking ? "REASONING ACTIVE..." : "REASONING TRACE"}</span>
        <span style="opacity:0.75">${thinkText.length} CHARS</span>
      </button>
      <div class="think-content-wrap"><div class="think-content">${escapeHtml(thinkText)}</div></div>
    </div>` : "";

  const toolsBlock = buildToolLogHtml(toolLog, !running);
  const answerHtml = answerMd ? renderMarkdownSafe(answerMd) : (running ? `<span style="color:var(--text-secondary); font-family:var(--font-mono); font-size:12px">Awaiting output...</span>` : "");
  const hbHtml = (running && heartbeat) ? `<div class="live-hb"><span class="spinner"></span><span>${escapeHtml(heartbeat)}</span></div>` : "";
  const imgsHtml = buildImagesHtml(images);
  const statsHtml = stats ? `<div class="msg-stats">${escapeHtml(stats)}</div>` : "";

  cardEl.innerHTML = `
    <div class="assistant-header">
      <span class="assistant-id"><span class="assistant-id-dot"></span><span>AGENT</span></span>
      <span>${running ? "RUNNING" : "COMPLETE"}</span>
    </div>
    ${toolsBlock}
    ${thinkBlock}
    <div class="md-body">${answerHtml}</div>
    ${hbHtml}
    ${imgsHtml}
    ${statsHtml}
  `;
  enhanceCodeAndMath(cardEl);
}

function appendUserBubble(text) {
  const hero = document.getElementById("welcomeHero");
  if (hero) hero.style.display = "none";
  const scroll = document.getElementById("chatScroll");
  const row = document.createElement("div");
  row.className = "msg-row user";
  row.innerHTML = `<div class="user-bubble">${escapeHtml(text)}</div>`;
  scroll.appendChild(row);
  scroll.scrollTop = scroll.scrollHeight;
}

function appendAssistantCard() {
  const scroll = document.getElementById("chatScroll");
  const row = document.createElement("div");
  row.className = "msg-row assistant";
  const card = document.createElement("div");
  card.className = "assistant-card";
  row.appendChild(card);
  scroll.appendChild(row);
  scroll.scrollTop = scroll.scrollHeight;
  return card;
}

async function sendMessage(overrideText = null) {
  if (isStreaming) return;
  const text = (overrideText !== null ? overrideText : msgInput.value).trim();
  if (!text) return;

  if (overrideText === null) {
    msgInput.value = "";
    autoResizeInput();
  }

  appendUserBubble(text);
  const card = appendAssistantCard();
  const state = { rawText: "", toolLog: [], heartbeat: "Initializing turn...", stats: "", images: [], running: true };
  renderAssistantCardContent(card, state);

  isStreaming = true;
  setStatus("working", "BUSY");
  activeAbortController = new AbortController();

  try {
    const resp = await fetch("/api/chat/stream", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({
        message: text,
        reasoning_mode: reasoningMode,
        max_steps: document.getElementById("endlessCb").checked ? 0 : (parseInt(stepsSlider.value, 10) || 36),
        auto_vision: document.getElementById("visionCb").checked,
      }),
      signal: activeAbortController.signal,
    });

    const reader = resp.body.getReader();
    const decoder = new TextDecoder();
    let buffer = "";
    const scroll = document.getElementById("chatScroll");

    while (true) {
      const { value, done } = await reader.read();
      if (done) break;
      buffer += decoder.decode(value, { stream: true });
      const parts = buffer.split("\n\n");
      buffer = parts.pop() || "";

      for (const chunk of parts) {
        if (!chunk.startsWith("data: ")) continue;
        let ev;
        try { ev = JSON.parse(chunk.slice(6)); } catch { continue; }

        if (ev.type === "tool_start" || ev.type === "tool_end" || ev.type === "status") {
          if (ev.entry) state.toolLog.push(ev.entry);
          if (ev.type === "tool_start") state.heartbeat = "Executing tool...";
          if (ev.type === "tool_end") { state.heartbeat = ""; refreshState(); }
          if (ev.type === "status") state.heartbeat = ev.text || "";
        } else if (ev.type === "heartbeat") {
          state.heartbeat = ev.text || "";
        } else if (ev.type === "stream") {
          state.rawText = ev.text || "";
          state.heartbeat = "";
        } else if (ev.type === "stats") {
          state.stats = ev.text || "";
          updateContextMeter(ev.context_used, ev.context_max);
        } else if (ev.type === "done") {
          state.rawText = ev.text || state.rawText;
          state.toolLog = ev.tool_log || state.toolLog;
          state.stats = ev.stats || state.stats;
          state.images = ev.images || [];
          state.heartbeat = "";
          state.running = false;
          updateContextMeter(ev.context_used, ev.context_max);
          refreshState();
        } else if (ev.type === "error") {
          state.rawText += `\n\n**Runtime Error:** \`${ev.text}\``;
          state.heartbeat = "";
          state.running = false;
        }

        const nearBottom = (scroll.scrollHeight - scroll.scrollTop - scroll.clientHeight) < 160;
        renderAssistantCardContent(card, state);
        if (nearBottom) scroll.scrollTop = scroll.scrollHeight;
      }
    }
  } catch (err) {
    if (err.name !== "AbortError") {
      state.rawText += `\n\n**Stream Error:** \`${err.message}\``;
    }
  } finally {
    state.running = false;
    state.heartbeat = "";
    renderAssistantCardContent(card, state);
    isStreaming = false;
    activeAbortController = null;
    setStatus("ready", "IDLE");
  }
}

function sendContinue() {
  sendMessage("continue");
}

async function stopAgent() {
  setStatus("working", "STOPPING");
  try { await fetch("/api/stop", { method: "POST" }); } catch {}
  if (activeAbortController) {
    try { activeAbortController.abort(); } catch {}
  }
}

async function clearSession() {
  await stopAgent();
  const res = await fetch("/api/clear", { method: "POST" });
  const data = await res.json();
  const scroll = document.getElementById("chatScroll");
  scroll.querySelectorAll(".msg-row").forEach(el => el.remove());
  const hero = document.getElementById("welcomeHero");
  if (hero) hero.style.display = "block";
  applyServerState(data);
  showToast("Session reset");
}

async function exportChat() {
  const res = await fetch("/api/export", { method: "POST" });
  const data = await res.json();
  renderArtifacts(data.artifacts || []);
  switchTab("artifacts");
  showToast(`Exported -> ${data.exported}`);
}

async function installGithubSkill() {
  const url = document.getElementById("ghUrlIn").value.trim();
  const sub = document.getElementById("ghSubIn").value.trim();
  if (!url) { showToast("Enter a GitHub URL first"); return; }
  document.getElementById("skillStatusMsg").textContent = "Cloning repository...";
  const res = await fetch("/api/skills/github", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ repo_url: url, subpath: sub }),
  });
  const data = await res.json();
  document.getElementById("skillStatusMsg").textContent = data.message || "";
  renderSkills(data.skills || []);
}

async function uploadSkillFile(file) {
  if (!file) return;
  const fd = new FormData();
  fd.append("file", file);
  document.getElementById("skillStatusMsg").textContent = "Installing skill...";
  const res = await fetch("/api/skills/upload", { method: "POST", body: fd });
  const data = await res.json();
  document.getElementById("skillStatusMsg").textContent = data.message || "";
  renderSkills(data.skills || []);
}

async function captureDesktop() {
  showToast("Capturing X11 display...");
  const res = await fetch("/api/screen/capture", { method: "POST" });
  const data = await res.json();
  renderScreen(data.screen);
}

async function switchProvider(prov) {
  const key = document.getElementById("geminiKeyIn").value.trim();
  const model = document.getElementById("geminiModelSel").value;
  const res = await fetch("/api/provider", {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ provider: prov, gemini_api_key: key || null, gemini_model: model }),
  });
  const data = await res.json();
  if (!data.ok) {
    switchTab("system");
    document.getElementById("inspectorPane").classList.remove("collapsed");
    document.getElementById("geminiKeyIn").focus();
    showToast(data.message || "Enter Gemini API key in System tab");
    return;
  }
  renderSystem(data.system);
  showToast(data.message || ("Switched to " + prov));
}

async function saveGeminiConfig() {
  await switchProvider("gemini");
}

async function restartServer() {
  showToast("Restarting llama.cpp server...");
  const res = await fetch("/api/system/restart", { method: "POST" });
  const data = await res.json();
  renderSystem(data.system);
  showToast(data.message || "Done");
}

function renderArtifacts(items) {
  const el = document.getElementById("artifactsList");
  if (!items || !items.length) {
    el.innerHTML = `<div style="font-size:11.5px; color:var(--text-muted); font-family:var(--font-mono)">No artifacts in /content/workspace/artifacts/</div>`;
    return;
  }
  el.innerHTML = items.map(it => `
    <div class="artifact-row">
      <div class="art-info">
        <div class="art-name" title="${escapeHtml(it.rel_path)}">${escapeHtml(it.rel_path)}</div>
        <div class="art-meta">${escapeHtml(it.size_str)} · ${escapeHtml(it.ext || "file")}</div>
      </div>
      <div style="display:flex; gap:5px; flex-shrink:0">
        <button class="btn btn-sm" onclick="previewArtifact('${escapeHtml(it.url)}', '${escapeHtml(it.name)}', '${escapeHtml(it.ext)}')">VIEW</button>
        <a class="btn btn-sm btn-primary" href="${escapeHtml(it.url)}&download=1" download="${escapeHtml(it.name)}" style="text-decoration:none">SAVE</a>
      </div>
    </div>`).join("");
}

function renderVideo(vid) {
  const card = document.getElementById("videoCard");
  const player = document.getElementById("videoPlayer");
  if (!vid) { card.style.display = "none"; return; }
  card.style.display = "block";
  document.getElementById("videoName").textContent = vid.name;
  if (player.dataset.src !== vid.url) {
    player.dataset.src = vid.url;
    player.src = vid.url;
  }
}

function renderImages(imgs) {
  const el = document.getElementById("imagesGallery");
  if (!imgs || !imgs.length) {
    el.innerHTML = `<div style="font-size:11.5px; color:var(--text-muted); font-family:var(--font-mono); grid-column:1/-1">No media cached.</div>`;
    return;
  }
  el.innerHTML = imgs.map(im => `
    <div class="gal-card">
      <img src="${escapeHtml(im.url)}" alt="${escapeHtml(im.name)}" onclick="openImageModal('${escapeHtml(im.url)}', '${escapeHtml(im.name)}')" />
      <div class="gal-foot">
        <span style="overflow:hidden; text-overflow:ellipsis; white-space:nowrap" title="${escapeHtml(im.name)}">${escapeHtml(im.name)}</span>
        <button class="btn btn-sm" style="padding:2px 6px; font-size:10px" onclick="navigator.clipboard.writeText('${escapeHtml(im.path)}'); showToast('Copied path')">PATH</button>
      </div>
    </div>`).join("");
}

function renderSkills(skills) {
  const el = document.getElementById("skillsList");
  if (!skills || !skills.length) {
    el.innerHTML = `<div style="font-size:11.5px; color:var(--text-muted); font-family:var(--font-mono)">No skills registered.</div>`;
    return;
  }
  el.innerHTML = skills.map(s => `
    <div class="skill-card">
      <div class="skill-head">
        <span class="skill-name">${escapeHtml(s.name)}</span>
        <div style="display:flex; gap:6px; align-items:center">
          ${s.resources && s.resources.length ? `<span class="meta-tag">${s.resources.length} FILES</span>` : ""}
          <button class="btn btn-sm" style="padding:2px 8px; font-size:10.5px" onclick="msgInput.value='Activate the ${escapeHtml(s.name)} skill and '; msgInput.focus();">USE</button>
        </div>
      </div>
      <div class="skill-desc">${escapeHtml(s.description)}</div>
      <div style="font-family:var(--font-mono); font-size:10px; color:var(--text-muted)">${escapeHtml(s.path)}</div>
    </div>`).join("");
}

function renderScreen(scr) {
  const el = document.getElementById("screenWrap");
  if (!scr) return;
  el.innerHTML = `<img src="${escapeHtml(scr.url)}" style="width:100%; border-radius:6px; border:1px solid var(--border-default); cursor:zoom-in" onclick="openImageModal('${escapeHtml(scr.url)}', 'Viewport Capture')" />`;
}

function renderSystem(sys) {
  if (!sys) return;
  const activeProv = sys.provider || "local";
  document.querySelectorAll("#providerSeg .seg-btn").forEach(b => b.classList.toggle("active", b.dataset.prov === activeProv));
  const ptag = document.getElementById("provStatusTag");
  if (ptag) ptag.textContent = activeProv === "gemini" ? `GEMINI (${sys.gemini_model || "2.5-flash"})` : "LOCAL";
  if (sys.gemini_model) {
    const sel = document.getElementById("geminiModelSel");
    if (sel) sel.value = sys.gemini_model;
  }
  const vb = document.getElementById("visionBadge");
  vb.textContent = sys.vision_active ? "VISION: ON" : "VISION: OFF";
  vb.className = "meta-tag" + (sys.vision_active ? " active-vision" : "");

  if (sys.tunnel_url) {
    publicTunnelUrl = sys.tunnel_url;
    const tb = document.getElementById("tunnelBadge");
    tb.style.display = "inline-flex";
    tb.onclick = () => { navigator.clipboard.writeText(publicTunnelUrl); showToast("Copied URL: " + publicTunnelUrl); };
  }

  const vramPct = sys.vram_total ? Math.min(100, Math.round((sys.vram_used / sys.vram_total) * 100)) : 0;
  document.getElementById("sysTelemetry").innerHTML = `
    <div>LLM SERVER: <span style="color:${sys.server_up ? '#10b981' : '#ef4444'}">${sys.server_up ? "ONLINE (127.0.0.1:8000)" : "OFFLINE (AUTO-RESTARTING)"}</span></div>
    <div>VISION: ${sys.vision_active ? "ENABLED" : "DISABLED"}  |  CTX: ${sys.context_size.toLocaleString()}  |  TEMP: ${sys.temperature}  |  RECOVERIES: ${sys.crash_count || 0}</div>
    <div style="margin-top:6px">GPU: ${escapeHtml(sys.gpu_name)} (${sys.gpu_util}% UTIL)</div>
    <div style="margin-top:2px; font-size:11px">VRAM: ${sys.vram_used} / ${sys.vram_total} MiB (${vramPct}%)</div>
    <div class="ctx-track" style="width:100%; height:4px; margin-top:4px"><div class="ctx-fill" style="width:${vramPct}%"></div></div>
    ${sys.tunnel_url ? `<div style="margin-top:8px">TUNNEL: <a href="${escapeHtml(sys.tunnel_url)}" target="_blank" style="color:var(--accent-cyan)">${escapeHtml(sys.tunnel_url)}</a></div>` : ""}
  `;
  document.getElementById("serverLogBox").textContent = sys.server_log || "(no log)";
}

function applyServerState(data) {
  updateContextMeter(data.context_used, data.context_max);
  renderArtifacts(data.artifacts);
  renderVideo(data.latest_video);
  renderImages(data.images);
  renderSkills(data.skills);
  renderScreen(data.latest_screen);
  renderSystem(data.system);
}

async function refreshState(hydrateChat = false) {
  try {
    const res = await fetch("/api/state");
    const data = await res.json();
    applyServerState(data);
    if (hydrateChat && data.messages && data.messages.length) {
      const scroll = document.getElementById("chatScroll");
      scroll.querySelectorAll(".msg-row").forEach(el => el.remove());
      for (const m of data.messages) {
        if (m.role === "user") {
          appendUserBubble(m.content);
        } else if (m.role === "assistant") {
          const card = appendAssistantCard();
          renderAssistantCardContent(card, {
            rawText: m.content || "",
            toolLog: m.tool_log || [],
            heartbeat: "",
            stats: m.stats || "",
            images: m.images || [],
            running: false,
          });
        }
      }
    }
  } catch (e) {}
}

function openImageModal(url, title) {
  document.getElementById("modalTitle").textContent = title || "IMAGE";
  const dl = document.getElementById("modalDownloadLink");
  dl.href = url + (url.includes("?") ? "&download=1" : "?download=1");
  dl.setAttribute("download", title || "image.png");
  document.getElementById("modalBody").innerHTML = `<img src="${escapeHtml(url)}" />`;
  document.getElementById("previewModal").classList.add("open");
}

async function previewArtifact(url, name, ext) {
  document.getElementById("modalTitle").textContent = name;
  const dl = document.getElementById("modalDownloadLink");
  dl.href = url + "&download=1";
  dl.setAttribute("download", name);
  const body = document.getElementById("modalBody");

  if ([".png", ".jpg", ".jpeg", ".webp", ".gif"].includes(ext)) {
    body.innerHTML = `<img src="${escapeHtml(url)}" />`;
  } else if ([".mp4", ".webm", ".mov"].includes(ext)) {
    body.innerHTML = `<video src="${escapeHtml(url)}" controls autoplay style="max-width:100%; max-height:74vh; border-radius:6px"></video>`;
  } else if ([".pdf", ".html"].includes(ext)) {
    body.innerHTML = `<iframe src="${escapeHtml(url)}"></iframe>`;
  } else {
    body.innerHTML = `<div style="color:var(--text-secondary); font-family:var(--font-mono)">Loading file...</div>`;
    try {
      const txt = await (await fetch(url)).text();
      body.innerHTML = `<pre>${escapeHtml(txt.slice(0, 60000))}</pre>`;
    } catch (e) {
      body.innerHTML = `<div style="color:#ef4444">Inline preview unavailable. Use Download above.</div>`;
    }
  }
  document.getElementById("previewModal").classList.add("open");
}

function closeModal() {
  document.getElementById("previewModal").classList.remove("open");
  document.getElementById("modalBody").innerHTML = "";
}

async function openVncControl() {
  const overlay = document.getElementById("vncOverlay");
  const tag = document.getElementById("vncStatusTag");
  overlay.classList.add("open");
  tag.textContent = "CHECKING VNC...";
  try {
    const r = await fetch("/api/vnc/status");
    const st = await r.json();
    if (!st.novnc_installed) {
      tag.textContent = "NOVNC MISSING";
      showToast("noVNC not installed. Re-run Cell 1 with ENABLE_VIRTUAL_DESKTOP=True.");
      return;
    }
    tag.textContent = st.vnc_up ? "LIVE INTERCEPT" : "VNC STARTING";
  } catch (e) {
    tag.textContent = "CONNECTED";
  }
  reloadVncFrame();
}

function reloadVncFrame() {
  const frame = document.getElementById("vncFrame");
  frame.src = `/novnc/vnc_lite.html?path=novnc/websockify&autoconnect=true&resize=scale&reconnect=true&show_dot=true&t=${Date.now()}`;
}

function closeVncControl() {
  document.getElementById("vncOverlay").classList.remove("open");
  document.getElementById("vncFrame").src = "about:blank";
}

document.addEventListener("mousemove", (e) => {
  const targets = document.querySelectorAll(".preset-card, .card, .artifact-row, .skill-card, .gal-card");
  for (const el of targets) {
    const rect = el.getBoundingClientRect();
    if (
      e.clientX >= rect.left - 80 && e.clientX <= rect.right + 80 &&
      e.clientY >= rect.top - 80 && e.clientY <= rect.bottom + 80
    ) {
      el.style.setProperty("--mx", `${e.clientX - rect.left}px`);
      el.style.setProperty("--my", `${e.clientY - rect.top}px`);
    }
  }
}, { passive: true });

refreshState(true);
</script>
</body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
def index():
    return HTMLResponse(WORKSPACE_HTML)

def _wait_local_server(port: int, timeout: float = 15.0) -> bool:
    import urllib.request
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/api/health", timeout=1.5) as r:
                if r.status == 200:
                    return True
        except Exception:
            time.sleep(0.3)
    return False

def _start_uvicorn():
    global _UVICORN_SERVER, _UVICORN_THREAD
    config = uvicorn.Config(app, host="127.0.0.1", port=SERVER_PORT, log_level="warning")
    _UVICORN_SERVER = uvicorn.Server(config)
    _UVICORN_THREAD = threading.Thread(target=_UVICORN_SERVER.run, daemon=True)
    _UVICORN_THREAD.start()
    return _wait_local_server(SERVER_PORT)

if not _start_uvicorn():
    raise RuntimeError(f"FastAPI server failed to bind on 127.0.0.1:{SERVER_PORT}")

cf_proc = None
public_url = ""

def _start_cloudflare_tunnel() -> str:
    global cf_proc, public_url
    if not (ENABLE_CLOUDFLARE_TUNNEL and os.path.exists("/usr/local/bin/cloudflared")):
        return ""
    subprocess.run(["pkill", "-f", f"cloudflared tunnel --url http://127.0.0.1:{SERVER_PORT}"], check=False)
    time.sleep(0.5)
    cf_log_f = open(CLOUDFLARE_LOG, "w")
    cf_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}", "--no-autoupdate"],
        stdout=cf_log_f,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    deadline = time.time() + 30.0
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            break
        try:
            with open(CLOUDFLARE_LOG, errors="replace") as f:
                txt = f.read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
            if m:
                public_url = m.group(0)
                STUDIO_STATE["tunnel_url"] = public_url
                return public_url
        except Exception:
            pass
        time.sleep(0.5)
    return ""

_start_cloudflare_tunnel()

try:
    from IPython.display import display, HTML
    if public_url:
        display(HTML(f"""
        <div style="padding:14px 18px; border-radius:10px; background:#090a0f; border:1px solid rgba(255,255,255,0.14); color:#f4f4f6; font-family:'Inter',system-ui,sans-serif; margin:8px 0;">
          <div style="font-size:10.5px; text-transform:uppercase; letter-spacing:0.08em; color:#22d3ee; font-family:monospace; font-weight:600; margin-bottom:6px;">
            CLOUDFLARE TUNNEL ACTIVE
          </div>
          <div style="display:flex; align-items:center; gap:12px; flex-wrap:wrap;">
            <a href="{public_url}" target="_blank" style="font-size:15px; font-weight:600; color:#f4f4f6; text-decoration:none; font-family:monospace; word-break:break-all;">
              {public_url}
            </a>
            <button onclick="navigator.clipboard.writeText('{public_url}'); this.textContent='COPIED'; setTimeout(()=>this.textContent='COPY URL',1400)"
                    style="padding:5px 10px; border-radius:6px; border:1px solid rgba(255,255,255,0.18); background:#18181b; color:#f4f4f6; font-family:monospace; font-size:11px; cursor:pointer;">
              COPY URL
            </button>
          </div>
          <div style="font-size:11.5px; color:#9ca3af; margin-top:6px;">
            Cell 5 is now running in persistent watchdog mode below for live bug diagnostics and automatic crash recovery.
          </div>
        </div>
        """))
    else:
        print(f"[Server] Local UI online at http://127.0.0.1:{SERVER_PORT} (Cloudflare tunnel unavailable; check {CLOUDFLARE_LOG}).")

    if EMBED_COLAB_IFRAME:
        from google.colab import output as colab_output
        colab_output.serve_kernel_port_as_iframe(SERVER_PORT, height=840)
except Exception:
    print(f"[OK] Studio UI online -> {public_url or f'http://127.0.0.1:{SERVER_PORT}'}")

if KEEP_CELL_RUNNING_FOREVER:
    print(f"\n[Watchdog] Persistent monitor active (Port {SERVER_PORT} | Tunnel: {public_url or 'local'}). Press Stop in Colab to exit.")
    _last_log_pos = os.path.getsize(SERVER_LOG) if os.path.exists(SERVER_LOG) else 0
    _last_ev_idx = len(globals().get("AGENT_EVENT_LOG", []))
    _tick = 0
    try:
        while True:
            time.sleep(3.0)
            _tick += 1

            ev_list = globals().get("AGENT_EVENT_LOG", [])
            if len(ev_list) < _last_ev_idx:
                _last_ev_idx = 0
            while _last_ev_idx < len(ev_list):
                print(f"[Agent] {ev_list[_last_ev_idx]}", flush=True)
                _last_ev_idx += 1

            if os.path.exists(SERVER_LOG):
                cur_size = os.path.getsize(SERVER_LOG)
                if cur_size < _last_log_pos:
                    _last_log_pos = 0
                if cur_size > _last_log_pos:
                    with open(SERVER_LOG, "r", errors="replace") as lf:
                        lf.seek(_last_log_pos)
                        new_chunk = lf.read()
                        _last_log_pos = lf.tell()
                    for line in new_chunk.splitlines():
                        low = line.lower()
                        if any(k in low for k in ("error", "assert", "exception", "traceback", "cuda", "out of memory", "segfault", "killed")):
                            print(f"[llama.cpp] {line}", flush=True)

            proc = globals().get("llama_proc")
            if proc is not None and proc.poll() is not None:
                print(f"[Watchdog] Detected llama.cpp exit (code {proc.returncode}). Restarting server...", flush=True)
                try:
                    start_or_restart_llama_server(force_restart=True)
                    print("[Watchdog] llama.cpp server recovered.", flush=True)
                except Exception as e:
                    print(f"[Watchdog] llama.cpp restart error: {e}", flush=True)

            if not _wait_local_server(SERVER_PORT, timeout=2.0):
                print("[Watchdog] FastAPI UI server unresponsive. Restarting Uvicorn...", flush=True)
                _start_uvicorn()

            if os.path.isdir("/usr/share/novnc") and not _tcp_open("127.0.0.1", 5900, timeout=0.3):
                _ensure_vnc_running()

            if ENABLE_CLOUDFLARE_TUNNEL and cf_proc is not None and cf_proc.poll() is not None:
                print("[Watchdog] cloudflared tunnel exited. Respawning tunnel...", flush=True)
                new_url = _start_cloudflare_tunnel()
                if new_url:
                    print(f"[Watchdog] New Cloudflare Tunnel URL: {new_url}", flush=True)

            if _tick % 20 == 0:
                sys_m = get_system_meta()
                print(
                    f"[Heartbeat {time.strftime('%H:%M:%S')}] "
                    f"LLM={'UP' if sys_m['server_up'] else 'DOWN'} | "
                    f"VRAM={sys_m['vram_used']}/{sys_m['vram_total']} MiB ({sys_m['gpu_util']}%) | "
                    f"Recoveries={sys_m['crash_count']} | "
                    f"URL={STUDIO_STATE.get('tunnel_url') or f'http://127.0.0.1:{SERVER_PORT}'}",
                    flush=True,
                )
    except KeyboardInterrupt:
        print("\n[Watchdog] Monitor loop stopped by user (servers remain running in background).")